# Why is evidence not helping? A bounded AVeriTeC limitation study

Run this notebook in a fresh **Colab GPU runtime**, from top to bottom. It embeds its Python helpers, so there is no repository pull or missing-script step. All checkpoints and results go to a new Google Drive directory. The large passage corpus is never re-embedded or re-indexed. The test split is never opened.

Our starting observations: BM25 URL-proxy Hit@3 = 0.1969; BM25-trained DistilBERT macro-F1 = 0.3469; supplying fusion dev passages gives 0.3808; claim-only = 0.3970; annotated answers = 0.5284. These observations do not establish whether retrieval or reading is the main bottleneck.

| Question | Controlled check | What it can tell us |
| --- | --- | --- |
| Is useful material present but ranked too low? | Existing top-20/top-100 coverage; URL-oracle raw-passage selection | Candidate/ranking opportunity, subject to the URL proxy |
| Are three results wasting space on the same source? | Distinct-source selection from existing top 20 | Whether simple diversity helps |
| Does the classifier lose useful details? | Same passages, first 110 tokens versus query-selected source sentences | Sensitivity to input packing, then matched training |
| Does it respond to evidence? | Remove it, give another claim's passages, reverse their order | Evidence sensitivity; changes alone do not prove reasoning |
| Does class weighting cause a bad tradeoff? | Balanced/unweighted loss with identical train holdout and text | Minority-label, calibration and safety tradeoffs |
| Did the trained reranker have enough trustworthy supervision? | Inspect existing train-only silver alignment coverage by verdict | Whether supervision is sparse or biased; audit quality separately |
| Does training overshoot its best checkpoint? | Record the selected checkpoint's epoch; refit for that duration | A possible selection/refit mismatch |
| Do earlier-to-later claims behave differently? | Additional temporal holdout inside official train | A limited temporal-shift check |
| Would stronger pretraining help? | Optional DeBERTa-v3-small with the selected text format | Whether encoder choice helps with the same evidence budget |

The fast audit/inference section should take minutes to tens of minutes. Seven train-holdout trials plus final refits may take tens of minutes to a few hours, depending on GPU and Drive. These are planning estimates; each cell prints its actual time. Optional stronger-model and seed checks add work. Start with the defaults.

Gold URL selection and annotated answers are **oracle diagnostics**. They are saved separately and cannot enter train-only pipeline selection. Existing dev has been inspected many times; new dev scores remain exploratory. A final model still needs untouched evaluation and a user study.

## 1. Install the text-only runtime and mount Drive

Use a fresh runtime if previous notebooks imported different Transformers versions. Experiments run in the notebook process so progress bars and tracebacks remain visible.

In [ ]:
from pathlib import Path
import gc, hashlib, json, os, runpy, subprocess, sys, time, traceback
os.environ['USE_TF'] = '0'
os.environ['USE_FLAX'] = '0'
os.environ['USE_TORCH'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['HF_HOME'] = '/content/huggingface_cache'
loaded_transformers = sys.modules.get('transformers')
if loaded_transformers is not None and loaded_transformers.__version__ != '4.57.1':
    raise RuntimeError('Restart the runtime, then run this notebook first (Transformers version differs).')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'transformers==4.57.1', 'accelerate>=1,<2', 'numpy>=1.26,<3',
                'scikit-learn>=1.5,<2', 'joblib>=1.4,<2', 'tqdm>=4,<5',
                'sentencepiece>=0.2,<0.3', 'protobuf>=4,<7'], check=True)
from google.colab import drive
drive.mount('/content/drive')
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer
if not torch.cuda.is_available():
    raise RuntimeError('Choose Runtime > Change runtime type > GPU, then reconnect and rerun setup.')
print('Python:', sys.version.split()[0], '| PyTorch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0))


## 2. Load the bundled study code

This cell contains the readable source snapshot used for this notebook. Its hash names an isolated runtime directory. It does not overwrite your project checkout.

In [ ]:
STUDY_SOURCES = json.loads('{"scripts/__init__.py": "", "scripts/run_evidence_ablation.py": "\\"\\"\\"Test a frozen verifier with original, missing, and mismatched evidence.\\n\\nThis is a diagnostic, not a deployable verifier. The same final model and\\nofficial-dev claims are used in each condition; no labels or qrels enter the\\nmodel inputs. Interrupted inference can resume from append-only JSONL files.\\n\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport argparse\\nimport json\\nimport sys\\nfrom pathlib import Path\\nfrom typing import Any\\n\\nPROJECT_ROOT = Path(__file__).resolve().parents[1]\\nsys.path.insert(0, str(PROJECT_ROOT))\\n\\nfrom scripts.train_tfidf_logreg import attach_retrieved_evidence  # noqa: E402\\nfrom src.data.averitec import CANONICAL_LABELS  # noqa: E402\\nfrom src.evaluation.classification import evaluate_predictions  # noqa: E402\\nfrom src.utils.io import read_jsonl, write_json, write_jsonl  # noqa: E402\\nfrom src.utils.progress import progress  # noqa: E402\\nfrom src.utils.runs import resolve_run_directory, sha256_file, write_run_manifest  # noqa: E402\\n\\n\\ndef evidence_variants(records: list[dict[str, Any]]) -> dict[str, list[dict[str, Any]]]:\\n    \\"\\"\\"Preserve the missing-pool claim and rotate only nonempty evidence pools.\\"\\"\\"\\n    ids = [str(row[\\"claim_id\\"]) for row in records]\\n    if len(ids) != len(set(ids)):\\n        raise ValueError(\\"Duplicate claim IDs\\")\\n    nonempty = sorted((row for row in records if row.get(\\"retrieved_evidence\\")), key=lambda row: str(row[\\"claim_id\\"]))\\n    if len(nonempty) < 2:\\n        raise ValueError(\\"Need at least two nonempty claim evidence pools for a swap diagnostic\\")\\n    donors = {\\n        str(row[\\"claim_id\\"]): nonempty[(index + 1) % len(nonempty)]\\n        for index, row in enumerate(nonempty)\\n    }\\n    blank = [{**row, \\"retrieved_evidence\\": []} for row in records]\\n    swapped = []\\n    for row in records:\\n        claim_id = str(row[\\"claim_id\\"])\\n        donor = donors.get(claim_id)\\n        swapped.append({\\n            **row,\\n            \\"retrieved_evidence\\": (donor.get(\\"retrieved_evidence\\") or [])[:3] if donor else [],\\n            \\"evidence_donor_claim_id\\": str(donor[\\"claim_id\\"]) if donor else None,\\n        })\\n    return {\\"blank\\": blank, \\"swapped\\": swapped}\\n\\n\\ndef load_partial_predictions(path: Path, expected_ids: set[str]) -> dict[str, dict[str, Any]]:\\n    \\"\\"\\"Discard one torn final line, but never silently accept duplicate IDs.\\"\\"\\"\\n    if not path.is_file():\\n        return {}\\n    data = path.read_bytes()\\n    lines = data.splitlines(keepends=True)\\n    last = max((index for index, line in enumerate(lines) if line.strip()), default=-1)\\n    found: dict[str, dict[str, Any]] = {}\\n    offset = 0\\n    for index, raw in enumerate(lines):\\n        start = offset\\n        offset += len(raw)\\n        if not raw.strip():\\n            continue\\n        try:\\n            row = json.loads(raw.decode(\\"utf-8\\"))\\n        except (UnicodeDecodeError, json.JSONDecodeError) as error:\\n            if index != last:\\n                raise ValueError(f\\"Corrupt checkpoint before final line: {path}\\") from error\\n            path.write_bytes(data[:start])\\n            print(f\\"Removed interrupted final checkpoint line from {path}\\", flush=True)\\n            break\\n        claim_id = str(row.get(\\"claim_id\\") or \\"\\")\\n        if claim_id not in expected_ids or claim_id in found:\\n            raise ValueError(f\\"Unexpected or duplicate checkpoint claim ID {claim_id!r} in {path}\\")\\n        if row.get(\\"predicted_label\\") not in CANONICAL_LABELS or row.get(\\"true_label\\") not in CANONICAL_LABELS:\\n            raise ValueError(f\\"Malformed checkpoint prediction for {claim_id!r}\\")\\n        found[claim_id] = row\\n    if data and not data.endswith(b\\"\\\\n\\") and len(found) == sum(bool(line.strip()) for line in lines):\\n        # A complete final JSON record can still lack the newline required\\n        # before the next append. Preserve it and add only the separator.\\n        with path.open(\\"ab\\") as handle:\\n            handle.write(b\\"\\\\n\\")\\n    return found\\n\\n\\ndef comparison(original: list[dict[str, Any]], altered: list[dict[str, Any]]) -> dict[str, int | float]:\\n    baseline = {str(row[\\"claim_id\\"]): row for row in original}\\n    changed = lost = fixed = 0\\n    for row in altered:\\n        old = baseline[str(row[\\"claim_id\\"])]\\n        if old[\\"true_label\\"] != row[\\"true_label\\"]:\\n            raise ValueError(f\\"Gold label changed for {row[\'claim_id\']}\\")\\n        changed += old[\\"predicted_label\\"] != row[\\"predicted_label\\"]\\n        lost += old[\\"predicted_label\\"] == old[\\"true_label\\"] and row[\\"predicted_label\\"] != row[\\"true_label\\"]\\n        fixed += old[\\"predicted_label\\"] != old[\\"true_label\\"] and row[\\"predicted_label\\"] == row[\\"true_label\\"]\\n    return {\\n        \\"changed_prediction_count\\": changed,\\n        \\"changed_prediction_fraction\\": changed / len(altered),\\n        \\"originally_correct_now_wrong\\": lost,\\n        \\"originally_wrong_now_correct\\": fixed,\\n    }\\n\\n\\ndef main() -> None:\\n    parser = argparse.ArgumentParser()\\n    parser.add_argument(\\"--claims\\", required=True, type=Path)\\n    parser.add_argument(\\"--rankings\\", required=True, type=Path)\\n    parser.add_argument(\\"--original-predictions\\", required=True, type=Path)\\n    parser.add_argument(\\"--model-dir\\", required=True, type=Path)\\n    parser.add_argument(\\"--model-run-config\\", required=True, type=Path)\\n    parser.add_argument(\\"--output-dir\\", required=True, type=Path)\\n    parser.add_argument(\\"--run-name\\", required=True)\\n    parser.add_argument(\\"--batch-size\\", type=int, default=16)\\n    parser.add_argument(\\"--checkpoint-every\\", type=int, default=32)\\n    parser.add_argument(\\"--device\\", default=\\"cuda\\")\\n    parser.add_argument(\\"--resume\\", action=\\"store_true\\")\\n    args = parser.parse_args()\\n    if min(args.batch_size, args.checkpoint_every) < 1:\\n        parser.error(\\"Batch size and checkpoint interval must be positive\\")\\n    for path in (args.claims, args.rankings, args.original_predictions, args.model_run_config, args.model_dir / \\"config.json\\"):\\n        if not path.is_file():\\n            raise FileNotFoundError(path)\\n    weight_files = sorted((*args.model_dir.glob(\\"*.safetensors\\"), *args.model_dir.glob(\\"pytorch_model*.bin\\")))\\n    if not weight_files:\\n        raise FileNotFoundError(f\\"No model weights found in {args.model_dir}\\")\\n    output = resolve_run_directory(args.output_dir, args.run_name)\\n    output.mkdir(parents=True, exist_ok=True)\\n    config = {\\n        \\"script\\": \\"run_evidence_ablation\\",\\n        \\"claims_sha256\\": sha256_file(args.claims),\\n        \\"rankings_sha256\\": sha256_file(args.rankings),\\n        \\"original_predictions_sha256\\": sha256_file(args.original_predictions),\\n        \\"model_run_config_sha256\\": sha256_file(args.model_run_config),\\n        \\"model_config_sha256\\": sha256_file(args.model_dir / \\"config.json\\"),\\n        \\"model_weights_sha256\\": {path.name: sha256_file(path) for path in weight_files},\\n        \\"model_dir\\": str(args.model_dir.resolve()),\\n        \\"batch_size\\": args.batch_size,\\n        \\"checkpoint_every\\": args.checkpoint_every,\\n        \\"device\\": args.device,\\n        \\"swap_policy\\": \\"next nonempty claim in sorted claim-ID order; missing-pool claims remain empty\\",\\n    }\\n    config_path = output / \\"ablation_config.json\\"\\n    prior = json.loads(config_path.read_text(encoding=\\"utf-8\\")) if config_path.is_file() else None\\n    if prior is not None and prior != config:\\n        raise ValueError(\\"Ablation inputs/settings changed; use a new run name\\")\\n    report_path = output / \\"ablation_report.json\\"\\n    if report_path.is_file() and (output / \\"ablation_manifest.json\\").is_file() and all(\\n        (output / f\\"{name}_predictions.jsonl\\").is_file() for name in (\\"blank\\", \\"swapped\\")\\n    ):\\n        print(report_path.read_text(encoding=\\"utf-8\\"))\\n        return\\n    write_json(config, config_path)\\n\\n    records = attach_retrieved_evidence(read_jsonl(args.claims), args.rankings)\\n    by_id = {str(row[\\"claim_id\\"]): row for row in records}\\n    if len(by_id) != len(records):\\n        raise ValueError(\\"Duplicate dev claim IDs\\")\\n    original = read_jsonl(args.original_predictions)\\n    original_by_id = {str(row[\\"claim_id\\"]): row for row in original}\\n    if len(original_by_id) != len(original) or set(original_by_id) != set(by_id):\\n        raise ValueError(\\"Original predictions must cover the exact dev claim set\\")\\n    for row in records:\\n        if original_by_id[str(row[\\"claim_id\\"])][\\"true_label\\"] != row[\\"label\\"]:\\n            raise ValueError(f\\"Original prediction label mismatch for {row[\'claim_id\']}\\")\\n    variants = evidence_variants(records)\\n    model_settings = json.loads(args.model_run_config.read_text(encoding=\\"utf-8\\"))\\n\\n    import torch\\n    from transformers import AutoModelForSequenceClassification, AutoTokenizer\\n    from scripts.train_transformer_verifier import compose_text\\n\\n    if args.device.startswith(\\"cuda\\") and not torch.cuda.is_available():\\n        raise RuntimeError(\\"CUDA unavailable; select a Colab GPU runtime\\")\\n    tokenizer = AutoTokenizer.from_pretrained(args.model_dir, use_fast=True)\\n    model = AutoModelForSequenceClassification.from_pretrained(args.model_dir).to(args.device).eval()\\n    if int(model.config.num_labels) != len(CANONICAL_LABELS):\\n        raise ValueError(\\"Final model has an unexpected number of verdict labels\\")\\n    if any(str(model.config.id2label.get(index)) != label for index, label in enumerate(CANONICAL_LABELS)):\\n        raise ValueError(\\"Final model label order differs from the original verifier\\")\\n    results: dict[str, Any] = {}\\n    for name, rows in variants.items():\\n        final_path = output / f\\"{name}_predictions.jsonl\\"\\n        partial_path = output / f\\"{name}_predictions.partial.jsonl\\"\\n        if final_path.is_file():\\n            predictions = read_jsonl(final_path)\\n            if len(predictions) != len(rows) or {r[\\"claim_id\\"] for r in predictions} != set(by_id):\\n                raise ValueError(f\\"Incomplete or duplicate final predictions: {final_path}\\")\\n        else:\\n            if partial_path.is_file() and not args.resume:\\n                raise ValueError(f\\"Partial checkpoint exists; rerun with --resume: {partial_path}\\")\\n            completed = load_partial_predictions(partial_path, set(by_id)) if args.resume else {}\\n            pending: list[dict[str, Any]] = []\\n\\n            def flush() -> None:\\n                nonlocal pending\\n                batch = pending\\n                pending = []\\n                if batch:\\n                    with partial_path.open(\\"a\\", encoding=\\"utf-8\\", newline=\\"\\\\n\\") as handle:\\n                        for item in batch:\\n                            handle.write(json.dumps(item, ensure_ascii=False) + \\"\\\\n\\")\\n                        handle.flush()\\n\\n            with progress(total=len(rows), initial=len(completed), description=f\\"Ablation {name}\\", unit=\\"claim\\") as bar:\\n                try:\\n                    for start in range(0, len(rows), args.batch_size):\\n                        batch_rows = [row for row in rows[start : start + args.batch_size] if row[\\"claim_id\\"] not in completed]\\n                        if not batch_rows:\\n                            continue\\n                        texts = [compose_text(\\n                            row, tokenizer,\\n                            int(model_settings[\\"claim_token_budget\\"]),\\n                            int(model_settings[\\"evidence_token_budget_per_passage\\"]),\\n                        ) for row in batch_rows]\\n                        encoded = tokenizer(\\n                            texts, padding=True, truncation=True,\\n                            max_length=int(model_settings[\\"max_length\\"]), return_tensors=\\"pt\\",\\n                        )\\n                        encoded = {key: value.to(args.device) for key, value in encoded.items()}\\n                        with torch.inference_mode():\\n                            probabilities = torch.softmax(model(**encoded).logits.float(), dim=-1).cpu().tolist()\\n                        for row, values in zip(batch_rows, probabilities, strict=True):\\n                            prediction = {\\n                                \\"claim_id\\": str(row[\\"claim_id\\"]),\\n                                \\"true_label\\": str(row[\\"label\\"]),\\n                                \\"predicted_label\\": CANONICAL_LABELS[max(range(len(values)), key=values.__getitem__)],\\n                                \\"probabilities\\": {label: float(values[index]) for index, label in enumerate(CANONICAL_LABELS)},\\n                                \\"evidence_item_count\\": min(3, len(row.get(\\"retrieved_evidence\\") or [])),\\n                                \\"evidence_donor_claim_id\\": row.get(\\"evidence_donor_claim_id\\"),\\n                            }\\n                            completed[prediction[\\"claim_id\\"]] = prediction\\n                            pending.append(prediction)\\n                        if len(pending) >= args.checkpoint_every:\\n                            flush()\\n                        bar.update(len(batch_rows))\\n                finally:\\n                    flush()\\n            if len(completed) != len(rows):\\n                raise RuntimeError(f\\"Only {len(completed)}/{len(rows)} {name} predictions completed\\")\\n            predictions = [completed[str(row[\\"claim_id\\"])] for row in rows]\\n            temporary = final_path.with_name(f\\"{final_path.name}.tmp\\")\\n            write_jsonl(predictions, temporary)\\n            temporary.replace(final_path)\\n        results[name] = {\\n            \\"metrics\\": evaluate_predictions(predictions),\\n            \\"versus_original\\": comparison(original, predictions),\\n            \\"predictions_path\\": str(final_path),\\n        }\\n    report = {\\n        **config,\\n        \\"claim_count\\": len(records),\\n        \\"original_metrics_recomputed\\": evaluate_predictions(original),\\n        \\"variants\\": results,\\n        \\"interpretation_warning\\": \\"Blank/swapped inputs are distribution-shift diagnostics; do not interpret score differences as a causal estimate of retrieval quality.\\",\\n    }\\n    write_json(report, report_path)\\n    write_run_manifest(\\n        output / \\"ablation_manifest.json\\", project_root=PROJECT_ROOT, config=report,\\n        input_paths={\\"claims\\": args.claims, \\"rankings\\": args.rankings, \\"original_predictions\\": args.original_predictions,\\n                     \\"model_config\\": args.model_dir / \\"config.json\\", \\"training_run_config\\": args.model_run_config},\\n    )\\n    print(json.dumps({\\n        \\"original_macro_f1\\": report[\\"original_metrics_recomputed\\"][\\"macro_f1\\"],\\n        \\"variants\\": {name: {\\"macro_f1\\": item[\\"metrics\\"][\\"macro_f1\\"], **item[\\"versus_original\\"]} for name, item in results.items()},\\n        \\"report_path\\": str(report_path),\\n    }, indent=2))\\n\\n\\nif __name__ == \\"__main__\\":\\n    main()\\n", "scripts/run_limitation_diagnostics.py": "\\"\\"\\"Bounded, resumable diagnostics using existing rankings; never scans the corpus.\\n\\nPrepare separates prediction-time variants from explicitly marked oracle inputs.\\nFrozen experiments change one saved verifier\'s inputs. Training and selection\\nare orchestrated by the companion notebook using official-train holdouts only.\\n\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport argparse\\nimport csv\\nimport json\\nimport math\\nimport random\\nimport re\\nimport sys\\nfrom collections import Counter, defaultdict\\nfrom pathlib import Path\\nfrom typing import Any\\n\\nPROJECT_ROOT = Path(__file__).resolve().parents[1]\\nsys.path.insert(0, str(PROJECT_ROOT))\\n\\nimport numpy as np\\n\\nfrom scripts.run_evidence_ablation import comparison, evidence_variants, load_partial_predictions\\nfrom src.data.averitec import CANONICAL_LABELS\\nfrom src.data.corpus import canonical_url\\nfrom src.evaluation.classification import evaluate_predictions\\nfrom src.evaluation.splits import parse_claim_date as parse_date\\nfrom src.retrieval.metrics import evaluate_rankings\\nfrom src.retrieval.question_guided import SENTENCE_BOUNDARY, words\\nfrom src.utils.io import read_jsonl, write_json, write_jsonl\\nfrom src.utils.progress import progress\\nfrom src.utils.runs import sha256_file\\n\\nVERSION = 1\\n\\n\\ndef atomic_rows(rows: list[dict], path: Path) -> None:\\n    path.parent.mkdir(parents=True, exist_ok=True)\\n    temp = path.with_name(path.name + \\".tmp\\")\\n    write_jsonl(rows, temp)\\n    temp.replace(path)\\n\\n\\ndef lock_config(path: Path, config: dict) -> None:\\n    if path.exists() and json.loads(path.read_text(encoding=\\"utf-8\\")) != config:\\n        raise ValueError(f\\"Inputs/settings changed. Choose a new output directory: {path.parent}\\")\\n    write_json(config, path)\\n\\n\\ndef index_rows(rows: list[dict], *, expected: set[str] | None = None) -> dict[str, dict]:\\n    result = {}\\n    for row in rows:\\n        key = str(row.get(\\"claim_id\\") or \\"\\")\\n        if not key or key in result:\\n            raise ValueError(f\\"Missing or duplicate claim ID: {key!r}\\")\\n        result[key] = row\\n    if expected is not None and set(result) != expected:\\n        raise ValueError(f\\"Claim coverage differs: missing={sorted(expected-set(result))[:3]}, extra={sorted(set(result)-expected)[:3]}\\")\\n    return result\\n\\n\\ndef load_rankings(path: Path, records: list[dict]) -> dict[str, dict]:\\n    found = index_rows(read_jsonl(path), expected={str(r[\\"claim_id\\"]) for r in records})\\n    for record in records:\\n        row = found[str(record[\\"claim_id\\"])]\\n        if row.get(\\"claim\\") and row[\\"claim\\"].strip() != record[\\"claim\\"].strip():\\n            raise ValueError(f\\"Claim text differs in {path}: {record[\'claim_id\']}\\")\\n        candidates = row.get(\\"retrieved\\")\\n        if not isinstance(candidates, list):\\n            raise ValueError(f\\"Malformed retrieved list: {record[\'claim_id\']}\\")\\n        ids = [str(p.get(\\"passage_id\\") or \\"\\") for p in candidates]\\n        if len(ids) != len(set(ids)) or any(not pid for pid in ids):\\n            raise ValueError(f\\"Missing/duplicate passage IDs: {record[\'claim_id\']}\\")\\n        if any(not str(p.get(\\"text\\") or \\"\\").strip() for p in candidates):\\n            raise ValueError(f\\"Blank passage: {record[\'claim_id\']}\\")\\n    return found\\n\\n\\ndef source_key(item: dict) -> str:\\n    return canonical_url(item.get(\\"url\\")) or f\\"passage:{item[\'passage_id\']}\\"\\n\\n\\ndef diverse_top3(candidates: list[dict]) -> list[dict]:\\n    chosen, seen = [], set()\\n    for item in candidates:\\n        key = source_key(item)\\n        if key not in seen:\\n            chosen.append(item)\\n            seen.add(key)\\n        if len(chosen) == 3:\\n            break\\n    used = {p[\\"passage_id\\"] for p in chosen}\\n    return (chosen + [p for p in candidates if p[\\"passage_id\\"] not in used])[:3]\\n\\n\\ndef query_excerpt(text: str, claim: str, tokenizer: Any, budget: int) -> str:\\n    \\"\\"\\"Pack verbatim sentences, or a contiguous word window for a long sentence.\\n\\n    Only the claim and raw passage are used. Nothing is generated or rewritten.\\n    Sentence order is restored after selection. The model tokenizer enforces\\n    the same per-passage budget as the original head-truncation baseline.\\n    \\"\\"\\"\\n    original = text.strip()\\n    count = lambda value: len(tokenizer.encode(value, add_special_tokens=False))\\n    if count(original) <= budget:\\n        return original\\n    sentences = [s.strip() for s in SENTENCE_BOUNDARY.split(original) if s.strip()]\\n    query = words(claim)\\n    order = sorted(range(len(sentences)), key=lambda i: (-len(words(sentences[i]) & query), i))\\n    selected = []\\n    for i in order:\\n        proposed = sorted([*selected, i])\\n        if count(\\" \\".join(sentences[j] for j in proposed)) <= budget:\\n            selected = proposed\\n    if selected:\\n        return \\" \\".join(sentences[i] for i in selected)\\n    # Long unpunctuated passages: search bounded, verbatim word windows.\\n    spans = list(re.finditer(r\\"\\\\S+\\", original))\\n    width = min(max(1, budget // 2), len(spans))\\n    starts = set(range(0, len(spans), max(1, width // 3)))\\n    starts.add(max(0, len(spans) - width))\\n    windows = []\\n    for start in sorted(starts):\\n        end = min(len(spans), start + width)\\n        while end > start and count(original[spans[start].start():spans[end - 1].end()]) > budget:\\n            end -= 1\\n        if end > start:\\n            excerpt = original[spans[start].start():spans[end - 1].end()]\\n            windows.append((-len(words(excerpt) & query), start, excerpt))\\n    return min(windows)[2] if windows else original[:1]\\n\\n\\ndef prediction_variant(claim_id: str, claim: str, candidates: list[dict], mode: str, tokenizer: Any, budget: int) -> dict:\\n    \\"\\"\\"Whitelist fields: gold labels, answers and URLs from annotations cannot enter.\\"\\"\\"\\n    selected = diverse_top3(candidates[:20]) if mode == \\"diverse_sentences\\" else candidates[:3]\\n    if mode == \\"claim_only\\":\\n        selected = []\\n    passages = []\\n    for item in selected:\\n        text = str(item[\\"text\\"])\\n        if mode in {\\"sentences\\", \\"diverse_sentences\\"}:\\n            text = query_excerpt(text, claim, tokenizer, budget)\\n        passages.append({\\"passage_id\\": str(item[\\"passage_id\\"]), \\"text\\": text, \\"url\\": item.get(\\"url\\")})\\n    return {\\"claim_id\\": claim_id, \\"claim\\": claim, \\"retrieved\\": passages}\\n\\n\\ndef audit_claim(record: dict, candidates: list[dict], qrels: set[str], tokenizer: Any, budget: int) -> dict:\\n    top = candidates[:3]\\n    row = {\\"claim_id\\": record[\\"claim_id\\"], \\"label\\": record[\\"label\\"], \\"judged\\": bool(qrels),\\n           \\"available_candidate_count\\": len(candidates), \\"top3_distinct_sources\\": len({source_key(p) for p in top}),\\n           \\"top3_passage_count\\": len(top), \\"claim_date\\": parse_date(record.get(\\"metadata\\", {}).get(\\"claim_date\\"))}\\n    for k in (3, 20, 100):\\n        # A short saved list is not a top-100 candidate pool.\\n        row[f\\"url_hit_at_available_{k}\\"] = bool({str(p[\\"passage_id\\"]) for p in candidates[:k]} & qrels) if qrels else None\\n    lengths, head_coverage, excerpt_coverage = [], [], []\\n    query = words(record[\\"claim\\"])\\n    for p in top:\\n        tokens = tokenizer.encode(p[\\"text\\"], add_special_tokens=False)\\n        lengths.append(len(tokens))\\n        full_terms = words(p[\\"text\\"]) & query\\n        head_terms = words(tokenizer.decode(tokens[:budget], skip_special_tokens=True)) & query\\n        excerpt_terms = words(query_excerpt(p[\\"text\\"], record[\\"claim\\"], tokenizer, budget)) & query\\n        if full_terms:\\n            head_coverage.append(len(head_terms) / len(full_terms))\\n            excerpt_coverage.append(len(excerpt_terms) / len(full_terms))\\n    row.update({\\"passages_over_budget\\": sum(n > budget for n in lengths), \\"passage_token_lengths\\": lengths,\\n                \\"head_query_term_retention\\": float(np.mean(head_coverage)) if head_coverage else None,\\n                \\"excerpt_query_term_retention\\": float(np.mean(excerpt_coverage)) if excerpt_coverage else None})\\n    return row\\n\\n\\ndef export_manual_audit(records: list[dict], pools: dict[str, dict], qrels: dict[str, set], audits: list[dict], output: Path, size: int, seed: int) -> None:\\n    groups = defaultdict(list)\\n    for row in audits:\\n        group = \\"unjudged\\" if not row[\\"judged\\"] else \\"top3_hit\\" if row[\\"url_hit_at_available_3\\"] else \\"deeper_hit\\" if row[\\"url_hit_at_available_100\\"] else \\"no_saved_hit\\"\\n        groups[group].append(row[\\"claim_id\\"])\\n    rng = random.Random(seed)\\n    for group in groups.values():\\n        rng.shuffle(group)\\n    selected = []\\n    while len(selected) < size and any(groups.values()):\\n        for group in sorted(groups):\\n            if groups[group] and len(selected) < size:\\n                selected.append(groups[group].pop())\\n    by_id = index_rows(records)\\n    exported, key = [], []\\n    for cid in selected:\\n        candidates = pools[cid][\\"retrieved\\"]\\n        chosen = list(candidates[:3])\\n        deeper = next((p for p in candidates[3:] if str(p[\\"passage_id\\"]) in qrels.get(cid, set())), None)\\n        if deeper:\\n            chosen.append(deeper)\\n        rng.shuffle(chosen)  # Hide rank and URL-match status from the annotator.\\n        for item in chosen:\\n            aid = f\\"audit_{len(exported):04d}\\"\\n            exported.append({\\"audit_id\\": aid, \\"claim\\": by_id[cid][\\"claim\\"], \\"source_url\\": item.get(\\"url\\"),\\n                             \\"passage_text\\": item[\\"text\\"], \\"relevance\\": \\"\\", \\"notes\\": \\"\\"})\\n            key.append({\\"audit_id\\": aid, \\"claim_id\\": cid, \\"passage_id\\": item[\\"passage_id\\"],\\n                        \\"proxy_match\\": str(item[\\"passage_id\\"]) in qrels.get(cid, set()), \\"gold_label\\": by_id[cid][\\"label\\"]})\\n    path = output / \\"manual_passage_audit.csv\\"\\n    if not path.exists():  # Preserve any judgments entered by the user on a rerun.\\n        with path.open(\\"w\\", encoding=\\"utf-8-sig\\", newline=\\"\\") as handle:\\n            writer = csv.DictWriter(handle, fieldnames=[\\"audit_id\\", \\"claim\\", \\"source_url\\", \\"passage_text\\", \\"relevance\\", \\"notes\\"])\\n            writer.writeheader()\\n            writer.writerows(exported)\\n    write_json({\\"rows\\": key, \\"sampling\\": \\"Balanced diagnostic strata, not a representative prevalence estimate.\\"}, output / \\"manual_audit_key.json\\")\\n\\n\\ndef prepare(args: argparse.Namespace) -> None:\\n    from transformers import AutoTokenizer\\n\\n    out = args.output\\n    out.mkdir(parents=True, exist_ok=True)\\n    inputs = {\\"train\\": args.train_claims, \\"dev\\": args.dev_claims, \\"train_rankings\\": args.train_rankings,\\n              \\"dev_rankings\\": args.dev_rankings, \\"qrels\\": args.qrels}\\n    if args.dev_top100:\\n        inputs[\\"dev_top100\\"] = args.dev_top100\\n    extras = {}\\n    for spec in args.extra:\\n        name, value = spec.split(\\"=\\", 1)\\n        if not re.fullmatch(r\\"[a-z0-9_]+\\", name) or name in extras:\\n            raise ValueError(\\"Extra ranking names must be unique lowercase letters/numbers/underscores\\")\\n        extras[name] = Path(value)\\n        inputs[f\\"extra_{name}\\"] = Path(value)\\n    config = {\\"version\\": VERSION, \\"inputs\\": {k: sha256_file(v) for k, v in inputs.items()},\\n              \\"tokenizer_config_sha256\\": sha256_file(args.model_dir / \\"config.json\\"),\\n              \\"budget\\": args.budget, \\"audit_size\\": args.audit_size, \\"seed\\": args.seed}\\n    lock_config(out / \\"prepare_config.json\\", config)\\n    tokenizer = AutoTokenizer.from_pretrained(args.model_dir, local_files_only=True, use_fast=True)\\n    records = {\\"train\\": read_jsonl(args.train_claims), \\"dev\\": read_jsonl(args.dev_claims)}\\n    for rows in records.values():\\n        index_rows(rows)\\n        if any(not str(r.get(\\"claim\\") or \\"\\").strip() or r.get(\\"label\\") not in CANONICAL_LABELS for r in rows):\\n            raise ValueError(\\"Missing claim text or invalid verdict label\\")\\n    if set(index_rows(records[\\"train\\"])) & set(index_rows(records[\\"dev\\"])):\\n        raise ValueError(\\"Train/dev claim IDs overlap\\")\\n    rankings = {\\"train\\": load_rankings(args.train_rankings, records[\\"train\\"]), \\"dev\\": load_rankings(args.dev_rankings, records[\\"dev\\"])}\\n    pools = load_rankings(args.dev_top100, records[\\"dev\\"]) if args.dev_top100 else rankings[\\"dev\\"]\\n    if args.dev_top100:\\n        for cid, row in pools.items():\\n            old = rankings[\\"dev\\"][cid][\\"retrieved\\"][:3]\\n            new = row[\\"retrieved\\"][:3]\\n            if [(p[\\"passage_id\\"], p[\\"text\\"]) for p in old] != [(p[\\"passage_id\\"], p[\\"text\\"]) for p in new]:\\n                raise ValueError(f\\"Top-100 and baseline top-three inputs differ: {cid}\\")\\n    qrels = {str(r[\\"claim_id\\"]): set(map(str, r.get(\\"relevant_passage_ids\\") or [])) for r in read_jsonl(args.qrels)}\\n    qrels = {cid: ids for cid, ids in qrels.items() if ids}\\n    for split, rows in records.items():\\n        generated = {mode: [] for mode in (\\"head\\", \\"sentences\\", \\"diverse_sentences\\", \\"claim_only\\")}\\n        with progress(total=len(rows), description=f\\"Prepare {split}\\", unit=\\"claim\\") as bar:\\n            for row in rows:\\n                cid = str(row[\\"claim_id\\"])\\n                candidates = rankings[split][cid][\\"retrieved\\"]\\n                for mode in generated:\\n                    generated[mode].append(prediction_variant(cid, row[\\"claim\\"], candidates, mode, tokenizer, args.budget))\\n                bar.update(1)\\n        for mode, materialized in generated.items():\\n            atomic_rows(materialized, out / \\"prepared\\" / mode / f\\"{split}_rankings.jsonl\\")\\n    # Gold selection is diagnostic only. No oracle train rankings are created.\\n    oracles = {\\"oracle_url_head\\": [], \\"oracle_url_sentences\\": [], \\"oracle_answers\\": []}\\n    audits = []\\n    for row in records[\\"dev\\"]:\\n        cid = str(row[\\"claim_id\\"])\\n        candidates = pools[cid][\\"retrieved\\"]\\n        relevant = qrels.get(cid, set())\\n        selected = [p for p in candidates if str(p[\\"passage_id\\"]) in relevant][:3]\\n        selected += [p for p in candidates if p not in selected][:3 - len(selected)]\\n        for name, mode in ((\\"oracle_url_head\\", \\"head\\"), (\\"oracle_url_sentences\\", \\"sentences\\")):\\n            oracles[name].append(prediction_variant(cid, row[\\"claim\\"], selected, mode, tokenizer, args.budget))\\n        answers = [p for p in row.get(\\"evidence\\", []) if str(p.get(\\"text\\") or \\"\\").strip()][:3]\\n        oracles[\\"oracle_answers\\"].append(prediction_variant(cid, row[\\"claim\\"], answers, \\"head\\", tokenizer, args.budget))\\n        audits.append(audit_claim(row, candidates, relevant, tokenizer, args.budget))\\n    for name, rows in oracles.items():\\n        atomic_rows(rows, out / \\"oracle\\" / name / \\"dev_rankings.jsonl\\")\\n    for name, path in extras.items():\\n        found = load_rankings(path, records[\\"dev\\"])\\n        rows = [prediction_variant(r[\\"claim_id\\"], r[\\"claim\\"], found[r[\\"claim_id\\"]][\\"retrieved\\"], \\"head\\", tokenizer, args.budget) for r in records[\\"dev\\"]]\\n        atomic_rows(rows, out / \\"prepared\\" / name / \\"dev_rankings.jsonl\\")\\n    atomic_rows(audits, out / \\"claim_audit.jsonl\\")\\n    export_manual_audit(records[\\"dev\\"], pools, qrels, audits, out, args.audit_size, args.seed)\\n    report = {\\"label_counts\\": {s: dict(Counter(r[\\"label\\"] for r in rows)) for s, rows in records.items()},\\n              \\"dev_claim_count\\": len(audits), \\"judged_dev_claim_count\\": sum(r[\\"judged\\"] for r in audits),\\n              \\"candidate_depth_range\\": [min(len(r[\\"retrieved\\"]) for r in pools.values()), max(len(r[\\"retrieved\\"]) for r in pools.values())],\\n              \\"dev_top100_supplied\\": bool(args.dev_top100), \\"candidate_coverage\\": {},\\n              \\"passages_over_budget\\": sum(r[\\"passages_over_budget\\"] for r in audits),\\n              \\"passages_measured\\": sum(r[\\"top3_passage_count\\"] for r in audits),\\n              \\"claims_with_repeated_top3_source\\": sum(r[\\"top3_distinct_sources\\"] < r[\\"top3_passage_count\\"] for r in audits),\\n              \\"notes\\": [\\"URL matches do not establish decisive passage relevance.\\",\\n                        \\"Oracle URL selection uses dev qrels and must never be used for model selection or deployment.\\",\\n                        \\"The supplied per-claim candidate pools simplify deployment; global search and publication-time availability are not tested.\\"]}\\n    judged = [r for r in audits if r[\\"judged\\"]]\\n    for k in (3, 20, 100):\\n        if k == 100 and not args.dev_top100:\\n            continue\\n        report[\\"candidate_coverage\\"][str(k)] = sum(r[f\\"url_hit_at_available_{k}\\"] for r in judged) / len(judged) if judged else None\\n    # Exact normalization checks surface duplicated claims; they do not prove\\n    # source independence or eliminate foundation-model pretraining exposure.\\n    normalized_train = defaultdict(list)\\n    for r in records[\\"train\\"]:\\n        normalized_train[\\" \\".join(re.findall(r\\"\\\\w+\\", r[\\"claim\\"].casefold()))].append(r[\\"claim_id\\"])\\n    overlaps = [{\\"dev_claim_id\\": r[\\"claim_id\\"], \\"train_claim_ids\\": normalized_train[key]}\\n                for r in records[\\"dev\\"] if (key := \\" \\".join(re.findall(r\\"\\\\w+\\", r[\\"claim\\"].casefold()))) in normalized_train]\\n    report[\\"normalized_exact_train_dev_claim_overlaps\\"] = overlaps\\n    report[\\"claim_date_coverage\\"] = {split: sum(parse_date(r.get(\\"metadata\\", {}).get(\\"claim_date\\")) is not None for r in rows) for split, rows in records.items()}\\n    report[\\"truncation\\"] = {key: float(np.mean(values)) if values else None for key in\\n                            (\\"head_query_term_retention\\", \\"excerpt_query_term_retention\\")\\n                            if (values := [r[key] for r in audits if r[key] is not None])}\\n    report[\\"coverage_by_gold_label\\"] = {}\\n    for label in CANONICAL_LABELS:\\n        group = [r for r in audits if r[\\"label\\"] == label]\\n        judged_group = [r for r in group if r[\\"judged\\"]]\\n        report[\\"coverage_by_gold_label\\"][label] = {\\n            \\"all_claims\\": len(group), \\"judged_claims\\": len(judged_group),\\n            \\"hits_at_3\\": sum(r[\\"url_hit_at_available_3\\"] for r in judged_group),\\n            \\"hits_in_saved_pool\\": sum(r[\\"url_hit_at_available_100\\"] for r in judged_group)}\\n    for name in (\\"head\\", \\"sentences\\", \\"diverse_sentences\\", *extras):\\n        rows = read_jsonl(out / \\"prepared\\" / name / \\"dev_rankings.jsonl\\")\\n        ids = {r[\\"claim_id\\"]: [p[\\"passage_id\\"] for p in r[\\"retrieved\\"]] for r in rows}\\n        report.setdefault(\\"top3_retrieval\\", {})[name] = evaluate_rankings(ids, qrels, k=3) if judged else None\\n    write_json(report, out / \\"data_audit.json\\")\\n    print(json.dumps(report, indent=2))\\n\\n\\ndef checked_predictions(path: Path, records: list[dict]) -> list[dict]:\\n    rows = read_jsonl(path)\\n    found = index_rows(rows, expected={str(r[\\"claim_id\\"]) for r in records})\\n    for r in records:\\n        pred = found[str(r[\\"claim_id\\"])]\\n        values = [float(pred.get(\\"probabilities\\", {}).get(label, float(\\"nan\\"))) for label in CANONICAL_LABELS]\\n        if pred[\\"true_label\\"] != r[\\"label\\"] or pred[\\"predicted_label\\"] not in CANONICAL_LABELS:\\n            raise ValueError(f\\"Prediction label mismatch: {r[\'claim_id\']}\\")\\n        if any(not math.isfinite(v) or not 0 <= v <= 1 for v in values) or abs(sum(values) - 1) > 1e-4:\\n            raise ValueError(f\\"Invalid probability vector: {r[\'claim_id\']}\\")\\n    return [found[str(r[\\"claim_id\\"])] for r in records]\\n\\n\\ndef frozen(args: argparse.Namespace) -> None:\\n    import torch\\n    from transformers import AutoModelForSequenceClassification, AutoTokenizer\\n    from scripts.train_transformer_verifier import compose_text\\n\\n    if args.device.startswith(\\"cuda\\") and not torch.cuda.is_available():\\n        raise RuntimeError(\\"Select a Colab GPU runtime before frozen inference\\")\\n    records = read_jsonl(args.dev_claims)\\n    settings = json.loads(args.model_config.read_text(encoding=\\"utf-8\\"))\\n    base = load_rankings(args.output / \\"prepared/head/dev_rankings.jsonl\\", records)\\n    attached = [{**r, \\"retrieved_evidence\\": base[r[\\"claim_id\\"]][\\"retrieved\\"]} for r in records]\\n    interventions = evidence_variants(attached)\\n    conditions = {\\"head\\": attached, \\"missing\\": interventions[\\"blank\\"], \\"wrong_claim\\": interventions[\\"swapped\\"],\\n                  \\"reverse_order\\": [{**r, \\"retrieved_evidence\\": list(reversed(r[\\"retrieved_evidence\\"]))} for r in attached]}\\n    paths = {}\\n    for directory in (args.output / \\"prepared\\", args.output / \\"oracle\\"):\\n        for path in sorted(directory.glob(\\"*/dev_rankings.jsonl\\")):\\n            if path.parent.name in {\\"head\\", \\"claim_only\\"}:\\n                continue\\n            found = load_rankings(path, records)\\n            conditions[path.parent.name] = [{**r, \\"retrieved_evidence\\": found[r[\\"claim_id\\"]][\\"retrieved\\"]} for r in records]\\n            paths[path.parent.name] = sha256_file(path)\\n    weights = sorted([*args.model_dir.glob(\\"*.safetensors\\"), *args.model_dir.glob(\\"pytorch_model*.bin\\")])\\n    if not weights:\\n        raise FileNotFoundError(f\\"No saved model weights in {args.model_dir}\\")\\n    config = {\\"version\\": VERSION, \\"dev_sha256\\": sha256_file(args.dev_claims),\\n              \\"model_config_sha256\\": sha256_file(args.model_dir / \\"config.json\\"),\\n              \\"weights\\": {p.name: sha256_file(p) for p in weights}, \\"settings\\": settings,\\n              \\"prepared_config_sha256\\": sha256_file(args.output / \\"prepare_config.json\\"),\\n              \\"variant_sha256\\": paths, \\"batch_size\\": args.batch_size, \\"device\\": args.device}\\n    folder = args.output / \\"frozen\\"\\n    folder.mkdir(parents=True, exist_ok=True)\\n    lock_config(folder / \\"config.json\\", config)\\n    tokenizer = AutoTokenizer.from_pretrained(args.model_dir, local_files_only=True, use_fast=True)\\n    model = AutoModelForSequenceClassification.from_pretrained(args.model_dir, local_files_only=True).to(args.device).eval()\\n    if [model.config.id2label.get(i) for i in range(4)] != list(CANONICAL_LABELS):\\n        raise ValueError(\\"Saved model label order differs from the evaluator\\")\\n    for name, rows in conditions.items():\\n        run = folder / name\\n        run.mkdir(parents=True, exist_ok=True)\\n        final, partial = run / \\"predictions.jsonl\\", run / \\"predictions.partial.jsonl\\"\\n        if final.exists():\\n            predictions = checked_predictions(final, rows)\\n        else:\\n            done = load_partial_predictions(partial, {str(r[\\"claim_id\\"]) for r in rows})\\n            expected_labels = {r[\\"claim_id\\"]: r[\\"label\\"] for r in rows}\\n            if any(p[\\"true_label\\"] != expected_labels[cid] for cid, p in done.items()):\\n                raise ValueError(\\"Partial checkpoint gold labels differ\\")\\n            pending = [r for r in rows if r[\\"claim_id\\"] not in done]\\n            with progress(total=len(rows), initial=len(done), description=f\\"Frozen {name}\\", unit=\\"claim\\") as bar:\\n                for start in range(0, len(pending), args.batch_size):\\n                    batch = pending[start:start + args.batch_size]\\n                    texts = [compose_text(r, tokenizer, int(settings[\\"claim_token_budget\\"]), int(settings[\\"evidence_token_budget_per_passage\\"])) for r in batch]\\n                    encoded = tokenizer(texts, padding=True, truncation=True, max_length=int(settings[\\"max_length\\"]), return_tensors=\\"pt\\")\\n                    with torch.inference_mode():\\n                        values = model(**{k: v.to(args.device) for k, v in encoded.items()}).logits.float().softmax(-1).cpu().tolist()\\n                    new = []\\n                    for r, vector in zip(batch, values, strict=True):\\n                        p = {\\"claim_id\\": r[\\"claim_id\\"], \\"true_label\\": r[\\"label\\"],\\n                             \\"predicted_label\\": CANONICAL_LABELS[int(np.argmax(vector))],\\n                             \\"probabilities\\": dict(zip(CANONICAL_LABELS, vector, strict=True)),\\n                             \\"evidence_passage_ids\\": [p[\\"passage_id\\"] for p in r[\\"retrieved_evidence\\"][:3]]}\\n                        done[r[\\"claim_id\\"]] = p\\n                        new.append(p)\\n                    # Commit every GPU batch; an interruption loses at most one batch.\\n                    with partial.open(\\"a\\", encoding=\\"utf-8\\", newline=\\"\\\\n\\") as handle:\\n                        for p in new:\\n                            handle.write(json.dumps(p, ensure_ascii=False) + \\"\\\\n\\")\\n                        handle.flush()\\n                    bar.update(len(new))\\n            atomic_rows([done[r[\\"claim_id\\"]] for r in rows], final)\\n            predictions = checked_predictions(final, rows)\\n        write_json({**evaluate_predictions(predictions), \\"oracle_only\\": name.startswith(\\"oracle_\\"),\\n                    \\"interpretation\\": \\"Frozen-model input intervention; training distribution unchanged.\\"}, run / \\"metrics.json\\")\\n        print(name, \\"macro-F1\\", round(evaluate_predictions(predictions)[\\"macro_f1\\"], 4), flush=True)\\n    if args.reference_predictions:\\n        old = checked_predictions(args.reference_predictions, records)\\n        now = checked_predictions(folder / \\"head/predictions.jsonl\\", records)\\n        write_json(comparison(old, now), folder / \\"baseline_reproduction.json\\")\\n\\n\\ndef paired_bootstrap(original: list[dict], altered: list[dict], trials: int = 1000, seed: int = 718) -> dict:\\n    \\"\\"\\"Paired, class-stratified intervals: conditional on this dev label mix.\\"\\"\\"\\n    old = index_rows(original)\\n    new = index_rows(altered, expected=set(old))\\n    ids = sorted(old)\\n    if any(old[cid][\\"true_label\\"] != new[cid][\\"true_label\\"] for cid in ids):\\n        raise ValueError(\\"Paired predictions disagree on gold labels\\")\\n    labels = {label: i for i, label in enumerate(CANONICAL_LABELS)}\\n    y = np.array([labels[old[c][\\"true_label\\"]] for c in ids])\\n    a = np.array([labels[old[c][\\"predicted_label\\"]] for c in ids])\\n    b = np.array([labels[new[c][\\"predicted_label\\"]] for c in ids])\\n\\n    def metrics(gold, pred):\\n        mat = np.bincount(gold * 4 + pred, minlength=16).reshape(4, 4)\\n        den = mat.sum(0) + mat.sum(1)\\n        f1 = np.divide(2 * mat.diagonal(), den, out=np.zeros(4), where=den != 0).mean()\\n        return np.array([f1, (gold == pred).mean()])\\n\\n    rng = np.random.default_rng(seed)\\n    groups = [np.flatnonzero(y == i) for i in range(4)]\\n    delta = []\\n    for _ in range(trials):\\n        indices = np.concatenate([rng.choice(g, size=len(g), replace=True) for g in groups if len(g)])\\n        delta.append(metrics(y[indices], b[indices]) - metrics(y[indices], a[indices]))\\n    bounds = np.quantile(np.asarray(delta), [0.025, 0.975], axis=0)\\n    return {\\"trials\\": trials, \\"method\\": \\"paired class-stratified bootstrap; conditional on dev class counts; training-seed variability excluded\\",\\n            \\"delta_macro_f1\\": float(metrics(y, b)[0] - metrics(y, a)[0]),\\n            \\"delta_macro_f1_95pct\\": bounds[:, 0].tolist(), \\"delta_accuracy_95pct\\": bounds[:, 1].tolist(),\\n            **comparison(original, altered)}\\n\\n\\ndef summarize(args: argparse.Namespace) -> None:\\n    out = args.output\\n    records = read_jsonl(args.dev_claims)\\n    reports, predictions, kinds = {}, {}, {}\\n    for path in sorted((out / \\"frozen\\").glob(\\"*/predictions.jsonl\\")):\\n        name = \\"frozen/\\" + path.parent.name\\n        predictions[name] = checked_predictions(path, records)\\n        kinds[name] = \\"oracle diagnostic\\" if path.parent.name.startswith(\\"oracle_\\") else \\"frozen input diagnostic\\"\\n    for path in sorted((out / \\"training\\").glob(\\"*/final_dev_predictions.jsonl\\")):\\n        name = \\"trained/\\" + path.parent.name\\n        predictions[name] = checked_predictions(path, records)\\n        kinds[name] = \\"matched train/dev pipeline\\"\\n    for name, rows in predictions.items():\\n        reports[name] = {**evaluate_predictions(rows), \\"kind\\": kinds[name]}\\n        n_non = sum(r[\\"true_label\\"] != \\"Supported\\" for r in rows)\\n        n_sup = len(rows) - n_non\\n        rates = reports[name][\\"decision_error_rates\\"]\\n        reports[name][\\"conditional_errors\\"] = {\\"fr_over_non_supported\\": rates[\\"false_reassurance_count\\"] / n_non if n_non else None,\\n                                               \\"fa_over_supported\\": rates[\\"false_alarm_count\\"] / n_sup if n_sup else None,\\n                                               \\"non_supported_count\\": n_non, \\"supported_count\\": n_sup}\\n    reference = predictions.get(\\"frozen/head\\")\\n    paired = {name: paired_bootstrap(reference, rows, args.bootstrap_trials) for name, rows in predictions.items()\\n              if reference is not None and name != \\"frozen/head\\"}\\n    # Compare trained interventions on train holdout and dev; no dev-based winner.\\n    validation = {}\\n    for path in sorted((out / \\"training\\").glob(\\"*/model_selection.json\\")):\\n        full_selection = json.loads(path.read_text(encoding=\\"utf-8\\"))\\n        validation[path.parent.name] = {**full_selection[\\"selected_trial\\"], **{\\n            k: full_selection.get(k) for k in (\\"validation_policy\\", \\"validation_seed\\", \\"training_seed\\", \\"selection_train_claim_count\\")}}\\n    audits = {r[\\"claim_id\\"]: r for r in read_jsonl(out / \\"claim_audit.jsonl\\")}\\n    subsets = {}\\n    for name, rows in predictions.items():\\n        groups = defaultdict(list)\\n        for p in rows:\\n            row = audits[p[\\"claim_id\\"]]\\n            group = \\"unjudged\\" if not row[\\"judged\\"] else \\"url_top3_hit\\" if row[\\"url_hit_at_available_3\\"] else \\"url_top3_miss\\"\\n            groups[group].append(p)\\n            meta = next(r for r in records if r[\\"claim_id\\"] == p[\\"claim_id\\"]).get(\\"metadata\\", {})\\n            date = row.get(\\"claim_date\\")\\n            if date:\\n                groups[\\"month/\\" + date[:7]].append(p)\\n            location = meta.get(\\"location_iso_code\\")\\n            if isinstance(location, str) and location:\\n                groups[\\"location/\\" + location].append(p)\\n        subsets[name] = {g: evaluate_predictions(items) for g, items in groups.items() if len(items) >= 20}\\n    report = {\\"results\\": reports, \\"paired_vs_frozen_head\\": paired, \\"train_holdout_selection\\": validation,\\n              \\"subgroups_minimum_n_20\\": subsets, \\"cautions\\": [\\n                  \\"Dev was inspected repeatedly. All dev findings are exploratory; reserve untouched data for the frozen final pipeline.\\",\\n                  \\"Bootstrap intervals do not account for repeated dev inspection, related claims, training seeds, or pretraining exposure.\\",\\n                  \\"Oracle URL matches do not certify decisive evidence. Gold-answer scores are not formal ceilings.\\",\\n                  \\"Subset differences are descriptive and do not identify causal effects.\\"]}\\n    report[\\"planned_contrasts\\"] = {}\\n    silver_path = out / \\"silver_supervision_audit.json\\"\\n    if silver_path.is_file():\\n        report[\\"silver_supervision_audit\\"] = json.loads(silver_path.read_text(encoding=\\"utf-8\\"))\\n    for a, b in ((\\"frozen/sentences\\", \\"frozen/diverse_sentences\\"),\\n                 (\\"trained/head_balanced\\", \\"trained/head_unweighted\\"),\\n                 (\\"trained/head_balanced\\", \\"trained/sentences_balanced\\"),\\n                 (\\"trained/head_unweighted\\", \\"trained/sentences_unweighted\\"),\\n                 (\\"trained/sentences_balanced\\", \\"trained/diverse_sentences_balanced\\"),\\n                 (\\"trained/claim_only_balanced\\", \\"trained/claim_only_unweighted\\"),\\n                 (\\"trained/head_balanced\\", \\"trained/head_balanced_requested_epochs\\")):\\n        if a in predictions and b in predictions:\\n            report[\\"planned_contrasts\\"][a + \\" -> \\" + b] = paired_bootstrap(predictions[a], predictions[b], args.bootstrap_trials)\\n    audit_path = out / \\"manual_passage_audit.csv\\"\\n    if audit_path.exists():\\n        allowed = {\\"decisive_support\\", \\"decisive_refute\\", \\"partial\\", \\"irrelevant\\", \\"unclear\\"}\\n        key = {r[\\"audit_id\\"]: r for r in json.loads((out / \\"manual_audit_key.json\\").read_text(encoding=\\"utf-8\\"))[\\"rows\\"]}\\n        counts = defaultdict(Counter)\\n        with audit_path.open(encoding=\\"utf-8-sig\\", newline=\\"\\") as handle:\\n            for row in csv.DictReader(handle):\\n                value = row[\\"relevance\\"].strip().lower()\\n                if not value:\\n                    continue\\n                if value not in allowed or row[\\"audit_id\\"] not in key:\\n                    raise ValueError(f\\"Invalid manual audit judgment: {row[\'audit_id\']}, {value}\\")\\n                group = \\"proxy_match\\" if key[row[\\"audit_id\\"]][\\"proxy_match\\"] else \\"no_proxy_match\\"\\n                counts[group][value] += 1\\n        report[\\"manual_audit\\"] = {\\"counts\\": {g: dict(v) for g, v in counts.items()},\\n                                  \\"warning\\": \\"Diagnostic sample balances strata; these counts do not estimate population relevance prevalence.\\"}\\n    write_json(report, out / \\"limitation_results.json\\")\\n    lines = [\\"# Limitation study results\\", \\"\\", \\"All verdict rows use the same official dev claims. FR and FA are counts.\\", \\"\\",\\n             \\"| Condition | Macro-F1 | Accuracy | FR | FA | Kind |\\", \\"| --- | ---: | ---: | ---: | ---: | --- |\\"]\\n    for name, m in reports.items():\\n        rates = m[\\"decision_error_rates\\"]\\n        lines.append(f\\"| {name} | {m[\'macro_f1\']:.4f} | {m[\'accuracy\']:.4f} | {rates[\'false_reassurance_count\']} | {rates[\'false_alarm_count\']} | {m[\'kind\']} |\\")\\n    lines.extend([\\"\\", \\"## Paired changes versus the frozen BM25 baseline\\", \\"\\",\\n                  \\"| Condition | Delta macro-F1 | 95% interval | Fixed errors | Lost correct answers |\\", \\"| --- | ---: | --- | ---: | ---: |\\"])\\n    for name, p in paired.items():\\n        lo, hi = p[\\"delta_macro_f1_95pct\\"]\\n        lines.append(f\\"| {name} | {p[\'delta_macro_f1\']:+.4f} | [{lo:+.4f}, {hi:+.4f}] | {p[\'originally_wrong_now_correct\']} | {p[\'originally_correct_now_wrong\']} |\\")\\n    lines.extend([\\"\\", \\"## Reading the results\\", \\"\\",\\n                  \\"- Missing/wrong evidence barely changes predictions: investigate weak evidence use. Large changes alone do not prove good reasoning.\\",\\n                  \\"- URL-oracle raw passages help: candidate ranking likely matters; check decisive relevance manually before trusting this proxy.\\",\\n                  \\"- Oracle answers help but URL-oracle passages do not: inspect passage decisiveness, input format and reading ability.\\",\\n                  \\"- Same-passage sentence packing helps only after matched training: input distribution and token allocation matter.\\",\\n                  \\"- Unweighted loss changes minority recall and safety errors: class weighting is a tradeoff, not an automatic improvement.\\",\\n                  \\"- Holdout gains vanish on later dev: investigate temporal/source shift and training uncertainty.\\", \\"\\",\\n                  \\"Use train holdout to choose configurations. Keep dev diagnostic and the test split untouched.\\", \\"\\",\\n                  \\"See limitation_results.json for per-class metrics, confusion matrices, calibration, subgroups and full paired comparisons.\\"])\\n    (out / \\"limitation_results.md\\").write_text(\\"\\\\n\\".join(lines) + \\"\\\\n\\", encoding=\\"utf-8\\")\\n    print(\\"\\\\n\\".join(lines))\\n\\n\\ndef main() -> None:\\n    parser = argparse.ArgumentParser(description=__doc__)\\n    sub = parser.add_subparsers(dest=\\"command\\", required=True)\\n    prep = sub.add_parser(\\"prepare\\")\\n    for name in (\\"train-claims\\", \\"dev-claims\\", \\"train-rankings\\", \\"dev-rankings\\", \\"qrels\\", \\"model-dir\\", \\"output\\"):\\n        prep.add_argument(\\"--\\" + name, type=Path, required=True)\\n    prep.add_argument(\\"--dev-top100\\", type=Path)\\n    prep.add_argument(\\"--extra\\", action=\\"append\\", default=[], help=\\"Optional name=dev_rankings path, e.g. bge or fusion\\")\\n    prep.add_argument(\\"--budget\\", type=int, default=110)\\n    prep.add_argument(\\"--audit-size\\", type=int, default=24)\\n    prep.add_argument(\\"--seed\\", type=int, default=718)\\n    infer = sub.add_parser(\\"frozen\\")\\n    for name in (\\"dev-claims\\", \\"model-dir\\", \\"model-config\\", \\"output\\"):\\n        infer.add_argument(\\"--\\" + name, type=Path, required=True)\\n    infer.add_argument(\\"--reference-predictions\\", type=Path)\\n    infer.add_argument(\\"--device\\", default=\\"cuda\\")\\n    infer.add_argument(\\"--batch-size\\", type=int, default=16)\\n    summ = sub.add_parser(\\"summarize\\")\\n    summ.add_argument(\\"--output\\", type=Path, required=True)\\n    summ.add_argument(\\"--dev-claims\\", type=Path, required=True)\\n    summ.add_argument(\\"--bootstrap-trials\\", type=int, default=1000)\\n    args = parser.parse_args()\\n    for name in (\\"budget\\", \\"batch_size\\", \\"bootstrap_trials\\", \\"audit_size\\"):\\n        if hasattr(args, name) and getattr(args, name) < 1:\\n            parser.error(f\\"{name} must be positive\\")\\n    {\\"prepare\\": prepare, \\"frozen\\": frozen, \\"summarize\\": summarize}[args.command](args)\\n\\n\\nif __name__ == \\"__main__\\":\\n    main()\\n", "scripts/train_tfidf_logreg.py": "\\"\\"\\"Train and evaluate the TF-IDF + logistic-regression verifier baseline.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport argparse\\nimport json\\nimport sys\\nfrom pathlib import Path\\nfrom typing import Any\\n\\nimport joblib\\n\\nPROJECT_ROOT = Path(__file__).resolve().parents[1]\\nsys.path.insert(0, str(PROJECT_ROOT))\\n\\nfrom src.evaluation.classification import evaluate_predictions  # noqa: E402\\nfrom src.evaluation.end_to_end import annotate_retrieval_coverage, evaluate_by_retrieval_coverage  # noqa: E402\\nfrom src.verification.tfidf_logreg import TfidfLogRegVerifier  # noqa: E402\\nfrom src.utils.io import read_jsonl, write_jsonl  # noqa: E402\\nfrom src.utils.runs import resolve_run_directory, write_run_manifest  # noqa: E402\\n\\n\\ndef attach_retrieved_evidence(records: list[dict[str, Any]], rankings_path: Path) -> list[dict[str, Any]]:\\n    rankings = {str(row[\\"claim_id\\"]): row.get(\\"retrieved\\") or [] for row in read_jsonl(rankings_path)}\\n    missing = [record[\\"claim_id\\"] for record in records if record[\\"claim_id\\"] not in rankings]\\n    if missing:\\n        raise ValueError(f\\"Rankings are missing {len(missing)} records, e.g. {missing[:3]}\\")\\n    return [{**record, \\"retrieved_evidence\\": rankings[str(record[\\"claim_id\\"])]} for record in records]\\n\\n\\ndef main() -> None:\\n    parser = argparse.ArgumentParser()\\n    parser.add_argument(\\"--evidence-mode\\", choices=(\\"claim_only\\", \\"gold\\", \\"gold_all\\", \\"retrieved\\"), default=\\"gold\\")\\n    parser.add_argument(\\"--data-root\\", type=Path, default=PROJECT_ROOT / \\"data\\")\\n    parser.add_argument(\\"--train-split\\", default=\\"train\\")\\n    parser.add_argument(\\"--eval-split\\", default=\\"dev\\")\\n    parser.add_argument(\\"--output-dir\\", type=Path, default=PROJECT_ROOT / \\"outputs\\" / \\"tfidf_logreg\\")\\n    parser.add_argument(\\"--run-name\\", help=\\"Optional name for an isolated, non-overwriting run subdirectory.\\")\\n    parser.add_argument(\\"--random-state\\", type=int, default=718)\\n    parser.add_argument(\\"--min-df\\", type=int, default=1)\\n    parser.add_argument(\\"--max-features\\", type=int)\\n    parser.add_argument(\\"--max-evidence\\", type=int, default=3)\\n    parser.add_argument(\\"--train-rankings\\", type=Path, help=\\"Required for --evidence-mode retrieved.\\")\\n    parser.add_argument(\\"--eval-rankings\\", type=Path, help=\\"Required for --evidence-mode retrieved.\\")\\n    parser.add_argument(\\"--eval-qrels\\", type=Path, help=\\"Defaults to <data-root>/processed/averitec/<eval-split>_qrels.jsonl in retrieved mode.\\")\\n    parser.add_argument(\\"--unbalanced\\", action=\\"store_true\\", help=\\"Disable class-balanced logistic-regression weights.\\")\\n    args = parser.parse_args()\\n    try:\\n        args.output_dir = resolve_run_directory(args.output_dir, args.run_name)\\n    except ValueError as error:\\n        parser.error(str(error))\\n\\n    processed = args.data_root / \\"processed\\" / \\"averitec\\"\\n    train_records = read_jsonl(processed / f\\"{args.train_split}.jsonl\\")\\n    eval_records = read_jsonl(processed / f\\"{args.eval_split}.jsonl\\")\\n    if args.evidence_mode == \\"retrieved\\":\\n        if not args.train_rankings or not args.eval_rankings:\\n            parser.error(\\"--train-rankings and --eval-rankings are required for retrieved evidence.\\")\\n        train_records = attach_retrieved_evidence(train_records, args.train_rankings)\\n        eval_records = attach_retrieved_evidence(eval_records, args.eval_rankings)\\n    verifier = TfidfLogRegVerifier(\\n        evidence_mode=args.evidence_mode,\\n        max_evidence=args.max_evidence,\\n        min_df=args.min_df,\\n        max_features=args.max_features,\\n        class_weight=None if args.unbalanced else \\"balanced\\",\\n        random_state=args.random_state,\\n    ).fit(train_records)\\n    predictions = verifier.predict_records(eval_records)\\n    metrics = evaluate_predictions(predictions)\\n    run_config = {\\n        \\"model\\": \\"tfidf_logreg\\",\\n        \\"evidence_mode\\": args.evidence_mode,\\n        \\"train_split\\": args.train_split,\\n        \\"eval_split\\": args.eval_split,\\n        \\"random_state\\": args.random_state,\\n        \\"min_df\\": args.min_df,\\n        \\"max_features\\": args.max_features,\\n        \\"max_evidence\\": args.max_evidence,\\n        \\"run_name\\": args.run_name,\\n        \\"class_weight\\": None if args.unbalanced else \\"balanced\\",\\n        \\"warning\\": (\\n            \\"gold mode is an isolated verifier experiment capped at three annotated answers. It must not be presented as an \\"\\n            \\"end-to-end retrieval result.\\"\\n            if args.evidence_mode == \\"gold\\"\\n            else \\"gold_all is an oracle upper bound with no three-item cap.\\" if args.evidence_mode == \\"gold_all\\" else None\\n        ),\\n    }\\n    args.output_dir.mkdir(parents=True, exist_ok=True)\\n    prefix = f\\"{args.evidence_mode}_{args.train_split}_to_{args.eval_split}\\"\\n    write_jsonl(predictions, args.output_dir / f\\"{prefix}_predictions.jsonl\\")\\n    (args.output_dir / f\\"{prefix}_metrics.json\\").write_text(json.dumps(metrics, indent=2) + \\"\\\\n\\", encoding=\\"utf-8\\")\\n    conditioned_input_paths: dict[str, Path] = {}\\n    if args.evidence_mode == \\"retrieved\\":\\n        qrels_path = args.eval_qrels or processed / f\\"{args.eval_split}_qrels.jsonl\\"\\n        if not qrels_path.is_file():\\n            raise FileNotFoundError(f\\"Retrieved-evidence diagnostics require qrels at {qrels_path}\\")\\n        qrels = {\\n            str(row[\\"claim_id\\"]): [str(value) for value in row.get(\\"relevant_passage_ids\\") or []]\\n            for row in read_jsonl(qrels_path)\\n            if row.get(\\"relevant_passage_ids\\")\\n        }\\n        eval_rankings = read_jsonl(args.eval_rankings)\\n        coverage_path = args.output_dir / f\\"{prefix}_evidence_coverage.jsonl\\"\\n        conditioned_metrics_path = args.output_dir / f\\"{prefix}_retrieval_conditioned_metrics.json\\"\\n        write_jsonl(annotate_retrieval_coverage(predictions, eval_rankings, qrels, k=args.max_evidence), coverage_path)\\n        conditioned_metrics_path.write_text(\\n            json.dumps(evaluate_by_retrieval_coverage(predictions, eval_rankings, qrels, k=args.max_evidence), indent=2) + \\"\\\\n\\",\\n            encoding=\\"utf-8\\",\\n        )\\n        run_config[\\"artifacts\\"] = {\\n            \\"evidence_coverage\\": str(coverage_path),\\n            \\"retrieval_conditioned_metrics\\": str(conditioned_metrics_path),\\n        }\\n        conditioned_input_paths[\\"eval_qrels\\"] = qrels_path\\n    (args.output_dir / f\\"{prefix}_config.json\\").write_text(json.dumps(run_config, indent=2) + \\"\\\\n\\", encoding=\\"utf-8\\")\\n    joblib.dump(verifier, args.output_dir / f\\"{prefix}_model.joblib\\")\\n    write_run_manifest(\\n        args.output_dir / f\\"{prefix}_manifest.json\\",\\n        project_root=PROJECT_ROOT,\\n        config=run_config,\\n        input_paths={\\n            \\"train_records\\": processed / f\\"{args.train_split}.jsonl\\",\\n            \\"eval_records\\": processed / f\\"{args.eval_split}.jsonl\\",\\n            **({\\"train_rankings\\": args.train_rankings, \\"eval_rankings\\": args.eval_rankings} if args.evidence_mode == \\"retrieved\\" else {}),\\n            **conditioned_input_paths,\\n        },\\n    )\\n    print(json.dumps({**run_config, **metrics}, indent=2))\\n\\n\\nif __name__ == \\"__main__\\":\\n    main()\\n", "scripts/train_transformer_verifier.py": "\\"\\"\\"Train a compact evidence-aware transformer with resumable tuning trials.\\n\\nModel selection uses a stratified holdout carved only from official train\\nclaims. The selected configuration is then refit on all official train claims\\nand evaluated once on the unchanged official development split.\\n\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport argparse\\nimport json\\nimport re\\nimport sys\\nfrom pathlib import Path\\nfrom typing import Any\\n\\nimport numpy as np\\nimport torch\\nfrom sklearn.metrics import accuracy_score, f1_score\\n\\nPROJECT_ROOT = Path(__file__).resolve().parents[1]\\nsys.path.insert(0, str(PROJECT_ROOT))\\n\\nfrom scripts.train_tfidf_logreg import attach_retrieved_evidence  # noqa: E402\\nfrom src.data.averitec import CANONICAL_LABELS  # noqa: E402\\nfrom src.evaluation.classification import evaluate_predictions  # noqa: E402\\nfrom src.evaluation.splits import selection_split  # noqa: E402\\nfrom src.utils.io import read_jsonl, write_json, write_jsonl  # noqa: E402\\nfrom src.utils.progress import progress  # noqa: E402\\nfrom src.utils.runs import resolve_run_directory, sha256_file, write_run_manifest  # noqa: E402\\n\\n\\nLABEL_TO_ID = {label: index for index, label in enumerate(CANONICAL_LABELS)}\\nID_TO_LABEL = {index: label for label, index in LABEL_TO_ID.items()}\\n\\n\\ndef parse_float_grid(value: str, name: str) -> list[float]:\\n    try:\\n        result = [float(item.strip()) for item in value.split(\\",\\") if item.strip()]\\n    except ValueError as error:\\n        raise argparse.ArgumentTypeError(f\\"{name} must be comma-separated numbers\\") from error\\n    if not result or any(item <= 0 for item in result):\\n        raise argparse.ArgumentTypeError(f\\"{name} values must be positive\\")\\n    return list(dict.fromkeys(result))\\n\\n\\ndef parse_int_grid(value: str, name: str) -> list[int]:\\n    try:\\n        result = [int(item.strip()) for item in value.split(\\",\\") if item.strip()]\\n    except ValueError as error:\\n        raise argparse.ArgumentTypeError(f\\"{name} must be comma-separated integers\\") from error\\n    if not result or any(item <= 0 for item in result):\\n        raise argparse.ArgumentTypeError(f\\"{name} values must be positive\\")\\n    return list(dict.fromkeys(result))\\n\\n\\ndef attach_rankings(records: list[dict[str, Any]], rankings_path: Path) -> list[dict[str, Any]]:\\n    return attach_retrieved_evidence(records, rankings_path)\\n\\n\\ndef compose_text(record: dict[str, Any], tokenizer: Any, claim_budget: int, evidence_budget: int) -> str:\\n    claim_tokens = tokenizer.encode(str(record[\\"claim\\"]), add_special_tokens=False)[:claim_budget]\\n    claim_text = tokenizer.decode(claim_tokens, skip_special_tokens=True)\\n    sections = [f\\"Claim: {claim_text}\\"]\\n    evidence = [item for item in (record.get(\\"retrieved_evidence\\") or [])[:3] if str(item.get(\\"text\\") or \\"\\").strip()]\\n    if not evidence:\\n        sections.append(\\"Evidence: No candidate passage was retrieved.\\")\\n    else:\\n        for index, item in enumerate(evidence, start=1):\\n            tokens = tokenizer.encode(str(item[\\"text\\"]), add_special_tokens=False)[:evidence_budget]\\n            text = tokenizer.decode(tokens, skip_special_tokens=True)\\n            sections.append(f\\"Evidence {index}: {text}\\")\\n    return \\"\\\\n\\".join(sections)\\n\\n\\nclass EncodedRecords(torch.utils.data.Dataset):\\n    def __init__(self, records: list[dict[str, Any]], tokenizer: Any, max_length: int, claim_budget: int, evidence_budget: int):\\n        self.records = records\\n        self.features = []\\n        tokenization_batch_size = 128\\n        with progress(total=len(records), description=\\"Tokenizing claims\\", unit=\\"claim\\", leave=False) as bar:\\n            for start in range(0, len(records), tokenization_batch_size):\\n                batch_records = records[start : start + tokenization_batch_size]\\n                texts = [compose_text(row, tokenizer, claim_budget, evidence_budget) for row in batch_records]\\n                encoded = tokenizer(\\n                    texts,\\n                    max_length=max_length,\\n                    truncation=True,\\n                    padding=False,\\n                    return_attention_mask=True,\\n                )\\n                for index, record in enumerate(batch_records):\\n                    feature = {key: values[index] for key, values in encoded.items()}\\n                    feature[\\"labels\\"] = LABEL_TO_ID[str(record[\\"label\\"])]\\n                    self.features.append(feature)\\n                bar.update(len(batch_records))\\n\\n    def __len__(self) -> int:\\n        return len(self.features)\\n\\n    def __getitem__(self, index: int) -> dict[str, Any]:\\n        return self.features[index]\\n\\n\\ndef prediction_metrics(eval_prediction: Any) -> dict[str, float]:\\n    logits = eval_prediction.predictions\\n    if isinstance(logits, tuple):\\n        logits = logits[0]\\n    gold = np.asarray(eval_prediction.label_ids)\\n    predicted = np.asarray(logits).argmax(axis=1)\\n    supported_id = LABEL_TO_ID[\\"Supported\\"]\\n    false_reassurance = np.logical_and(gold != supported_id, predicted == supported_id)\\n    false_alarm = np.logical_and(gold == supported_id, predicted != supported_id)\\n    return {\\n        \\"macro_f1\\": float(f1_score(gold, predicted, labels=list(range(len(CANONICAL_LABELS))), average=\\"macro\\", zero_division=0)),\\n        \\"accuracy\\": float(accuracy_score(gold, predicted)),\\n        \\"false_reassurance_rate\\": float(false_reassurance.mean()),\\n        \\"false_alarm_rate\\": float(false_alarm.mean()),\\n    }\\n\\n\\ndef safe_name(value: float) -> str:\\n    return re.sub(r\\"[^A-Za-z0-9]+\\", \\"p\\", f\\"{value:.0e}\\" if value < 0.001 else f\\"{value:g}\\")\\n\\n\\ndef read_json(path: Path) -> dict[str, Any] | None:\\n    if not path.is_file():\\n        return None\\n    return json.loads(path.read_text(encoding=\\"utf-8\\"))\\n\\n\\ndef best_checkpoint(trial_dir: Path) -> str | None:\\n    \\"\\"\\"Return the newest complete Trainer checkpoint, ignoring torn Drive writes.\\"\\"\\"\\n    required_state = (\\"trainer_state.json\\", \\"optimizer.pt\\", \\"scheduler.pt\\", \\"rng_state.pth\\")\\n    valid: list[tuple[int, Path]] = []\\n    for path in trial_dir.glob(\\"checkpoint-*\\"):\\n        try:\\n            step = int(path.name.removeprefix(\\"checkpoint-\\"))\\n        except ValueError:\\n            continue\\n        has_model = any(path.glob(\\"model*.safetensors\\")) or any(path.glob(\\"pytorch_model*.bin\\"))\\n        if has_model and all((path / name).is_file() for name in required_state):\\n            valid.append((step, path))\\n    return str(max(valid, default=(0, None), key=lambda pair: pair[0])[1]) if valid else None\\n\\n\\ndef main() -> None:\\n    parser = argparse.ArgumentParser()\\n    parser.add_argument(\\"--data-root\\", type=Path, required=True)\\n    parser.add_argument(\\"--train-rankings\\", required=True, type=Path)\\n    parser.add_argument(\\"--eval-rankings\\", required=True, type=Path)\\n    parser.add_argument(\\"--output-dir\\", type=Path, required=True)\\n    parser.add_argument(\\"--run-name\\", required=True)\\n    parser.add_argument(\\"--model\\", default=\\"distilbert-base-uncased\\")\\n    parser.add_argument(\\"--revision\\", help=\\"Optional immutable Hugging Face model revision.\\")\\n    parser.add_argument(\\"--max-length\\", type=int, default=512)\\n    parser.add_argument(\\"--claim-token-budget\\", type=int, default=96)\\n    parser.add_argument(\\"--evidence-token-budget\\", type=int, default=110)\\n    parser.add_argument(\\"--validation-fraction\\", type=float, default=0.10)\\n    parser.add_argument(\\"--validation-policy\\", choices=(\\"stratified\\", \\"temporal\\"), default=\\"stratified\\")\\n    parser.add_argument(\\"--learning-rates\\", default=\\"2e-5,5e-5\\")\\n    parser.add_argument(\\"--epoch-grid\\", default=\\"2,3\\")\\n    parser.add_argument(\\"--batch-size\\", type=int, default=8)\\n    parser.add_argument(\\"--gradient-accumulation-steps\\", type=int, default=2)\\n    parser.add_argument(\\"--checkpoint-steps\\", type=int, default=100)\\n    parser.add_argument(\\"--seed\\", type=int, default=718)\\n    parser.add_argument(\\"--validation-seed\\", type=int, help=\\"Fix the holdout while varying model-training seed; defaults to --seed.\\")\\n    parser.add_argument(\\"--resume\\", action=\\"store_true\\")\\n    parser.add_argument(\\"--class-weighting\\", choices=(\\"balanced\\", \\"none\\"), default=\\"balanced\\")\\n    parser.add_argument(\\"--selection-only\\", action=\\"store_true\\", help=\\"Stop after train-only holdout evaluation; do not refit or score dev.\\")\\n    parser.add_argument(\\"--refit-epoch-policy\\", choices=(\\"requested\\", \\"best_checkpoint\\"), default=\\"requested\\",\\n                        help=\\"Refit for the grid epoch count (legacy) or the epoch of the checkpoint actually selected on train holdout.\\")\\n    args = parser.parse_args()\\n\\n    if args.max_length < 32 or args.claim_token_budget < 1 or args.evidence_token_budget < 1:\\n        parser.error(\\"Token lengths must be positive and max-length must be at least 32\\")\\n    if not 0.01 <= args.validation_fraction <= 0.4:\\n        parser.error(\\"validation-fraction must be between 0.01 and 0.4\\")\\n    if min(args.batch_size, args.gradient_accumulation_steps, args.checkpoint_steps) < 1:\\n        parser.error(\\"batch size, gradient accumulation, and checkpoint steps must be positive\\")\\n    try:\\n        learning_rates = parse_float_grid(args.learning_rates, \\"learning-rates\\")\\n        epochs_grid = parse_int_grid(args.epoch_grid, \\"epoch-grid\\")\\n    except argparse.ArgumentTypeError as error:\\n        parser.error(str(error))\\n\\n    try:\\n        output_dir = resolve_run_directory(args.output_dir, args.run_name)\\n    except ValueError as error:\\n        parser.error(str(error))\\n    output_dir.mkdir(parents=True, exist_ok=True)\\n\\n    processed = args.data_root / \\"processed\\" / \\"averitec\\"\\n    train_path = processed / \\"train.jsonl\\"\\n    eval_path = processed / \\"dev.jsonl\\"\\n    for path in (train_path, eval_path, args.train_rankings, args.eval_rankings):\\n        if not path.is_file():\\n            raise FileNotFoundError(path)\\n\\n    train_records = attach_rankings(read_jsonl(train_path), args.train_rankings)\\n    eval_records = attach_rankings(read_jsonl(eval_path), args.eval_rankings)\\n    validation_seed = args.seed if args.validation_seed is None else args.validation_seed\\n    train_indices, validation_indices = selection_split(train_records, args.validation_fraction, validation_seed, args.validation_policy)\\n    selection_train_records = [train_records[int(index)] for index in train_indices]\\n    validation_records = [train_records[int(index)] for index in validation_indices]\\n\\n    config = {\\n        \\"script\\": \\"train_transformer_verifier\\",\\n        \\"model\\": args.model,\\n        \\"requested_revision\\": args.revision,\\n        \\"max_length\\": args.max_length,\\n        \\"claim_token_budget\\": args.claim_token_budget,\\n        \\"evidence_token_budget_per_passage\\": args.evidence_token_budget,\\n        \\"max_evidence\\": 3,\\n        \\"evidence_source\\": str(args.train_rankings),\\n        \\"train_rankings_sha256\\": sha256_file(args.train_rankings),\\n        \\"eval_rankings\\": str(args.eval_rankings),\\n        \\"eval_rankings_sha256\\": sha256_file(args.eval_rankings),\\n        \\"validation_fraction\\": args.validation_fraction,\\n        \\"validation_seed\\": validation_seed,\\n        \\"training_seed\\": args.seed,\\n        \\"validation_policy\\": args.validation_policy,\\n        \\"learning_rate_grid\\": learning_rates,\\n        \\"epoch_grid\\": epochs_grid,\\n        \\"batch_size\\": args.batch_size,\\n        \\"gradient_accumulation_steps\\": args.gradient_accumulation_steps,\\n        \\"checkpoint_steps\\": args.checkpoint_steps,\\n        \\"loss\\": \\"inverse_frequency_weighted_cross_entropy\\" if args.class_weighting == \\"balanced\\" else \\"unweighted_cross_entropy\\",\\n        \\"class_weighting\\": args.class_weighting,\\n        \\"refit_epoch_policy\\": args.refit_epoch_policy,\\n        \\"split_policy\\": f\\"official train split with internal {args.validation_policy} validation; official dev evaluated after refit\\",\\n    }\\n    config_path = output_dir / \\"transformer_run_config.json\\"\\n    existing_config = read_json(config_path)\\n    if existing_config is not None:\\n        existing_config.setdefault(\\"class_weighting\\", \\"balanced\\")\\n        existing_config.setdefault(\\"refit_epoch_policy\\", \\"requested\\")\\n        existing_config.setdefault(\\"validation_policy\\", \\"stratified\\")\\n        existing_config.setdefault(\\"training_seed\\", existing_config.get(\\"validation_seed\\", 718))\\n    if existing_config is not None and any(existing_config.get(key) != value for key, value in config.items()):\\n        raise ValueError(f\\"Run configuration in {config_path} differs; choose a new --run-name.\\")\\n    config = {**(existing_config or {}), **config}\\n    model_revision = args.revision or (existing_config or {}).get(\\"resolved_model_revision\\")\\n    if existing_config is None:\\n        write_json(config, config_path)\\n\\n    final_metrics_path = output_dir / \\"final_dev_metrics.json\\"\\n    final_predictions_path = output_dir / \\"final_dev_predictions.jsonl\\"\\n    if not args.selection_only and final_metrics_path.exists() and final_predictions_path.exists():\\n        manifest_path = output_dir / \\"transformer_manifest.json\\"\\n        if not manifest_path.is_file():\\n            selection_report = read_json(output_dir / \\"model_selection.json\\") or {}\\n            write_run_manifest(\\n                manifest_path,\\n                project_root=PROJECT_ROOT,\\n                config={**config, \\"selection\\": selection_report, \\"final_dev_metrics_path\\": str(final_metrics_path)},\\n                input_paths={\\n                    \\"train_claims\\": train_path,\\n                    \\"dev_claims\\": eval_path,\\n                    \\"train_rankings\\": args.train_rankings,\\n                    \\"dev_rankings\\": args.eval_rankings,\\n                },\\n            )\\n        print(final_metrics_path.read_text(encoding=\\"utf-8\\"))\\n        return\\n\\n    try:\\n        from transformers import (\\n            AutoModelForSequenceClassification,\\n            AutoTokenizer,\\n            DataCollatorWithPadding,\\n            Trainer,\\n            TrainingArguments,\\n            set_seed,\\n        )\\n    except ImportError as error:\\n        raise RuntimeError(\\"Transformer training requires transformers; install requirements-colab.txt first.\\") from error\\n\\n    class WeightedTrainer(Trainer):\\n        def __init__(self, *trainer_args: Any, class_weights: torch.Tensor, **trainer_kwargs: Any):\\n            super().__init__(*trainer_args, **trainer_kwargs)\\n            self.class_weights = class_weights\\n\\n        def compute_loss(\\n            self,\\n            model: Any,\\n            inputs: dict[str, torch.Tensor],\\n            return_outputs: bool = False,\\n            num_items_in_batch: torch.Tensor | None = None,\\n        ):\\n            labels = inputs[\\"labels\\"]\\n            model_inputs = {key: value for key, value in inputs.items() if key != \\"labels\\"}\\n            outputs = model(**model_inputs)\\n            weights = self.class_weights.to(outputs.logits.device)\\n            loss = torch.nn.functional.cross_entropy(outputs.logits, labels, weight=weights)\\n            return (loss, outputs) if return_outputs else loss\\n\\n    tokenizer = AutoTokenizer.from_pretrained(args.model, revision=model_revision or None, use_fast=True)\\n    resolved_model_revision = getattr(tokenizer, \\"init_kwargs\\", {}).get(\\"_commit_hash\\")\\n    if resolved_model_revision:\\n        config[\\"resolved_model_revision\\"] = resolved_model_revision\\n        write_json(config, config_path)\\n    collator = DataCollatorWithPadding(tokenizer=tokenizer)\\n    selection_train_dataset = EncodedRecords(\\n        selection_train_records, tokenizer, args.max_length, args.claim_token_budget, args.evidence_token_budget\\n    )\\n    validation_dataset = EncodedRecords(\\n        validation_records, tokenizer, args.max_length, args.claim_token_budget, args.evidence_token_budget\\n    )\\n\\n    counts = np.bincount(\\n        [LABEL_TO_ID[str(row[\\"label\\"])] for row in selection_train_records], minlength=len(CANONICAL_LABELS)\\n    )\\n    if np.any(counts == 0):\\n        raise ValueError(f\\"Internal training fold is missing one or more labels: counts={counts.tolist()}\\")\\n    class_weights = torch.tensor(len(selection_train_records) / (len(CANONICAL_LABELS) * counts), dtype=torch.float32)\\n    if args.class_weighting == \\"none\\":\\n        class_weights = torch.ones(len(CANONICAL_LABELS), dtype=torch.float32)\\n    trials_dir = output_dir / \\"trials\\"\\n    trials_dir.mkdir(parents=True, exist_ok=True)\\n    trial_rows: list[dict[str, Any]] = []\\n\\n    def new_model():\\n        nonlocal resolved_model_revision\\n        model = AutoModelForSequenceClassification.from_pretrained(\\n            args.model,\\n            revision=model_revision or None,\\n            num_labels=len(CANONICAL_LABELS),\\n            label2id=LABEL_TO_ID,\\n            id2label=ID_TO_LABEL,\\n        )\\n        resolved_model_revision = getattr(model.config, \\"_commit_hash\\", None) or resolved_model_revision\\n        if resolved_model_revision and config.get(\\"resolved_model_revision\\") != resolved_model_revision:\\n            config[\\"resolved_model_revision\\"] = resolved_model_revision\\n            write_json(config, config_path)\\n        return model\\n\\n    def train_arguments(path: Path, *, lr: float, epochs: float, select_best: bool) -> TrainingArguments:\\n        common = {\\n            \\"output_dir\\": str(path),\\n            \\"learning_rate\\": lr,\\n            \\"num_train_epochs\\": epochs,\\n            \\"per_device_train_batch_size\\": args.batch_size,\\n            \\"per_device_eval_batch_size\\": args.batch_size,\\n            \\"gradient_accumulation_steps\\": args.gradient_accumulation_steps,\\n            \\"weight_decay\\": 0.01,\\n            \\"warmup_ratio\\": 0.06,\\n            \\"logging_steps\\": 25,\\n            \\"logging_first_step\\": True,\\n            \\"disable_tqdm\\": False,\\n            \\"save_strategy\\": \\"steps\\",\\n            \\"save_steps\\": args.checkpoint_steps,\\n            \\"save_total_limit\\": 2,\\n            \\"seed\\": args.seed,\\n            \\"fp16\\": bool(torch.cuda.is_available()),\\n            \\"report_to\\": \\"none\\",\\n            \\"dataloader_num_workers\\": 0,\\n            \\"save_safetensors\\": True,\\n        }\\n        if select_best:\\n            common.update(\\n                {\\n                    \\"eval_strategy\\": \\"steps\\",\\n                    \\"eval_steps\\": args.checkpoint_steps,\\n                    \\"load_best_model_at_end\\": True,\\n                    \\"metric_for_best_model\\": \\"macro_f1\\",\\n                    \\"greater_is_better\\": True,\\n                }\\n            )\\n        else:\\n            common[\\"eval_strategy\\"] = \\"no\\"\\n            common[\\"load_best_model_at_end\\"] = False\\n        return TrainingArguments(**common)\\n\\n    for learning_rate in learning_rates:\\n        for epochs in epochs_grid:\\n            trial_name = f\\"lr_{safe_name(learning_rate)}_epochs_{epochs}\\"\\n            trial_dir = trials_dir / trial_name\\n            summary_path = trial_dir / \\"trial_summary.json\\"\\n            summary = read_json(summary_path)\\n            if summary is None:\\n                if trial_dir.exists() and any(trial_dir.glob(\\"checkpoint-*\\")) and not args.resume:\\n                    raise ValueError(f\\"Found interrupted trial at {trial_dir}; rerun with --resume.\\")\\n                trial_dir.mkdir(parents=True, exist_ok=True)\\n                set_seed(args.seed)\\n                trainer = WeightedTrainer(\\n                    model=new_model(),\\n                    args=train_arguments(trial_dir, lr=learning_rate, epochs=epochs, select_best=True),\\n                    train_dataset=selection_train_dataset,\\n                    eval_dataset=validation_dataset,\\n                    data_collator=collator,\\n                    compute_metrics=prediction_metrics,\\n                    class_weights=class_weights,\\n                )\\n                last = best_checkpoint(trial_dir) if args.resume else None\\n                trainer.train(resume_from_checkpoint=last)\\n                validation_logits = trainer.predict(validation_dataset).predictions\\n                if isinstance(validation_logits, tuple):\\n                    validation_logits = validation_logits[0]\\n                probabilities = torch.softmax(torch.tensor(validation_logits), dim=1).cpu().numpy()\\n                predicted_ids = probabilities.argmax(axis=1)\\n                validation_predictions = []\\n                for row, predicted_id, probability_row in zip(\\n                    validation_records, predicted_ids, probabilities, strict=True\\n                ):\\n                    validation_predictions.append(\\n                        {\\n                            \\"claim_id\\": str(row[\\"claim_id\\"]),\\n                            \\"true_label\\": str(row[\\"label\\"]),\\n                            \\"predicted_label\\": ID_TO_LABEL[int(predicted_id)],\\n                            \\"probabilities\\": {\\n                                ID_TO_LABEL[index]: float(probability_row[index])\\n                                for index in range(len(CANONICAL_LABELS))\\n                            },\\n                        }\\n                    )\\n                metrics = evaluate_predictions(validation_predictions)\\n                selected_checkpoint = trainer.state.best_model_checkpoint\\n                checkpoint_state = read_json(Path(selected_checkpoint) / \\"trainer_state.json\\") if selected_checkpoint else None\\n                summary = {\\n                    \\"trial\\": trial_name,\\n                    \\"learning_rate\\": learning_rate,\\n                    \\"epochs\\": epochs,\\n                    \\"best_checkpoint\\": selected_checkpoint,\\n                    \\"best_checkpoint_epoch\\": checkpoint_state.get(\\"epoch\\") if checkpoint_state else None,\\n                    \\"selection_validation_claim_count\\": len(validation_records),\\n                    \\"macro_f1\\": metrics[\\"macro_f1\\"],\\n                    \\"accuracy\\": metrics[\\"accuracy\\"],\\n                    \\"decision_error_rates\\": metrics[\\"decision_error_rates\\"],\\n                    \\"calibration\\": metrics.get(\\"calibration\\"),\\n                    \\"predictions_path\\": str(trial_dir / \\"validation_predictions.jsonl\\"),\\n                }\\n                write_jsonl(validation_predictions, trial_dir / \\"validation_predictions.jsonl\\")\\n                write_json(summary, summary_path)\\n                del trainer\\n                if torch.cuda.is_available():\\n                    torch.cuda.empty_cache()\\n            trial_rows.append(summary)\\n\\n    selected = sorted(\\n        trial_rows,\\n        key=lambda row: (\\n            -float(row[\\"macro_f1\\"]),\\n            float(row[\\"decision_error_rates\\"][\\"false_reassurance_rate\\"]),\\n            float(row[\\"decision_error_rates\\"][\\"false_alarm_rate\\"]),\\n            float(row[\\"learning_rate\\"]),\\n            int(row[\\"epochs\\"]),\\n        ),\\n    )[0]\\n    selection_report = {\\n        \\"selection_metric\\": \\"validation macro-F1; ties broken by lower false reassurance, then lower false alarm\\",\\n        \\"selected_trial\\": selected,\\n        \\"all_trials\\": trial_rows,\\n        \\"validation_policy\\": args.validation_policy,\\n        \\"validation_seed\\": validation_seed,\\n        \\"training_seed\\": args.seed,\\n        \\"selection_train_claim_count\\": len(selection_train_records),\\n        \\"selection_validation_claim_count\\": len(validation_records),\\n        \\"selection_train_claim_ids\\": [r[\\"claim_id\\"] for r in selection_train_records],\\n        \\"selection_validation_claim_ids\\": [r[\\"claim_id\\"] for r in validation_records],\\n    }\\n    write_json(selection_report, output_dir / \\"model_selection.json\\")\\n    if args.selection_only:\\n        print(json.dumps({k: v for k, v in selection_report.items() if not k.endswith(\\"claim_ids\\")}, indent=2))\\n        return\\n\\n    refit_epochs = float(selected[\\"epochs\\"])\\n    if args.refit_epoch_policy == \\"best_checkpoint\\":\\n        if not selected.get(\\"best_checkpoint_epoch\\"):\\n            raise ValueError(\\"Cached trial lacks best_checkpoint_epoch; use a new run name for best-checkpoint refitting.\\")\\n        refit_epochs = float(selected[\\"best_checkpoint_epoch\\"])\\n\\n    final_dir = output_dir / \\"final_model\\"\\n    final_complete_path = output_dir / \\"final_train_complete.json\\"\\n    if not final_complete_path.exists():\\n        if final_dir.exists() and any(final_dir.glob(\\"checkpoint-*\\")) and not args.resume:\\n            raise ValueError(f\\"Found interrupted final training at {final_dir}; rerun with --resume.\\")\\n        final_dir.mkdir(parents=True, exist_ok=True)\\n        full_counts = np.bincount(\\n            [LABEL_TO_ID[str(row[\\"label\\"])] for row in train_records], minlength=len(CANONICAL_LABELS)\\n        )\\n        full_weights = torch.tensor(len(train_records) / (len(CANONICAL_LABELS) * full_counts), dtype=torch.float32)\\n        if args.class_weighting == \\"none\\":\\n            full_weights = torch.ones(len(CANONICAL_LABELS), dtype=torch.float32)\\n        set_seed(args.seed)\\n        final_trainer = WeightedTrainer(\\n            model=new_model(),\\n            args=train_arguments(final_dir, lr=float(selected[\\"learning_rate\\"]), epochs=refit_epochs, select_best=False),\\n            train_dataset=EncodedRecords(\\n                train_records, tokenizer, args.max_length, args.claim_token_budget, args.evidence_token_budget\\n            ),\\n            data_collator=collator,\\n            class_weights=full_weights,\\n        )\\n        last = best_checkpoint(final_dir) if args.resume else None\\n        final_trainer.train(resume_from_checkpoint=last)\\n        final_trainer.save_model(str(final_dir))\\n        tokenizer.save_pretrained(final_dir)\\n        write_json(\\n            {\\n                \\"selected_trial\\": selected[\\"trial\\"],\\n                \\"training_claim_count\\": len(train_records),\\n                \\"epochs\\": refit_epochs,\\n                \\"refit_epoch_policy\\": args.refit_epoch_policy,\\n                \\"learning_rate\\": selected[\\"learning_rate\\"],\\n            },\\n            final_complete_path,\\n        )\\n        del final_trainer\\n        if torch.cuda.is_available():\\n            torch.cuda.empty_cache()\\n\\n    final_model = AutoModelForSequenceClassification.from_pretrained(final_dir)\\n    final_trainer = WeightedTrainer(\\n        model=final_model,\\n        args=train_arguments(output_dir / \\"prediction_tmp\\", lr=float(selected[\\"learning_rate\\"]), epochs=int(selected[\\"epochs\\"]), select_best=False),\\n        data_collator=collator,\\n        class_weights=torch.ones(len(CANONICAL_LABELS), dtype=torch.float32),\\n    )\\n    eval_dataset = EncodedRecords(eval_records, tokenizer, args.max_length, args.claim_token_budget, args.evidence_token_budget)\\n    prediction_output = final_trainer.predict(eval_dataset)\\n    logits = prediction_output.predictions[0] if isinstance(prediction_output.predictions, tuple) else prediction_output.predictions\\n    probabilities = torch.softmax(torch.tensor(logits), dim=1).cpu().numpy()\\n    predicted_ids = probabilities.argmax(axis=1)\\n    eval_predictions = []\\n    for row, predicted_id, probability_row in zip(eval_records, predicted_ids, probabilities, strict=True):\\n        retrieved = row.get(\\"retrieved_evidence\\") or []\\n        eval_predictions.append(\\n            {\\n                \\"claim_id\\": str(row[\\"claim_id\\"]),\\n                \\"true_label\\": str(row[\\"label\\"]),\\n                \\"predicted_label\\": ID_TO_LABEL[int(predicted_id)],\\n                \\"probabilities\\": {\\n                    ID_TO_LABEL[index]: float(probability_row[index])\\n                    for index in range(len(CANONICAL_LABELS))\\n                },\\n                \\"evidence_item_count\\": min(3, len(retrieved)),\\n                \\"evidence_passage_ids\\": [str(item[\\"passage_id\\"]) for item in retrieved[:3]],\\n            }\\n        )\\n    metrics = evaluate_predictions(eval_predictions)\\n    write_jsonl(eval_predictions, final_predictions_path)\\n    final_report = {\\n        **metrics,\\n        \\"selection\\": selected,\\n        \\"eval_split\\": \\"official dev\\",\\n        \\"eval_claim_count\\": len(eval_records),\\n        \\"evidence_source\\": str(args.eval_rankings),\\n        \\"model\\": args.model,\\n        \\"model_path\\": str(final_dir),\\n        \\"refit_epochs\\": refit_epochs,\\n        \\"refit_epoch_policy\\": args.refit_epoch_policy,\\n    }\\n    write_json(final_report, final_metrics_path)\\n    write_run_manifest(\\n        output_dir / \\"transformer_manifest.json\\",\\n        project_root=PROJECT_ROOT,\\n        config={**config, \\"selection\\": selection_report, \\"final_dev_metrics_path\\": str(final_metrics_path)},\\n        input_paths={\\n            \\"train_claims\\": train_path,\\n            \\"dev_claims\\": eval_path,\\n            \\"train_rankings\\": args.train_rankings,\\n            \\"dev_rankings\\": args.eval_rankings,\\n        },\\n    )\\n    print(json.dumps(final_report, indent=2))\\n\\n\\nif __name__ == \\"__main__\\":\\n    main()\\n", "src/__init__.py": "\\"\\"\\"Misinformation screening assistant source package.\\"\\"\\"\\n", "src/data/__init__.py": "\\"\\"\\"Dataset loading and canonical record schemas.\\"\\"\\"\\n", "src/data/averitec.py": "\\"\\"\\"AVeriTeC claim loading, normalization, and audit helpers.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport json\\nfrom collections import Counter\\nfrom pathlib import Path\\nfrom typing import Any, Iterable\\n\\nfrom src.data.schema import ClaimRecord, EvidenceItem\\n\\nCANONICAL_LABELS = (\\n    \\"Supported\\",\\n    \\"Refuted\\",\\n    \\"Not Enough Evidence\\",\\n    \\"Conflicting Evidence\\",\\n)\\n\\n_LABEL_ALIASES = {\\n    \\"supported\\": \\"Supported\\",\\n    \\"refuted\\": \\"Refuted\\",\\n    \\"not enough evidence\\": \\"Not Enough Evidence\\",\\n    \\"conflicting evidence/cherrypicking\\": \\"Conflicting Evidence\\",\\n    \\"conflicting evidence/cherry-picking\\": \\"Conflicting Evidence\\",\\n    \\"conflicting evidence\\": \\"Conflicting Evidence\\",\\n}\\n\\n\\ndef normalize_label(label: str | None) -> str | None:\\n    \\"\\"\\"Return the project label corresponding to an official AVeriTeC label.\\"\\"\\"\\n    if label is None:\\n        return None\\n    cleaned = \\" \\".join(label.strip().split()).casefold()\\n    try:\\n        return _LABEL_ALIASES[cleaned]\\n    except KeyError as exc:\\n        raise ValueError(f\\"Unsupported AVeriTeC label: {label!r}\\") from exc\\n\\n\\ndef _read_json_or_jsonl(path: Path) -> list[dict[str, Any]]:\\n    with path.open(\\"r\\", encoding=\\"utf-8\\") as handle:\\n        text = handle.read().strip()\\n    if not text:\\n        return []\\n    parsed = json.loads(text)\\n    if isinstance(parsed, list):\\n        return parsed\\n    if isinstance(parsed, dict):\\n        return [parsed]\\n    raise ValueError(f\\"Expected a JSON object or list in {path}\\")\\n\\n\\ndef _answers_to_evidence(questions: Iterable[dict[str, Any]], claim_id: str) -> tuple[EvidenceItem, ...]:\\n    \\"\\"\\"Expose annotated answers for gold-evidence verification only.\\n\\n    These records must never be used to construct the retrieval corpus.\\n    \\"\\"\\"\\n    evidence: list[EvidenceItem] = []\\n    for question_index, question in enumerate(questions):\\n        question_text = str(question.get(\\"question\\") or \\"\\").strip()\\n        for answer_index, answer in enumerate(question.get(\\"answers\\") or []):\\n            answer_text = str(answer.get(\\"answer\\") or \\"\\").strip()\\n            if not answer_text:\\n                continue\\n            source_url = str(answer.get(\\"source_url\\") or \\"\\").strip() or None\\n            evidence.append(\\n                EvidenceItem(\\n                    passage_id=f\\"{claim_id}:q{question_index}:a{answer_index}\\",\\n                    text=answer_text,\\n                    url=source_url,\\n                    metadata={\\n                        \\"question\\": question_text,\\n                        \\"answer_type\\": answer.get(\\"answer_type\\"),\\n                        \\"source_medium\\": answer.get(\\"source_medium\\"),\\n                        \\"is_gold_answer\\": True,\\n                    },\\n                )\\n            )\\n    return tuple(evidence)\\n\\n\\ndef raw_to_record(raw: dict[str, Any], split: str, index: int) -> ClaimRecord:\\n    \\"\\"\\"Convert one official AVeriTeC object into the project record shape.\\"\\"\\"\\n    claim = str(raw.get(\\"claim\\") or \\"\\").strip()\\n    if not claim:\\n        raise ValueError(f\\"{split}[{index}] has an empty claim\\")\\n    claim_id = str(raw.get(\\"claim_id\\") or f\\"averitec:{split}:{index:05d}\\")\\n    label = normalize_label(raw.get(\\"label\\"))\\n    questions = raw.get(\\"questions\\") or []\\n    if not isinstance(questions, list):\\n        raise ValueError(f\\"{claim_id} has a non-list questions field\\")\\n    source_urls = tuple(\\n        item.url for item in _answers_to_evidence(questions, claim_id) if item.url is not None\\n    )\\n    return ClaimRecord(\\n        claim_id=claim_id,\\n        claim=claim,\\n        label=label,\\n        source_urls=tuple(dict.fromkeys(source_urls)),\\n        evidence=_answers_to_evidence(questions, claim_id),\\n        metadata={\\n            \\"dataset\\": \\"AVeriTeC\\",\\n            \\"split\\": split,\\n            \\"source_index\\": index,\\n            \\"claim_date\\": raw.get(\\"claim_date\\"),\\n            \\"speaker\\": raw.get(\\"speaker\\"),\\n            \\"reporting_source\\": raw.get(\\"reporting_source\\"),\\n            \\"location_iso_code\\": raw.get(\\"location_ISO_code\\"),\\n            \\"required_reannotation\\": raw.get(\\"required_reannotation\\"),\\n            \\"fact_checking_article\\": raw.get(\\"fact_checking_article\\"),\\n        },\\n    )\\n\\n\\ndef load_split(path: str | Path, split: str) -> list[ClaimRecord]:\\n    \\"\\"\\"Load one official AVeriTeC split from a JSON file.\\"\\"\\"\\n    source_path = Path(path)\\n    records = [raw_to_record(raw, split, index) for index, raw in enumerate(_read_json_or_jsonl(source_path))]\\n    ids = [record.claim_id for record in records]\\n    if len(ids) != len(set(ids)):\\n        raise ValueError(f\\"Duplicate claim IDs found in {source_path}\\")\\n    return records\\n\\n\\ndef load_split_with_rejections(\\n    path: str | Path, split: str\\n) -> tuple[list[ClaimRecord], list[dict[str, Any]]]:\\n    \\"\\"\\"Load a split while preserving an audit trail for unusable source rows.\\n\\n    An empty claim cannot be evaluated by any project model, so it is excluded\\n    from prepared output. Any other schema or label error remains a hard failure\\n    until it is explicitly understood.\\n    \\"\\"\\"\\n    source_path = Path(path)\\n    records: list[ClaimRecord] = []\\n    rejections: list[dict[str, Any]] = []\\n    for index, raw in enumerate(_read_json_or_jsonl(source_path)):\\n        if not str(raw.get(\\"claim\\") or \\"\\").strip():\\n            rejections.append(\\n                {\\n                    \\"source_index\\": index,\\n                    \\"reason\\": \\"empty_claim\\",\\n                    \\"source_label\\": raw.get(\\"label\\"),\\n                    \\"fact_checking_article\\": raw.get(\\"fact_checking_article\\"),\\n                }\\n            )\\n            continue\\n        records.append(raw_to_record(raw, split, index))\\n    ids = [record.claim_id for record in records]\\n    if len(ids) != len(set(ids)):\\n        raise ValueError(f\\"Duplicate claim IDs found in {source_path}\\")\\n    return records, rejections\\n\\n\\ndef split_audit(records: Iterable[ClaimRecord]) -> dict[str, Any]:\\n    \\"\\"\\"Return a small, serializable audit report for a prepared split.\\"\\"\\"\\n    materialized = list(records)\\n    return {\\n        \\"record_count\\": len(materialized),\\n        \\"label_counts\\": dict(sorted(Counter(record.label for record in materialized).items())),\\n        \\"records_with_gold_answers\\": sum(bool(record.evidence) for record in materialized),\\n        \\"gold_answer_count\\": sum(len(record.evidence) for record in materialized),\\n    }\\n\\n\\ndef write_jsonl(records: Iterable[ClaimRecord], path: str | Path) -> None:\\n    destination = Path(path)\\n    destination.parent.mkdir(parents=True, exist_ok=True)\\n    with destination.open(\\"w\\", encoding=\\"utf-8\\", newline=\\"\\\\n\\") as handle:\\n        for record in records:\\n            handle.write(json.dumps(record.to_dict(), ensure_ascii=False) + \\"\\\\n\\")\\n", "src/data/corpus.py": "\\"\\"\\"Candidate-document validation, passage chunking, and URL-based qrels.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport re\\nfrom collections import Counter, defaultdict\\nfrom collections.abc import Iterable\\nfrom typing import Any\\nfrom urllib.parse import urlsplit, urlunsplit\\n\\n\\ndef canonical_url(value: str | None) -> str | None:\\n    \\"\\"\\"Normalize only URL details that should not change document identity.\\"\\"\\"\\n    if not value:\\n        return None\\n    raw = value.strip()\\n    if not raw:\\n        return None\\n    parts = urlsplit(raw)\\n    if not parts.scheme or not parts.netloc:\\n        return raw.rstrip(\\"/\\")\\n    return urlunsplit((parts.scheme.casefold(), parts.netloc.casefold(), parts.path.rstrip(\\"/\\"), parts.query, \\"\\"))\\n\\n\\ndef chunk_text(text: str, max_words: int = 160, overlap_words: int = 40) -> list[str]:\\n    \\"\\"\\"Split a document into deterministic overlapping word passages.\\"\\"\\"\\n    if max_words < 1 or overlap_words < 0 or overlap_words >= max_words:\\n        raise ValueError(\\"Require max_words > 0 and 0 <= overlap_words < max_words\\")\\n    words = re.findall(r\\"\\\\S+\\", text)\\n    if not words:\\n        return []\\n    passages = []\\n    step = max_words - overlap_words\\n    for start in range(0, len(words), step):\\n        passage = \\" \\".join(words[start : start + max_words]).strip()\\n        if passage:\\n            passages.append(passage)\\n        if start + max_words >= len(words):\\n            break\\n    return passages\\n\\n\\ndef documents_to_passages(\\n    documents: Iterable[dict[str, Any]], max_words: int = 160, overlap_words: int = 40\\n) -> tuple[list[dict[str, Any]], dict[str, Any]]:\\n    \\"\\"\\"Convert normalized candidate documents into provenance-preserving passages.\\n\\n    Input documents must have a non-empty `text` and should have a stable `url`\\n    or `document_id`. This function is intentionally format-agnostic; the\\n    AVeriTeC archive inspector identifies how to create these documents.\\n    \\"\\"\\"\\n    passages: list[dict[str, Any]] = []\\n    rejected = 0\\n    for document_index, document in enumerate(documents):\\n        text = str(document.get(\\"text\\") or \\"\\").strip()\\n        if not text:\\n            rejected += 1\\n            continue\\n        document_id = str(document.get(\\"document_id\\") or document.get(\\"url\\") or f\\"document:{document_index}\\")\\n        url = document.get(\\"url\\")\\n        for passage_index, passage_text in enumerate(chunk_text(text, max_words, overlap_words)):\\n            passages.append(\\n                {\\n                    \\"passage_id\\": f\\"{document_id}:p{passage_index:04d}\\",\\n                    \\"text\\": passage_text,\\n                    \\"url\\": url,\\n                    \\"metadata\\": {\\n                        \\"document_id\\": document_id,\\n                        \\"chunk_index\\": passage_index,\\n                        \\"chunk_max_words\\": max_words,\\n                        \\"chunk_overlap_words\\": overlap_words,\\n                    },\\n                }\\n            )\\n    return passages, {\\"passage_count\\": len(passages), \\"rejected_empty_document_count\\": rejected}\\n\\n\\ndef build_url_qrels(\\n    claims: Iterable[dict[str, Any]], passages: Iterable[dict[str, Any]]\\n) -> tuple[list[dict[str, Any]], dict[str, Any]]:\\n    \\"\\"\\"Build *source-URL* relevance judgments from AVeriTeC annotations.\\n\\n    This is an auditable weak passage-level proxy: every passage from a human\\n    annotated evidence URL is marked relevant. Reports must call it URL-based\\n    evidence coverage, not exact human passage relevance.\\n    \\"\\"\\"\\n    # Claims are compact; materialize them once.  The large corpus itself is\\n    # streamed and only passage IDs for annotated URLs are retained.\\n    claim_rows = list(claims)\\n    annotated_urls = {\\n        canonical_url(url)\\n        for claim in claim_rows\\n        for url in (claim.get(\\"source_urls\\") or [])\\n    }\\n    annotated_urls.discard(None)\\n    passage_ids_by_url: dict[str, list[tuple[str, str | None]]] = defaultdict(list)\\n    for passage in passages:\\n        url = canonical_url(passage.get(\\"url\\"))\\n        if url in annotated_urls:\\n            candidate_claim_id = passage.get(\\"metadata\\", {}).get(\\"candidate_claim_id\\")\\n            passage_ids_by_url[url].append((str(passage[\\"passage_id\\"]), str(candidate_claim_id) if candidate_claim_id else None))\\n\\n    qrels: list[dict[str, Any]] = []\\n    unresolved_url_count = 0\\n    for claim in claim_rows:\\n        gold_urls = {canonical_url(url) for url in claim.get(\\"source_urls\\") or []}\\n        gold_urls.discard(None)\\n        relevant_ids = sorted(\\n            {\\n                passage_id\\n                for url in gold_urls\\n                for passage_id, candidate_claim_id in passage_ids_by_url.get(str(url), [])\\n                if candidate_claim_id is None or candidate_claim_id == str(claim[\\"claim_id\\"])\\n            }\\n        )\\n        unresolved = sorted(\\n            url\\n            for url in gold_urls\\n            if not any(\\n                candidate_claim_id is None or candidate_claim_id == str(claim[\\"claim_id\\"])\\n                for _, candidate_claim_id in passage_ids_by_url.get(str(url), [])\\n            )\\n        )\\n        unresolved_url_count += len(unresolved)\\n        qrels.append(\\n            {\\n                \\"claim_id\\": claim[\\"claim_id\\"],\\n                \\"relevant_passage_ids\\": relevant_ids,\\n                \\"judgment_type\\": \\"annotated_source_url_proxy\\",\\n                \\"annotated_source_url_count\\": len(gold_urls),\\n                \\"unresolved_source_urls\\": unresolved,\\n            }\\n        )\\n    audit = {\\n        \\"judgment_type\\": \\"annotated_source_url_proxy\\",\\n        \\"claim_count\\": len(qrels),\\n        \\"judged_claim_count\\": sum(bool(row[\\"relevant_passage_ids\\"]) for row in qrels),\\n        \\"unresolved_annotated_source_url_count\\": unresolved_url_count,\\n        \\"relevant_passage_count_distribution\\": dict(\\n            sorted(Counter(len(row[\\"relevant_passage_ids\\"]) for row in qrels).items())\\n        ),\\n    }\\n    return qrels, audit\\n", "src/data/knowledge_store.py": "\\"\\"\\"Safe AVeriTeC knowledge-store archive inspection for Colab.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport json\\nimport csv\\nimport io\\nimport zipfile\\nimport re\\nfrom collections import Counter\\nfrom pathlib import Path, PurePosixPath\\nfrom typing import Any\\n\\nfrom src.utils.progress import progress\\n\\n\\nURL_KEYS = (\\"url\\", \\"source_url\\", \\"document_url\\")\\nTEXT_KEYS = (\\"text\\", \\"scraped_text\\", \\"content\\", \\"lines\\", \\"document\\", \\"body\\")\\n_CLAIM_FILE_PATTERN = re.compile(r\\"(?:^|/)(\\\\d+)\\\\.json(?:#\\\\d+)?$\\")\\n\\n\\ndef inspect_zip_archive(path: str | Path, sample_members: int = 10) -> dict[str, Any]:\\n    \\"\\"\\"Inspect archive structure without extracting large files or assuming schema.\\"\\"\\"\\n    archive_path = Path(path)\\n    with zipfile.ZipFile(archive_path) as archive:\\n        members = [member for member in archive.infolist() if not member.is_dir()]\\n        suffix_counts = Counter(PurePosixPath(member.filename).suffix.casefold() or \\"[no extension]\\" for member in members)\\n        samples = []\\n        inspected_members = members[:sample_members]\\n        with progress(total=len(inspected_members), description=\\"Inspecting archive\\", unit=\\"file\\") as bar:\\n            for member in inspected_members:\\n                sample: dict[str, Any] = {\\"name\\": member.filename, \\"compressed_size\\": member.compress_size, \\"file_size\\": member.file_size}\\n                suffix = PurePosixPath(member.filename).suffix.casefold()\\n                if suffix in {\\".json\\", \\".jsonl\\"} and member.file_size <= 2_000_000:\\n                    with archive.open(member) as handle:\\n                        text = handle.read().decode(\\"utf-8\\", errors=\\"replace\\")\\n                    try:\\n                        parsed = json.loads(text) if suffix == \\".json\\" else json.loads(next(line for line in text.splitlines() if line.strip()))\\n                    except json.JSONDecodeError:\\n                        # Official AVeriTeC stores JSONL rows in files named\\n                        # `N.json`; show the first row shape instead of calling it\\n                        # an opaque parse failure in the inspection report.\\n                        try:\\n                            parsed = json.loads(next(line for line in text.splitlines() if line.strip()))\\n                            sample[\\"jsonl_with_json_suffix\\"] = suffix == \\".json\\"\\n                        except (json.JSONDecodeError, StopIteration):\\n                            sample[\\"json_parse_error\\"] = True\\n                            parsed = None\\n                    except StopIteration:\\n                        sample[\\"json_parse_error\\"] = True\\n                        parsed = None\\n                    if parsed is not None:\\n                        sample[\\"json_type\\"] = type(parsed).__name__\\n                        if isinstance(parsed, dict):\\n                            sample[\\"json_keys\\"] = sorted(parsed)[:30]\\n                        elif isinstance(parsed, list) and parsed and isinstance(parsed[0], dict):\\n                            sample[\\"first_row_keys\\"] = sorted(parsed[0])[:30]\\n                elif suffix in {\\".tsv\\", \\".csv\\"} and member.file_size <= 2_000_000:\\n                    with archive.open(member) as handle:\\n                        sample[\\"header\\"] = handle.readline().decode(\\"utf-8\\", errors=\\"replace\\").strip()\\n                samples.append(sample)\\n                bar.update(1)\\n    return {\\n        \\"archive\\": str(archive_path),\\n        \\"archive_size_bytes\\": archive_path.stat().st_size,\\n        \\"member_count\\": len(members),\\n        \\"suffix_counts\\": dict(sorted(suffix_counts.items())),\\n        \\"sample_members\\": samples,\\n    }\\n\\n\\ndef _text_value(value: Any) -> str | None:\\n    if isinstance(value, str):\\n        return value.strip() or None\\n    if isinstance(value, list) and all(isinstance(item, str) for item in value):\\n        text = \\"\\\\n\\".join(item.strip() for item in value if item.strip())\\n        return text or None\\n    return None\\n\\n\\ndef _candidate_claim_index(source_member: str) -> int | None:\\n    \\"\\"\\"Read the zero-based claim index from official `output_<split>/N.json` names.\\"\\"\\"\\n    match = _CLAIM_FILE_PATTERN.search(source_member)\\n    return int(match.group(1)) if match else None\\n\\n\\ndef _documents_from_value(\\n    value: Any, source_member: str, *, url2text_mode: str = \\"sentence\\"\\n) -> list[dict[str, Any]]:\\n    \\"\\"\\"Interpret common URL/text JSON shapes without fabricating content.\\"\\"\\"\\n    if url2text_mode not in {\\"sentence\\", \\"source_document\\"}:\\n        raise ValueError(\\"url2text_mode must be \'sentence\' or \'source_document\'\\")\\n    if isinstance(value, list):\\n        documents = []\\n        for index, item in enumerate(value):\\n            documents.extend(\\n                _documents_from_value(item, f\\"{source_member}#{index}\\", url2text_mode=url2text_mode)\\n            )\\n        return documents\\n    if not isinstance(value, dict):\\n        return []\\n    url = next((str(value[key]).strip() for key in URL_KEYS if value.get(key)), None)\\n    # The official AVeriTeC knowledge store contains one JSONL file per claim.\\n    # Each row associates a URL with `url2text`, a list of extracted sentences.\\n    # Preserve the candidate-pool boundary so retrieval never crosses claims.\\n    sentence_values = value.get(\\"url2text\\")\\n    if url and isinstance(sentence_values, list) and all(isinstance(item, str) for item in sentence_values):\\n        candidate_claim_index = _candidate_claim_index(source_member)\\n        cleaned_sentences = [sentence.strip() for sentence in sentence_values if sentence.strip()]\\n        if url2text_mode == \\"source_document\\" and cleaned_sentences:\\n            # Each official JSONL row is a URL and its extracted sentences.\\n            # Keep that natural source-document unit, then let the later\\n            # passage builder produce overlapping 160-word retrieval chunks.\\n            # This is substantially smaller than making every sentence an\\n            # independent candidate while preserving the same evidence text.\\n            return [\\n                {\\n                    \\"document_id\\": f\\"{source_member}:source_document\\",\\n                    \\"url\\": url,\\n                    \\"text\\": \\"\\\\n\\".join(cleaned_sentences),\\n                    \\"metadata\\": {\\n                        \\"archive_member\\": source_member.split(\\"#\\", 1)[0],\\n                        \\"candidate_claim_index\\": candidate_claim_index,\\n                        \\"source_format\\": \\"averitec_url2text_source_document\\",\\n                    },\\n                }\\n            ]\\n        return [\\n            {\\n                \\"document_id\\": f\\"{source_member}:sentence:{sentence_index}\\",\\n                \\"url\\": url,\\n                \\"text\\": sentence.strip(),\\n                \\"metadata\\": {\\n                    \\"archive_member\\": source_member.split(\\"#\\", 1)[0],\\n                    \\"candidate_claim_index\\": candidate_claim_index,\\n                    \\"source_format\\": \\"averitec_url2text\\",\\n                },\\n            }\\n            for sentence_index, sentence in enumerate(sentence_values)\\n            if sentence.strip()\\n        ]\\n    text = next((_text_value(value[key]) for key in TEXT_KEYS if _text_value(value.get(key))), None)\\n    if url and text:\\n        return [{\\"document_id\\": f\\"{source_member}:{url}\\", \\"url\\": url, \\"text\\": text, \\"metadata\\": {\\"archive_member\\": source_member}}]\\n    # Some archives use a dictionary mapping source URL to extracted text/lines.\\n    documents = []\\n    for key, nested_value in value.items():\\n        if key.startswith((\\"http://\\", \\"https://\\")):\\n            nested_text = _text_value(nested_value)\\n            if nested_text:\\n                documents.append(\\n                    {\\"document_id\\": f\\"{source_member}:{key}\\", \\"url\\": key, \\"text\\": nested_text, \\"metadata\\": {\\"archive_member\\": source_member}}\\n                )\\n    return documents\\n\\n\\ndef normalize_zip_to_documents(\\n    archive_path: str | Path, *, url2text_mode: str = \\"sentence\\"\\n) -> tuple[list[dict[str, Any]], dict[str, Any]]:\\n    \\"\\"\\"Extract recognized JSON/JSONL/TSV/CSV URL-text records directly from a zip.\\n\\n    Unknown members are counted and omitted rather than guessed. Review the\\n    resulting audit before treating the documents as a valid evidence corpus.\\n    \\"\\"\\"\\n    source = Path(archive_path)\\n    documents: list[dict[str, Any]] = []\\n    audit: dict[str, Any] = {\\"archive\\": str(source), \\"member_counts\\": Counter(), \\"recognized_member_counts\\": Counter(), \\"skipped_member_samples\\": []}\\n    with zipfile.ZipFile(source) as archive:\\n        for member in archive.infolist():\\n            if member.is_dir():\\n                continue\\n            suffix = PurePosixPath(member.filename).suffix.casefold()\\n            audit[\\"member_counts\\"][suffix or \\"[no extension]\\"] += 1\\n            try:\\n                with archive.open(member) as binary_handle:\\n                    if suffix == \\".json\\":\\n                        value = json.load(io.TextIOWrapper(binary_handle, encoding=\\"utf-8\\", errors=\\"replace\\"))\\n                        parsed = _documents_from_value(value, member.filename, url2text_mode=url2text_mode)\\n                    elif suffix == \\".jsonl\\":\\n                        parsed = []\\n                        for index, line in enumerate(io.TextIOWrapper(binary_handle, encoding=\\"utf-8\\", errors=\\"replace\\")):\\n                            if line.strip():\\n                                parsed.extend(\\n                                    _documents_from_value(\\n                                        json.loads(line), f\\"{member.filename}#{index}\\", url2text_mode=url2text_mode\\n                                    )\\n                                )\\n                    elif suffix in {\\".tsv\\", \\".csv\\"}:\\n                        delimiter = \\"\\\\t\\" if suffix == \\".tsv\\" else \\",\\"\\n                        reader = csv.DictReader(io.TextIOWrapper(binary_handle, encoding=\\"utf-8\\", errors=\\"replace\\"), delimiter=delimiter)\\n                        parsed = _documents_from_value(list(reader), member.filename, url2text_mode=url2text_mode)\\n                    else:\\n                        parsed = []\\n                if parsed:\\n                    audit[\\"recognized_member_counts\\"][suffix] += 1\\n                    documents.extend(parsed)\\n                elif len(audit[\\"skipped_member_samples\\"]) < 20:\\n                    audit[\\"skipped_member_samples\\"].append(member.filename)\\n            except (json.JSONDecodeError, UnicodeError, csv.Error) as error:\\n                if len(audit[\\"skipped_member_samples\\"]) < 20:\\n                    audit[\\"skipped_member_samples\\"].append(f\\"{member.filename} ({type(error).__name__})\\")\\n    audit[\\"member_counts\\"] = dict(sorted(audit[\\"member_counts\\"].items()))\\n    audit[\\"recognized_member_counts\\"] = dict(sorted(audit[\\"recognized_member_counts\\"].items()))\\n    audit[\\"normalized_document_count\\"] = len(documents)\\n    audit[\\"documents_with_url\\"] = sum(bool(document.get(\\"url\\")) for document in documents)\\n    audit[\\"url2text_mode\\"] = url2text_mode\\n    return documents, audit\\n\\n\\ndef normalize_zip_to_jsonl(\\n    archive_path: str | Path,\\n    output_path: str | Path,\\n    *,\\n    max_documents: int | None = None,\\n    url2text_mode: str = \\"sentence\\",\\n) -> dict[str, Any]:\\n    \\"\\"\\"Write recognized archive records to JSONL without retaining the corpus in RAM.\\n\\n    This is the Colab-facing path.  It streams JSONL/CSV/TSV records and\\n    writes each normalized document immediately.  JSON members still need to\\n    be valid standalone JSON values, but no list of documents from the whole\\n    archive is accumulated in memory.\\n    \\"\\"\\"\\n    if max_documents is not None and max_documents < 1:\\n        raise ValueError(\\"max_documents must be positive when supplied\\")\\n    if url2text_mode not in {\\"sentence\\", \\"source_document\\"}:\\n        raise ValueError(\\"url2text_mode must be \'sentence\' or \'source_document\'\\")\\n    source = Path(archive_path)\\n    destination = Path(output_path)\\n    destination.parent.mkdir(parents=True, exist_ok=True)\\n    audit: dict[str, Any] = {\\n        \\"archive\\": str(source),\\n        \\"member_counts\\": Counter(),\\n        \\"recognized_member_counts\\": Counter(),\\n        \\"skipped_member_samples\\": [],\\n        \\"max_documents\\": max_documents,\\n        \\"url2text_mode\\": url2text_mode,\\n    }\\n    normalized_document_count = 0\\n    stopped_early = False\\n\\n    with zipfile.ZipFile(source) as archive, destination.open(\\"w\\", encoding=\\"utf-8\\", newline=\\"\\\\n\\") as output:\\n        members = [member for member in archive.infolist() if not member.is_dir()]\\n        with progress(total=len(members), description=\\"Normalizing archive\\", unit=\\"file\\") as bar:\\n            for member in members:\\n                suffix = PurePosixPath(member.filename).suffix.casefold()\\n                audit[\\"member_counts\\"][suffix or \\"[no extension]\\"] += 1\\n                recognized_member = False\\n                try:\\n                    with archive.open(member) as binary_handle:\\n                        text_handle = io.TextIOWrapper(binary_handle, encoding=\\"utf-8\\", errors=\\"replace\\")\\n                        if suffix == \\".json\\":\\n                            # AVeriTeC calls its per-claim JSONL files `.json`.\\n                            # Try ordinary JSON first, then safely fall back to one\\n                            # JSON object per line without treating an invalid file\\n                            # as a generic format.\\n                            content = text_handle.read()\\n                            try:\\n                                parsed_rows = _documents_from_value(\\n                                    json.loads(content), member.filename, url2text_mode=url2text_mode\\n                                )\\n                                row_iterator = iter(parsed_rows)\\n                            except json.JSONDecodeError:\\n                                def json_rows_with_json_suffix() -> Any:\\n                                    for index, line in enumerate(content.splitlines()):\\n                                        if line.strip():\\n                                            yield from _documents_from_value(\\n                                                json.loads(line),\\n                                                f\\"{member.filename}#{index}\\",\\n                                                url2text_mode=url2text_mode,\\n                                            )\\n                                row_iterator = json_rows_with_json_suffix()\\n                        elif suffix == \\".jsonl\\":\\n                            def jsonl_rows() -> Any:\\n                                for index, line in enumerate(text_handle):\\n                                    if line.strip():\\n                                        yield from _documents_from_value(\\n                                            json.loads(line),\\n                                            f\\"{member.filename}#{index}\\",\\n                                            url2text_mode=url2text_mode,\\n                                        )\\n                            row_iterator = jsonl_rows()\\n                        elif suffix in {\\".tsv\\", \\".csv\\"}:\\n                            delimiter = \\"\\\\t\\" if suffix == \\".tsv\\" else \\",\\"\\n                            reader = csv.DictReader(text_handle, delimiter=delimiter)\\n                            def tabular_rows() -> Any:\\n                                for index, row in enumerate(reader):\\n                                    yield from _documents_from_value(\\n                                        row, f\\"{member.filename}#{index}\\", url2text_mode=url2text_mode\\n                                    )\\n                            row_iterator = tabular_rows()\\n                        else:\\n                            row_iterator = iter(())\\n                        for document in row_iterator:\\n                            output.write(json.dumps(document, ensure_ascii=False) + \\"\\\\n\\")\\n                            normalized_document_count += 1\\n                            recognized_member = True\\n                            if max_documents is not None and normalized_document_count >= max_documents:\\n                                stopped_early = True\\n                                break\\n                    if recognized_member:\\n                        audit[\\"recognized_member_counts\\"][suffix] += 1\\n                    elif len(audit[\\"skipped_member_samples\\"]) < 20:\\n                        audit[\\"skipped_member_samples\\"].append(member.filename)\\n                except (json.JSONDecodeError, UnicodeError, csv.Error) as error:\\n                    if len(audit[\\"skipped_member_samples\\"]) < 20:\\n                        audit[\\"skipped_member_samples\\"].append(f\\"{member.filename} ({type(error).__name__})\\")\\n                bar.update(1)\\n                bar.set_postfix(documents=normalized_document_count)\\n                if stopped_early:\\n                    break\\n\\n    audit[\\"member_counts\\"] = dict(sorted(audit[\\"member_counts\\"].items()))\\n    audit[\\"recognized_member_counts\\"] = dict(sorted(audit[\\"recognized_member_counts\\"].items()))\\n    audit[\\"normalized_document_count\\"] = normalized_document_count\\n    audit[\\"stopped_early_at_document_limit\\"] = stopped_early\\n    audit[\\"warning\\"] = (\\n        \\"This normalized corpus is intentionally incomplete because max_documents was used. \\"\\n        \\"Do not report retrieval metrics from it as full-corpus AVeriTeC results.\\"\\n        if stopped_early\\n        else None\\n    )\\n    return audit\\n", "src/data/quality.py": "\\"\\"\\"Data-quality and split-overlap audits for leakage-aware experiments.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport itertools\\nimport re\\nfrom collections import defaultdict\\nfrom collections.abc import Mapping, Sequence\\nfrom typing import Any\\n\\nfrom src.data.corpus import canonical_url\\nfrom src.data.schema import ClaimRecord\\n\\n\\ndef normalized_claim(text: str) -> str:\\n    return \\" \\".join(re.findall(r\\"\\\\w+\\", text.casefold()))\\n\\n\\ndef _values_to_ids(records: Sequence[ClaimRecord], value_fn) -> dict[str, set[str]]:\\n    output: dict[str, set[str]] = defaultdict(set)\\n    for record in records:\\n        for value in value_fn(record):\\n            if value:\\n                output[str(value)].add(record.claim_id)\\n    return output\\n\\n\\ndef split_overlap_audit(split_records: Mapping[str, Sequence[ClaimRecord]], sample_limit: int = 10) -> dict[str, Any]:\\n    \\"\\"\\"Report exact overlaps for inspection; it never silently drops examples.\\"\\"\\"\\n    claim_values = {\\n        split: _values_to_ids(records, lambda record: [normalized_claim(record.claim)])\\n        for split, records in split_records.items()\\n    }\\n    article_values = {\\n        split: _values_to_ids(records, lambda record: [canonical_url(record.metadata.get(\\"fact_checking_article\\"))])\\n        for split, records in split_records.items()\\n    }\\n    source_values = {\\n        split: _values_to_ids(records, lambda record: [canonical_url(url) for url in record.source_urls])\\n        for split, records in split_records.items()\\n    }\\n    report: dict[str, Any] = {}\\n    for left, right in itertools.combinations(sorted(split_records), 2):\\n        pair = f\\"{left}__{right}\\"\\n        report[pair] = {}\\n        for name, values in ((\\"normalized_claim\\", claim_values), (\\"fact_checking_article_url\\", article_values), (\\"annotated_source_url\\", source_values)):\\n            overlap = sorted(set(values[left]) & set(values[right]))\\n            report[pair][name] = {\\"overlap_count\\": len(overlap), \\"samples\\": overlap[:sample_limit]}\\n    return report\\n", "src/data/schema.py": "\\"\\"\\"Shared, serializable records used throughout the project.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nfrom dataclasses import asdict, dataclass, field\\nfrom typing import Any\\n\\n\\n@dataclass(frozen=True)\\nclass EvidenceItem:\\n    \\"\\"\\"A retrievable evidence passage with its source provenance.\\"\\"\\"\\n\\n    passage_id: str\\n    text: str\\n    url: str | None = None\\n    domain: str | None = None\\n    score: float | None = None\\n    metadata: dict[str, Any] = field(default_factory=dict)\\n\\n    def to_dict(self) -> dict[str, Any]:\\n        return asdict(self)\\n\\n\\n@dataclass(frozen=True)\\nclass ClaimRecord:\\n    \\"\\"\\"Canonical claim record emitted by every dataset loader.\\"\\"\\"\\n\\n    claim_id: str\\n    claim: str\\n    label: str | None\\n    source_urls: tuple[str, ...] = ()\\n    evidence: tuple[EvidenceItem, ...] = ()\\n    metadata: dict[str, Any] = field(default_factory=dict)\\n\\n    def to_dict(self) -> dict[str, Any]:\\n        data = asdict(self)\\n        data[\\"source_urls\\"] = list(self.source_urls)\\n        return data\\n", "src/data/snopes.py": "\\"\\"\\"DisinfoMM Snopes filtering and conservative external-label normalization.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport csv\\nfrom collections import Counter\\nfrom pathlib import Path\\nfrom typing import Any, Iterable\\nfrom urllib.parse import urlparse\\n\\nfrom src.data.schema import ClaimRecord\\n\\nSNOPES_EXTERNAL_LABELS = (\\"Supported\\", \\"Refuted\\", \\"Not Enough Evidence\\")\\n\\n\\ndef normalize_snopes_label(evaluation: str | None) -> str | None:\\n    \\"\\"\\"Map DisinfoMM\'s harmonized verdict to the external three-class task.\\n\\n    The archival CSV retains spelling/capitalization variants. `None` means the\\n    source verdict cannot be mapped safely and should be excluded with an audit\\n    record rather than forced into a project class.\\n    \\"\\"\\"\\n    value = \\" \\".join((evaluation or \\"\\").strip().casefold().split())\\n    value = value.replace(\\"moslty\\", \\"mostly\\").replace(\\"exagerated\\", \\"exaggerated\\")\\n    if value == \\"true\\" or value.startswith(\\"mostly true\\"):\\n        return \\"Supported\\"\\n    if value == \\"false\\" or value.startswith(\\"mostly false\\"):\\n        return \\"Refuted\\"\\n    if value == \\"incomplete\\":\\n        return \\"Not Enough Evidence\\"\\n    return None\\n\\n\\ndef parse_source_urls(value: str | None) -> tuple[str, ...]:\\n    \\"\\"\\"Parse the historical comma-separated cited-source field conservatively.\\"\\"\\"\\n    urls: list[str] = []\\n    for candidate in (value or \\"\\").split(\\",\\"):\\n        url = candidate.strip().rstrip(\\".,;:)\\")\\n        parsed = urlparse(url)\\n        if parsed.scheme not in {\\"http\\", \\"https\\"} or not parsed.netloc:\\n            continue\\n        if url not in urls:\\n            urls.append(url)\\n    return tuple(urls)\\n\\n\\ndef row_to_record(row: dict[str, str], source_index: int) -> ClaimRecord:\\n    claim = (row.get(\\"Claim\\") or \\"\\").strip()\\n    if not claim:\\n        raise ValueError(\\"empty_claim\\")\\n    label = normalize_snopes_label(row.get(\\"Evaluation\\"))\\n    if label is None:\\n        raise ValueError(\\"unmappable_evaluation\\")\\n    source_id = (row.get(\\"Num\\") or str(source_index)).strip()\\n    return ClaimRecord(\\n        claim_id=f\\"snopes:{source_id}\\",\\n        claim=claim,\\n        label=label,\\n        source_urls=parse_source_urls(row.get(\\"Article sources\\")),\\n        metadata={\\n            \\"dataset\\": \\"DisinfoMM\\",\\n            \\"source\\": \\"snopes\\",\\n            \\"source_index\\": source_index,\\n            \\"source_id\\": source_id,\\n            \\"date\\": row.get(\\"Date\\"),\\n            \\"source_evaluation\\": row.get(\\"Evaluation\\"),\\n            \\"source_label\\": row.get(\\"Label\\"),\\n            \\"fact_check_url\\": row.get(\\"Article Link\\"),\\n            \\"declaration_url\\": row.get(\\"Declaration Link\\"),\\n            \\"keywords\\": row.get(\\"Keywords\\"),\\n            \\"tags\\": row.get(\\"Tags\\"),\\n            # Deliberately do not retain the post-hoc Explanation as model input.\\n            \\"has_explanation\\": bool((row.get(\\"Explanation\\") or \\"\\").strip()),\\n        },\\n    )\\n\\n\\ndef load_snopes_records(\\n    path: str | Path,\\n) -> tuple[list[ClaimRecord], dict[str, Any], list[dict[str, Any]]]:\\n    \\"\\"\\"Load only English Snopes rows and return records plus an audit trail.\\"\\"\\"\\n    source_path = Path(path)\\n    records: list[ClaimRecord] = []\\n    rejections: list[dict[str, Any]] = []\\n    raw_evaluations: Counter[str] = Counter()\\n    source_rows = 0\\n    snopes_rows = 0\\n    with source_path.open(\\"r\\", encoding=\\"utf-8-sig\\", newline=\\"\\") as handle:\\n        reader = csv.DictReader(handle)\\n        required = {\\"Num\\", \\"Website\\", \\"Claim\\", \\"Evaluation\\", \\"Article sources\\"}\\n        missing = required - set(reader.fieldnames or [])\\n        if missing:\\n            raise ValueError(f\\"DisinfoMM is missing required columns: {sorted(missing)}\\")\\n        for source_index, row in enumerate(reader):\\n            source_rows += 1\\n            if (row.get(\\"Website\\") or \\"\\").strip().casefold() != \\"snopes\\":\\n                continue\\n            snopes_rows += 1\\n            raw_evaluations[(row.get(\\"Evaluation\\") or \\"\\").strip()] += 1\\n            try:\\n                records.append(row_to_record(row, source_index))\\n            except ValueError as error:\\n                rejections.append(\\n                    {\\n                        \\"source_index\\": source_index,\\n                        \\"source_id\\": row.get(\\"Num\\"),\\n                        \\"reason\\": str(error),\\n                        \\"source_evaluation\\": row.get(\\"Evaluation\\"),\\n                    }\\n                )\\n    claim_ids = [record.claim_id for record in records]\\n    if len(claim_ids) != len(set(claim_ids)):\\n        raise ValueError(\\"Duplicate DisinfoMM Snopes record IDs after filtering\\")\\n    audit = {\\n        \\"dataset\\": \\"DisinfoMM\\",\\n        \\"filter\\": \\"Website == snopes\\",\\n        \\"source_row_count\\": source_rows,\\n        \\"snopes_row_count\\": snopes_rows,\\n        \\"accepted_record_count\\": len(records),\\n        \\"label_counts\\": dict(sorted(Counter(record.label for record in records).items())),\\n        \\"raw_evaluation_counts\\": dict(sorted(raw_evaluations.items())),\\n        \\"records_with_source_urls\\": sum(bool(record.source_urls) for record in records),\\n        \\"rejection_count\\": len(rejections),\\n    }\\n    return records, audit, rejections\\n", "src/evaluation/__init__.py": "\\"\\"\\"Shared evaluation functions for project baselines and models.\\"\\"\\"\\n", "src/evaluation/classification.py": "\\"\\"\\"Standardized four-way classification reporting.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nfrom collections.abc import Iterable, Sequence\\nfrom typing import Any\\n\\nfrom sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score\\n\\nfrom src.data.averitec import CANONICAL_LABELS\\n\\n\\ndef evaluate_predictions(\\n    predictions: Iterable[dict[str, Any]], labels: Sequence[str] = CANONICAL_LABELS\\n) -> dict[str, Any]:\\n    \\"\\"\\"Compute standardized verdict metrics from prediction artifacts.\\"\\"\\"\\n    materialized = list(predictions)\\n    if not materialized:\\n        raise ValueError(\\"Cannot evaluate an empty prediction list\\")\\n    true_labels = [str(row[\\"true_label\\"]) for row in materialized]\\n    predicted_labels = [str(row[\\"predicted_label\\"]) for row in materialized]\\n    invalid = (set(true_labels) | set(predicted_labels)) - set(labels)\\n    if invalid:\\n        raise ValueError(f\\"Predictions contain invalid labels: {sorted(invalid)}\\")\\n    report: dict[str, Any] = {\\n        \\"claim_count\\": len(materialized),\\n        \\"macro_f1\\": float(f1_score(true_labels, predicted_labels, labels=labels, average=\\"macro\\", zero_division=0)),\\n        \\"accuracy\\": float(accuracy_score(true_labels, predicted_labels)),\\n        \\"per_class\\": classification_report(\\n            true_labels,\\n            predicted_labels,\\n            labels=labels,\\n            output_dict=True,\\n            zero_division=0,\\n        ),\\n        \\"confusion_matrix\\": {\\n            \\"labels\\": list(labels),\\n            \\"matrix\\": confusion_matrix(true_labels, predicted_labels, labels=labels).tolist(),\\n        },\\n    }\\n    supported = \\"Supported\\"\\n    report[\\"decision_error_rates\\"] = {\\n        \\"false_reassurance_count\\": sum(true != supported and predicted == supported for true, predicted in zip(true_labels, predicted_labels, strict=True)),\\n        \\"false_reassurance_rate\\": sum(true != supported and predicted == supported for true, predicted in zip(true_labels, predicted_labels, strict=True)) / len(materialized),\\n        \\"false_alarm_count\\": sum(true == supported and predicted != supported for true, predicted in zip(true_labels, predicted_labels, strict=True)),\\n        \\"false_alarm_rate\\": sum(true == supported and predicted != supported for true, predicted in zip(true_labels, predicted_labels, strict=True)) / len(materialized),\\n    }\\n    if all(isinstance(row.get(\\"probabilities\\"), dict) for row in materialized):\\n        report[\\"calibration\\"] = calibration_metrics(materialized, labels)\\n    return report\\n\\n\\ndef calibration_metrics(predictions: Sequence[dict[str, Any]], labels: Sequence[str], bin_count: int = 10) -> dict[str, float | int]:\\n    \\"\\"\\"Compute multiclass Brier score and ECE from model probabilities.\\n\\n    LLM self-reported confidence is intentionally not accepted as a probability\\n    distribution and therefore does not receive a calibration score.\\n    \\"\\"\\"\\n    brier_total = 0.0\\n    confidence_bins: list[list[tuple[float, float]]] = [[] for _ in range(bin_count)]\\n    for row in predictions:\\n        probabilities = row[\\"probabilities\\"]\\n        values = {label: float(probabilities.get(label, 0.0)) for label in labels}\\n        brier_total += sum((values[label] - float(row[\\"true_label\\"] == label)) ** 2 for label in labels)\\n        predicted_label, confidence = max(values.items(), key=lambda item: item[1])\\n        bin_index = min(int(confidence * bin_count), bin_count - 1)\\n        confidence_bins[bin_index].append((confidence, float(predicted_label == row[\\"true_label\\"])))\\n    ece = 0.0\\n    for values in confidence_bins:\\n        if not values:\\n            continue\\n        avg_confidence = sum(value[0] for value in values) / len(values)\\n        avg_accuracy = sum(value[1] for value in values) / len(values)\\n        ece += len(values) / len(predictions) * abs(avg_confidence - avg_accuracy)\\n    return {\\"multiclass_brier_score\\": brier_total / len(predictions), \\"expected_calibration_error\\": ece, \\"ece_bin_count\\": bin_count}\\n", "src/evaluation/end_to_end.py": "\\"\\"\\"Diagnostic reports joining retrieval coverage and final verdict predictions.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nfrom collections.abc import Iterable\\nfrom typing import Any\\n\\nfrom src.data.averitec import CANONICAL_LABELS\\nfrom src.evaluation.classification import evaluate_predictions\\nfrom src.retrieval.metrics import evidence_hit_at_k\\n\\n\\ndef annotate_retrieval_coverage(\\n    predictions: Iterable[dict[str, Any]], rankings: Iterable[dict[str, Any]], qrels: dict[str, list[str]], k: int = 3\\n) -> list[dict[str, Any]]:\\n    \\"\\"\\"Attach an auditable evidence-coverage status to each verdict prediction.\\n\\n    A prediction may explicitly identify the passages the verifier used.  This\\n    matters for the LLM reranking baseline: its selected three are the evidence\\n    set being diagnosed, rather than the original BM25 top three.\\n    \\"\\"\\"\\n    ranking_by_claim = {str(row[\\"claim_id\\"]): row for row in rankings}\\n    annotations = []\\n    for prediction in predictions:\\n        claim_id = str(prediction[\\"claim_id\\"])\\n        ranking = ranking_by_claim.get(claim_id)\\n        evidence_ids = prediction.get(\\"evidence_passage_ids\\")\\n        if evidence_ids is None and ranking is not None:\\n            evidence_ids = [item[\\"passage_id\\"] for item in ranking.get(\\"retrieved\\") or []]\\n        if evidence_ids is None or claim_id not in qrels:\\n            status = \\"unjudged\\"\\n        elif evidence_hit_at_k([str(item) for item in evidence_ids], qrels[claim_id], k):\\n            status = \\"evidence_covered\\"\\n        else:\\n            status = \\"evidence_missed\\"\\n        annotations.append(\\n            {\\n                \\"claim_id\\": claim_id,\\n                \\"retrieval_coverage\\": status,\\n                \\"evidence_passage_ids_used\\": [str(item) for item in evidence_ids or []],\\n            }\\n        )\\n    return annotations\\n\\n\\ndef evaluate_by_retrieval_coverage(\\n    predictions: Iterable[dict[str, Any]], rankings: Iterable[dict[str, Any]], qrels: dict[str, list[str]], k: int = 3\\n) -> dict[str, Any]:\\n    \\"\\"\\"Report verdict quality overall and conditioned on URL/passage coverage.\\n\\n    This is diagnostic conditioning, not a causal claim that retrieval alone\\n    caused every miss or hit.\\n    \\"\\"\\"\\n    materialized_predictions = list(predictions)\\n    coverage_by_claim = {\\n        row[\\"claim_id\\"]: row[\\"retrieval_coverage\\"]\\n        for row in annotate_retrieval_coverage(materialized_predictions, rankings, qrels, k)\\n    }\\n    covered = [row for row in materialized_predictions if coverage_by_claim[row[\\"claim_id\\"]] == \\"evidence_covered\\"]\\n    missed = [row for row in materialized_predictions if coverage_by_claim[row[\\"claim_id\\"]] == \\"evidence_missed\\"]\\n    unjudged = [row for row in materialized_predictions if coverage_by_claim[row[\\"claim_id\\"]] == \\"unjudged\\"]\\n    report: dict[str, Any] = {\\n        \\"k\\": k,\\n        \\"judged_prediction_count\\": len(covered) + len(missed),\\n        \\"unjudged_prediction_count\\": len(unjudged),\\n        \\"retrieval_covered_count\\": len(covered),\\n        \\"retrieval_missed_count\\": len(missed),\\n        \\"warning\\": \\"Conditioning on retrieval coverage is diagnostic, not causal attribution.\\",\\n    }\\n    if covered:\\n        report[\\"verdict_metrics_when_evidence_covered\\"] = evaluate_predictions(covered, CANONICAL_LABELS)\\n    if missed:\\n        report[\\"verdict_metrics_when_evidence_missed\\"] = evaluate_predictions(missed, CANONICAL_LABELS)\\n    return report\\n", "src/evaluation/splits.py": "\\"\\"\\"Train-only holdouts for diagnostics; never uses official dev claims.\\"\\"\\"\\nfrom __future__ import annotations\\n\\nfrom datetime import datetime\\nfrom math import ceil\\nfrom typing import Any\\n\\nimport numpy as np\\nfrom sklearn.model_selection import train_test_split\\n\\n\\ndef parse_claim_date(value: Any) -> str | None:\\n    if not value:\\n        return None\\n    text = str(value).strip()\\n    for fmt in (\\"%Y-%m-%d\\", \\"%d-%m-%Y\\", \\"%d/%m/%Y\\", \\"%Y/%m/%d\\", \\"%m/%d/%Y\\", \\"%B %d, %Y\\"):\\n        try:\\n            return datetime.strptime(text[:10] if fmt == \\"%Y-%m-%d\\" else text, fmt).date().isoformat()\\n        except ValueError:\\n            pass\\n    return None\\n\\n\\ndef selection_split(records: list[dict], fraction: float, seed: int, policy: str = \\"stratified\\") -> tuple[np.ndarray, np.ndarray]:\\n    indices = np.arange(len(records))\\n    if policy == \\"stratified\\":\\n        return train_test_split(indices, test_size=fraction, random_state=seed, stratify=[r[\\"label\\"] for r in records])\\n    if policy != \\"temporal\\":\\n        raise ValueError(f\\"Unknown validation policy: {policy}\\")\\n    dates = [parse_claim_date(r.get(\\"metadata\\", {}).get(\\"claim_date\\")) for r in records]\\n    if any(date is None for date in dates):\\n        raise ValueError(\\"Temporal holdout requires a parseable claim_date for every train claim; audit missing dates first\\")\\n    ordered = sorted(indices, key=lambda i: (dates[i], str(records[i][\\"claim_id\\"])))\\n    boundary = dates[ordered[max(1, len(records) - ceil(fraction * len(records)))]]\\n    train = np.asarray([i for i in ordered if dates[i] < boundary], dtype=int)\\n    holdout = np.asarray([i for i in ordered if dates[i] >= boundary], dtype=int)\\n    if not len(train) or not len(holdout):\\n        raise ValueError(\\"Temporal dates do not admit a nonempty earlier/later split\\")\\n    return train, holdout\\n", "src/external/__init__.py": "\\"\\"\\"External-corpus preparation with provenance and leakage safeguards.\\"\\"\\"\\n", "src/external/sampling.py": "\\"\\"\\"Deterministic, label-stratified sampling for manageable external corpora.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport random\\nfrom collections import Counter, defaultdict\\nfrom collections.abc import Iterable, Sequence\\nfrom typing import Any\\n\\n\\ndef stratified_sample(\\n    records: Iterable[dict[str, Any]], labels: Sequence[str], total_size: int, seed: int\\n) -> list[dict[str, Any]]:\\n    \\"\\"\\"Sample as evenly as possible across labels without oversampling records.\\n\\n    Selection uses labels only to create an evaluation set with meaningful\\n    coverage of rare classes. Labels remain unavailable to every retriever and\\n    verifier at inference time.\\n    \\"\\"\\"\\n    if total_size < 1:\\n        raise ValueError(\\"total_size must be positive\\")\\n    buckets: dict[str, list[dict[str, Any]]] = defaultdict(list)\\n    for record in records:\\n        label = record.get(\\"label\\")\\n        if label in labels:\\n            buckets[str(label)].append(record)\\n    unknown = set(labels) - set(buckets)\\n    if unknown:\\n        raise ValueError(f\\"Cannot sample absent labels: {sorted(unknown)}\\")\\n    rng = random.Random(seed)\\n    for bucket in buckets.values():\\n        rng.shuffle(bucket)\\n\\n    selected: list[dict[str, Any]] = []\\n    remaining = total_size\\n    active_labels = list(labels)\\n    while active_labels and remaining:\\n        quota = max(1, remaining // len(active_labels))\\n        next_active: list[str] = []\\n        for label in active_labels:\\n            take = min(quota, len(buckets[label]))\\n            selected.extend(buckets[label][:take])\\n            buckets[label] = buckets[label][take:]\\n            remaining -= take\\n            if buckets[label]:\\n                next_active.append(label)\\n            if not remaining:\\n                break\\n        active_labels = next_active\\n    if remaining:\\n        raise ValueError(\\"Requested more records than are available\\")\\n    return sorted(selected, key=lambda record: str(record[\\"claim_id\\"]))\\n\\n\\ndef sample_audit(records: Iterable[dict[str, Any]]) -> dict[str, Any]:\\n    materialized = list(records)\\n    return {\\n        \\"record_count\\": len(materialized),\\n        \\"label_counts\\": dict(sorted(Counter(str(record[\\"label\\"]) for record in materialized).items())),\\n        \\"source_url_count\\": sum(len(record.get(\\"source_urls\\") or []) for record in materialized),\\n    }\\n", "src/external/snopes_manifest.py": "\\"\\"\\"Build an auditable cited-source manifest before any web fetching occurs.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nfrom collections import Counter\\nfrom collections.abc import Iterable\\nfrom typing import Any\\nfrom urllib.parse import urlparse\\n\\n\\n# Snopes itself is always prohibited as evaluation evidence. Other fact-checking\\n# domains are flagged for review rather than silently discarded: they may be\\n# useful provenance but often contain verdict leakage.\\nREJECTED_EVIDENCE_DOMAINS = {\\"snopes.com\\", \\"www.snopes.com\\"}\\nFACT_CHECK_REVIEW_DOMAINS = {\\n    \\"politifact.com\\",\\n    \\"www.politifact.com\\",\\n    \\"factcheck.org\\",\\n    \\"www.factcheck.org\\",\\n    \\"fullfact.org\\",\\n    \\"fullfact.org\\",\\n    \\"leadstories.com\\",\\n    \\"www.leadstories.com\\",\\n    \\"checkyourfact.com\\",\\n    \\"www.checkyourfact.com\\",\\n    \\"africacheck.org\\",\\n    \\"www.africacheck.org\\",\\n}\\n\\n\\ndef domain_for_url(url: str) -> str:\\n    return urlparse(url).netloc.casefold().removeprefix(\\"www.\\")\\n\\n\\ndef source_screening_status(domain: str) -> str:\\n    normalized = domain.casefold().removeprefix(\\"www.\\")\\n    if normalized in {item.removeprefix(\\"www.\\") for item in REJECTED_EVIDENCE_DOMAINS}:\\n        return \\"excluded_fact_check_source\\"\\n    if normalized in {item.removeprefix(\\"www.\\") for item in FACT_CHECK_REVIEW_DOMAINS}:\\n        return \\"requires_fact_check_review\\"\\n    return \\"unreviewed\\"\\n\\n\\ndef build_source_manifest(records: Iterable[dict[str, Any]]) -> tuple[list[dict[str, Any]], dict[str, Any]]:\\n    \\"\\"\\"Deduplicate cited URLs while retaining every claim that cited each URL.\\"\\"\\"\\n    by_url: dict[str, dict[str, Any]] = {}\\n    for record in records:\\n        for url in record.get(\\"source_urls\\") or []:\\n            url = str(url)\\n            domain = domain_for_url(url)\\n            entry = by_url.setdefault(\\n                url,\\n                {\\n                    \\"url\\": url,\\n                    \\"domain\\": domain,\\n                    \\"screening_status\\": source_screening_status(domain),\\n                    \\"claim_ids\\": [],\\n                },\\n            )\\n            claim_id = str(record[\\"claim_id\\"])\\n            if claim_id not in entry[\\"claim_ids\\"]:\\n                entry[\\"claim_ids\\"].append(claim_id)\\n    manifest = sorted(by_url.values(), key=lambda row: row[\\"url\\"])\\n    status_counts = Counter(row[\\"screening_status\\"] for row in manifest)\\n    domain_counts = Counter(row[\\"domain\\"] for row in manifest)\\n    audit = {\\n        \\"unique_source_url_count\\": len(manifest),\\n        \\"screening_status_counts\\": dict(sorted(status_counts.items())),\\n        \\"top_domains\\": [\\n            {\\"domain\\": domain, \\"url_count\\": count}\\n            for domain, count in domain_counts.most_common(25)\\n        ],\\n    }\\n    return manifest, audit\\n", "src/retrieval/__init__.py": "\\"\\"\\"Evidence retrieval implementations and metrics.\\"\\"\\"\\n", "src/retrieval/answer_aware.py": "\\"\\"\\"Pure helpers for extractive, question-conditioned evidence selection.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport math\\nfrom typing import Any\\n\\nfrom src.data.corpus import canonical_url\\n\\n\\nFEATURE_KEYS = (\\n    \\"max_answer_margin\\",\\n    \\"mean_answer_margin\\",\\n    \\"bm25_reciprocal_rank\\",\\n    \\"question_guided_top3\\",\\n    \\"answer_word_count\\",\\n)\\n\\n\\ndef choose_candidates(\\n    bm25_row: dict[str, Any], question_row: dict[str, Any], *, limit: int, bm25_keep: int\\n) -> list[dict[str, Any]]:\\n    \\"\\"\\"Use only prediction-time rankings; retain a lexical safety net.\\"\\"\\"\\n    if limit < 1 or not 0 <= bm25_keep <= limit:\\n        raise ValueError(\\"Require limit > 0 and 0 <= bm25_keep <= limit\\")\\n    bm25 = bm25_row.get(\\"retrieved\\") or []\\n    by_id = {str(item[\\"passage_id\\"]): item for item in bm25}\\n    if len(by_id) != len(bm25):\\n        raise ValueError(\\"Duplicate passage ID in BM25 ranking\\")\\n    selected = {str(item[\\"passage_id\\"]) for item in (question_row.get(\\"retrieved\\") or [])[:3]}\\n    ordered_ids = [str(item[\\"passage_id\\"]) for item in bm25[:bm25_keep]]\\n    ordered_ids += [str(item[\\"passage_id\\"]) for item in (question_row.get(\\"retrieved\\") or [])[:3]]\\n    ordered_ids += [str(value) for value in question_row.get(\\"shortlist_passage_ids\\") or []]\\n    ordered_ids += [str(item[\\"passage_id\\"]) for item in bm25]\\n    seen: set[str] = set()\\n    chosen: list[dict[str, Any]] = []\\n    bm25_positions = {str(item[\\"passage_id\\"]): index + 1 for index, item in enumerate(bm25)}\\n    for passage_id in ordered_ids:\\n        if passage_id in seen or passage_id not in by_id:\\n            continue\\n        seen.add(passage_id)\\n        item = by_id[passage_id]\\n        if not str(item.get(\\"text\\") or \\"\\").strip():\\n            continue\\n        chosen.append({\\n            **item,\\n            \\"bm25_rank\\": bm25_positions[passage_id],\\n            \\"question_guided_top3\\": int(passage_id in selected),\\n        })\\n        if len(chosen) >= limit:\\n            break\\n    return chosen\\n\\n\\ndef best_extractable_span(\\n    passage: str,\\n    offsets: list[tuple[int, int]],\\n    sequence_ids: list[int | None],\\n    start_logits: list[float],\\n    end_logits: list[float],\\n    *,\\n    cls_index: int,\\n    max_answer_tokens: int,\\n    max_logit_candidates: int = 20,\\n) -> tuple[str, float]:\\n    \\"\\"\\"Return a verbatim span and SQuAD2-style answer-vs-null margin.\\"\\"\\"\\n    if not (len(offsets) == len(sequence_ids) == len(start_logits) == len(end_logits)):\\n        raise ValueError(\\"Token arrays must have identical lengths\\")\\n    null_score = start_logits[cls_index] + end_logits[cls_index]\\n    if max_answer_tokens < 1 or max_logit_candidates < 1:\\n        raise ValueError(\\"Span and logit candidate limits must be positive\\")\\n    context_tokens = [index for index, segment in enumerate(sequence_ids)\\n                      if segment == 1 and offsets[index][1] > offsets[index][0]]\\n    top_starts = sorted(context_tokens, key=lambda index: -start_logits[index])[:max_logit_candidates]\\n    top_ends = sorted(context_tokens, key=lambda index: -end_logits[index])[:max_logit_candidates]\\n    best_score = -math.inf\\n    best_text = \\"\\"\\n    for start in top_starts:\\n        for end in top_ends:\\n            if end < start or end >= start + max_answer_tokens:\\n                continue\\n            begin_char, end_char = offsets[start][0], offsets[end][1]\\n            if not 0 <= begin_char < end_char <= len(passage):\\n                continue\\n            score = start_logits[start] + end_logits[end]\\n            if score > best_score:\\n                best_score = score\\n                best_text = passage[begin_char:end_char]\\n    if not best_text:\\n        return \\"\\", -100.0\\n    return best_text, float(best_score - null_score)\\n\\n\\ndef source_excerpt(passage: str, answer: str, *, max_chars: int = 330) -> str:\\n    \\"\\"\\"Keep the exact source substring around a candidate answer.\\"\\"\\"\\n    if max_chars < 1:\\n        raise ValueError(\\"max_chars must be positive\\")\\n    position = passage.find(answer) if answer else -1\\n    if position < 0:\\n        return passage[:max_chars].strip()\\n    start = max(0, position - max(0, (max_chars - len(answer)) // 2))\\n    end = min(len(passage), start + max_chars)\\n    start = max(0, end - max_chars)\\n    return passage[start:end].strip()\\n\\n\\ndef qa_text(question: str, answer: str, passage: str, margin: float) -> str:\\n    excerpt = source_excerpt(passage, answer if margin > 0 else \\"\\")\\n    if margin > 0 and answer:\\n        return f\\"Question: {question}\\\\nCandidate answer (verbatim): {answer}\\\\nSource excerpt: {excerpt}\\"\\n    return f\\"Question: {question}\\\\nNo confident extractive answer. Source excerpt: {excerpt}\\"\\n\\n\\ndef feature_vector(item: dict[str, Any]) -> list[float]:\\n    return [float(item[key]) for key in FEATURE_KEYS]\\n\\n\\ndef rank_with_source_diversity(items: list[dict[str, Any]], scores: list[float], *, k: int = 3) -> list[dict[str, Any]]:\\n    if len(items) != len(scores):\\n        raise ValueError(\\"One score is required per passage\\")\\n    ordered = sorted(range(len(items)), key=lambda index: (-scores[index], str(items[index][\\"passage_id\\"])))\\n    first: list[int] = []\\n    seen_sources: set[str] = set()\\n    for index in ordered:\\n        item = items[index]\\n        source = canonical_url(item.get(\\"url\\")) or str(item[\\"passage_id\\"])\\n        if source not in seen_sources:\\n            seen_sources.add(source)\\n            first.append(index)\\n            if len(first) == k:\\n                break\\n    if len(first) < min(k, len(items)):\\n        first.extend(index for index in ordered if index not in first)\\n    indices = first[:k] + [index for index in ordered if index not in first[:k]]\\n    return [{**items[index], \\"selector_score\\": float(scores[index])} for index in indices]\\n", "src/retrieval/bm25.py": "\\"\\"\\"Deterministic BM25 retrieval over a static passage corpus.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport heapq\\nimport re\\nfrom collections.abc import Sequence\\n\\nfrom rank_bm25 import BM25Okapi\\n\\nfrom src.data.schema import EvidenceItem\\n\\n_TOKEN_PATTERN = re.compile(r\\"\\\\b\\\\w+\\\\b\\", flags=re.UNICODE)\\n\\n\\ndef tokenize(text: str) -> list[str]:\\n    \\"\\"\\"A transparent, deterministic tokenizer suitable for a lexical baseline.\\"\\"\\"\\n    return _TOKEN_PATTERN.findall(text.casefold())\\n\\n\\nclass BM25Retriever:\\n    \\"\\"\\"BM25 retriever with stable tie-breaking by passage ID.\\"\\"\\"\\n\\n    def __init__(self, corpus: Sequence[EvidenceItem]) -> None:\\n        if not corpus:\\n            raise ValueError(\\"BM25 corpus cannot be empty\\")\\n        self.corpus = tuple(corpus)\\n        self._bm25 = BM25Okapi([tokenize(item.text) for item in self.corpus])\\n\\n    def retrieve(self, claim: str, k: int = 3) -> list[EvidenceItem]:\\n        if k < 1:\\n            raise ValueError(\\"k must be at least 1\\")\\n        scores = self._bm25.get_scores(tokenize(claim))\\n        # Ranking a large candidate pool only needs the first k results.  A\\n        # full sort is O(n log n), whereas nsmallest is O(n log k).  This is\\n        # exactly the same ordering rule as the previous full sort, including\\n        # deterministic passage-ID tie breaking, but avoids sorting millions\\n        # of unreturned candidates in AVeriTeC pools.\\n        ranked_indices = heapq.nsmallest(\\n            min(k, len(self.corpus)),\\n            range(len(self.corpus)),\\n            key=lambda index: (-float(scores[index]), self.corpus[index].passage_id),\\n        )\\n        return [\\n            EvidenceItem(\\n                passage_id=self.corpus[index].passage_id,\\n                text=self.corpus[index].text,\\n                url=self.corpus[index].url,\\n                domain=self.corpus[index].domain,\\n                score=float(scores[index]),\\n                metadata=self.corpus[index].metadata,\\n            )\\n            for index in ranked_indices\\n        ]\\n", "src/retrieval/metrics.py": "\\"\\"\\"Retrieval metrics for passage-level relevance judgments.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nfrom collections.abc import Iterable, Mapping, Sequence\\n\\n\\ndef reciprocal_rank(retrieved_ids: Sequence[str], relevant_ids: Iterable[str]) -> float:\\n    relevant = set(relevant_ids)\\n    for rank, passage_id in enumerate(retrieved_ids, start=1):\\n        if passage_id in relevant:\\n            return 1.0 / rank\\n    return 0.0\\n\\n\\ndef evidence_hit_at_k(retrieved_ids: Sequence[str], relevant_ids: Iterable[str], k: int) -> float:\\n    \\"\\"\\"Whether at least one judged evidence unit appears in the first *k* results.\\"\\"\\"\\n    relevant = set(relevant_ids)\\n    if not relevant:\\n        raise ValueError(\\"Evidence hit rate is undefined without relevance judgments\\")\\n    return float(bool(set(retrieved_ids[:k]) & relevant))\\n\\n\\ndef passage_recall_at_k(retrieved_ids: Sequence[str], relevant_ids: Iterable[str], k: int) -> float:\\n    \\"\\"\\"Fraction of all judged relevant passages returned in the first *k* results.\\"\\"\\"\\n    relevant = set(relevant_ids)\\n    if not relevant:\\n        raise ValueError(\\"Passage recall is undefined without relevance judgments\\")\\n    return len(set(retrieved_ids[:k]) & relevant) / len(relevant)\\n\\n\\ndef precision_at_k(retrieved_ids: Sequence[str], relevant_ids: Iterable[str], k: int) -> float:\\n    \\"\\"\\"Relevant retrieved items divided by k; missing ranks count as non-relevant.\\"\\"\\"\\n    if k < 1:\\n        raise ValueError(\\"k must be at least one\\")\\n    return len(set(retrieved_ids[:k]) & set(relevant_ids)) / k\\n\\n\\ndef recall_at_k(retrieved_ids: Sequence[str], relevant_ids: Iterable[str], k: int) -> float:\\n    \\"\\"\\"Backward-compatible alias for the project-defined evidence hit metric.\\"\\"\\"\\n    return evidence_hit_at_k(retrieved_ids, relevant_ids, k)\\n\\n\\ndef evaluate_rankings(\\n    rankings: Mapping[str, Sequence[str]], qrels: Mapping[str, Iterable[str]], k: int = 3\\n) -> dict[str, float | int]:\\n    \\"\\"\\"Evaluate only judged claims and report their number explicitly.\\"\\"\\"\\n    judged_ids = sorted(set(rankings) & set(qrels))\\n    if not judged_ids:\\n        raise ValueError(\\"No overlap between rankings and relevance judgments\\")\\n    hit_rates = [evidence_hit_at_k(rankings[claim_id], qrels[claim_id], k) for claim_id in judged_ids]\\n    passage_recalls = [passage_recall_at_k(rankings[claim_id], qrels[claim_id], k) for claim_id in judged_ids]\\n    precisions = [precision_at_k(rankings[claim_id], qrels[claim_id], k) for claim_id in judged_ids]\\n    reciprocal_ranks = [reciprocal_rank(rankings[claim_id], qrels[claim_id]) for claim_id in judged_ids]\\n    return {\\n        f\\"evidence_hit_at_{k}\\": sum(hit_rates) / len(hit_rates),\\n        f\\"passage_recall_at_{k}\\": sum(passage_recalls) / len(passage_recalls),\\n        f\\"precision_at_{k}\\": sum(precisions) / len(precisions),\\n        \\"mrr\\": sum(reciprocal_ranks) / len(reciprocal_ranks),\\n        \\"judged_claim_count\\": len(judged_ids),\\n    }\\n", "src/retrieval/question_guided.py": "\\"\\"\\"Pure, testable parts of question-guided evidence selection.\\n\\nNo labels, gold answers, or qrels are inputs to these functions.\\n\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport json\\nimport math\\nimport re\\nfrom typing import Any\\n\\nfrom src.data.corpus import canonical_url\\n\\n\\nTOKEN = re.compile(r\\"[a-z0-9]+\\", re.IGNORECASE)\\nSENTENCE_BOUNDARY = re.compile(r\\"(?<=[.!?])\\\\s+(?=[A-Z0-9\\\\\\"\'([])\\")\\nSTOPWORDS = {\\n    \\"a\\", \\"an\\", \\"and\\", \\"are\\", \\"as\\", \\"at\\", \\"be\\", \\"by\\", \\"did\\", \\"do\\", \\"does\\", \\"for\\", \\"from\\",\\n    \\"has\\", \\"have\\", \\"how\\", \\"in\\", \\"is\\", \\"it\\", \\"of\\", \\"on\\", \\"or\\", \\"that\\", \\"the\\", \\"their\\",\\n    \\"there\\", \\"this\\", \\"to\\", \\"was\\", \\"were\\", \\"what\\", \\"when\\", \\"where\\", \\"which\\", \\"who\\", \\"why\\",\\n    \\"with\\", \\"would\\", \\"could\\", \\"should\\", \\"claim\\", \\"evidence\\", \\"supports\\", \\"contradicts\\",\\n}\\n\\n\\ndef words(text: str) -> set[str]:\\n    return set(TOKEN.findall(text.lower())) - STOPWORDS\\n\\n\\ndef parse_questions(response: str, claim: str, *, limit: int = 2) -> list[str]:\\n    \\"\\"\\"Accept JSON questions only; fall back to a claim-derived question.\\"\\"\\"\\n    found: list[str] = []\\n    decoder = json.JSONDecoder()\\n    for index, char in enumerate(response):\\n        if char != \\"{\\":\\n            continue\\n        try:\\n            value, _ = decoder.raw_decode(response[index:])\\n        except json.JSONDecodeError:\\n            continue\\n        if isinstance(value, dict) and isinstance(value.get(\\"questions\\"), list):\\n            for item in value[\\"questions\\"]:\\n                if isinstance(item, str):\\n                    question = \\" \\".join(item.split())[:240]\\n                    if 8 <= len(question) and question.lower() not in {q.lower() for q in found}:\\n                        found.append(question)\\n                if len(found) == limit:\\n                    break\\n            break\\n    if len(found) < limit:\\n        return [\\n            f\\"What evidence supports or contradicts this claim: {claim[:180]}?\\",\\n            f\\"Which source verifies the date, number, or entity in this claim: {claim[:180]}?\\",\\n        ][:limit]\\n    return found\\n\\n\\ndef shortlist_candidates(row: dict[str, Any], questions: list[str], *, limit: int, bm25_keep: int) -> list[dict[str, Any]]:\\n    \\"\\"\\"Retain BM25 leaders and candidates matching the question terms.\\"\\"\\"\\n    candidates = list(row.get(\\"retrieved\\") or [])\\n    ids = [str(item[\\"passage_id\\"]) for item in candidates]\\n    if len(ids) != len(set(ids)):\\n        raise ValueError(f\\"Duplicate candidate passage IDs for {row.get(\'claim_id\')}\\")\\n    if limit < 1 or bm25_keep < 0:\\n        raise ValueError(\\"limit must be positive and bm25_keep nonnegative\\")\\n    query_terms = words(\\" \\".join([str(row.get(\\"claim\\") or \\"\\"), *questions]))\\n    leader_indices = list(range(min(bm25_keep, limit, len(candidates))))\\n    scored = []\\n    for index, item in enumerate(candidates):\\n        overlap = len(query_terms & words(str(item.get(\\"text\\") or \\"\\")[:1600]))\\n        scored.append((-overlap, index))\\n    chosen = set(leader_indices)\\n    for _, index in sorted(scored):\\n        if len(chosen) >= min(limit, len(candidates)):\\n            break\\n        chosen.add(index)\\n    return [candidates[index] for index in sorted(chosen)]\\n\\n\\ndef reciprocal_rank_scores(score_matrix: list[list[float]], *, claim_weight: float = 1.0) -> list[float]:\\n    \\"\\"\\"Fuse per-query cross-encoder ranks without comparing raw logits across queries.\\"\\"\\"\\n    if not score_matrix:\\n        return []\\n    size = len(score_matrix[0])\\n    if any(len(row) != size or any(not math.isfinite(float(value)) for value in row) for row in score_matrix):\\n        raise ValueError(\\"Cross-encoder score matrix is ragged or non-finite\\")\\n    result = [0.0] * size\\n    for query_index, scores in enumerate(score_matrix):\\n        weight = claim_weight if query_index == 0 else 1.0\\n        for rank, index in enumerate(sorted(range(size), key=lambda i: (-scores[i], i)), start=1):\\n            result[index] += weight / (20 + rank)\\n    return result\\n\\n\\ndef select_diverse(candidates: list[dict[str, Any]], scores: list[float], *, k: int = 3) -> list[dict[str, Any]]:\\n    \\"\\"\\"Select at most k unique-source passages, then fill if source count is low.\\"\\"\\"\\n    if len(candidates) != len(scores):\\n        raise ValueError(\\"Candidate and score counts differ\\")\\n    order = sorted(range(len(candidates)), key=lambda i: (-scores[i], str(candidates[i][\\"passage_id\\"])))\\n    selected: list[int] = []\\n    seen_sources: set[str] = set()\\n    for index in order:\\n        item = candidates[index]\\n        url = canonical_url(item.get(\\"url\\"))\\n        source = f\\"url:{url}\\" if url else f\\"passage:{item[\'passage_id\']}\\"\\n        if source not in seen_sources:\\n            selected.append(index)\\n            seen_sources.add(source)\\n        if len(selected) == k:\\n            break\\n    if len(selected) < k:\\n        selected.extend(index for index in order if index not in selected)\\n    return [{**candidates[index], \\"score\\": float(scores[index])} for index in selected[:k]]\\n\\n\\ndef extract_sentences(text: str, claim: str, questions: list[str], *, max_chars: int = 900, max_sentences: int = 3) -> str:\\n    \\"\\"\\"Select exact source sentences by term coverage; never generate evidence text.\\"\\"\\"\\n    original = str(text).strip()\\n    if len(original) <= max_chars:\\n        return original\\n    sentences = [item.strip() for item in SENTENCE_BOUNDARY.split(original) if item.strip()]\\n    if len(sentences) < 2:\\n        return original[:max_chars].rsplit(\\" \\", 1)[0] or original[:max_chars]\\n    query_sets = [words(claim), *(words(q) for q in questions)]\\n    scored = []\\n    for index, sentence in enumerate(sentences):\\n        tokens = words(sentence)\\n        coverage = sum(len(tokens & query) / max(1, len(query)) for query in query_sets)\\n        scored.append((-coverage, index))\\n    chosen: list[int] = []\\n    used = 0\\n    for _, index in sorted(scored):\\n        sentence = sentences[index]\\n        if used + len(sentence) + (1 if chosen else 0) > max_chars:\\n            continue\\n        chosen.append(index)\\n        used += len(sentence) + (1 if len(chosen) > 1 else 0)\\n        if len(chosen) == max_sentences:\\n            break\\n    if not chosen:\\n        return original[:max_chars].rsplit(\\" \\", 1)[0] or original[:max_chars]\\n    return \\" \\".join(sentences[index] for index in sorted(chosen))\\n", "src/utils/__init__.py": "\\"\\"\\"Small shared utilities for reproducible experiment scripts.\\"\\"\\"\\n", "src/utils/io.py": "\\"\\"\\"Minimal JSONL helpers used by scripts and tests.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport json\\nfrom pathlib import Path\\nfrom typing import Any, Iterable\\n\\n\\ndef read_jsonl(path: str | Path) -> list[dict[str, Any]]:\\n    return list(iter_jsonl(path))\\n\\n\\ndef iter_jsonl(path: str | Path) -> Iterable[dict[str, Any]]:\\n    \\"\\"\\"Read JSONL lazily for large Colab corpus files.\\"\\"\\"\\n    source = Path(path)\\n    with source.open(\\"r\\", encoding=\\"utf-8\\") as handle:\\n        for line in handle:\\n            if line.strip():\\n                yield json.loads(line)\\n\\n\\ndef write_jsonl(rows: Iterable[dict[str, Any]], path: str | Path) -> None:\\n    destination = Path(path)\\n    destination.parent.mkdir(parents=True, exist_ok=True)\\n    with destination.open(\\"w\\", encoding=\\"utf-8\\", newline=\\"\\\\n\\") as handle:\\n        for row in rows:\\n            handle.write(json.dumps(row, ensure_ascii=False) + \\"\\\\n\\")\\n\\n\\ndef write_json(data: dict[str, Any], path: str | Path) -> None:\\n    \\"\\"\\"Atomically replace a small JSON artifact after a complete write.\\"\\"\\"\\n    destination = Path(path)\\n    destination.parent.mkdir(parents=True, exist_ok=True)\\n    temporary = destination.with_name(f\\"{destination.name}.tmp\\")\\n    temporary.write_text(json.dumps(data, indent=2, ensure_ascii=False) + \\"\\\\n\\", encoding=\\"utf-8\\")\\n    temporary.replace(destination)\\n", "src/utils/progress.py": "\\"\\"\\"Optional notebook-friendly progress bars with a dependency-free fallback.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nfrom typing import Any\\n\\n\\nclass _NoOpProgress:\\n    def update(self, _: int | float = 1) -> None:\\n        return None\\n\\n    def set_postfix(self, **_: Any) -> None:\\n        return None\\n\\n    def __enter__(self) -> \\"_NoOpProgress\\":\\n        return self\\n\\n    def __exit__(self, *_: Any) -> None:\\n        return None\\n\\n\\ndef progress(*, total: int | None, description: str, unit: str = \\"item\\", **kwargs: Any) -> Any:\\n    \\"\\"\\"Return `tqdm.auto` when installed; scripts remain usable without it.\\"\\"\\"\\n    try:\\n        from tqdm.auto import tqdm\\n    except ImportError:\\n        return _NoOpProgress()\\n    return tqdm(total=total, desc=description, unit=unit, **kwargs)\\n", "src/utils/runs.py": "\\"\\"\\"Run manifests that make every reported experiment traceable.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport hashlib\\nimport platform\\nimport re\\nimport subprocess\\nimport sys\\nfrom datetime import UTC, datetime\\nfrom pathlib import Path\\nfrom typing import Any\\n\\nfrom src.utils.io import write_json\\n\\n\\n_RUN_NAME_PATTERN = re.compile(r\\"[A-Za-z0-9][A-Za-z0-9_.-]*$\\")\\n\\n\\ndef sha256_file(path: str | Path) -> str:\\n    digest = hashlib.sha256()\\n    with Path(path).open(\\"rb\\") as handle:\\n        for chunk in iter(lambda: handle.read(1024 * 1024), b\\"\\"):\\n            digest.update(chunk)\\n    return digest.hexdigest()\\n\\n\\ndef git_commit(project_root: Path) -> str | None:\\n    try:\\n        return subprocess.check_output(\\n            [\\"git\\", \\"rev-parse\\", \\"HEAD\\"], cwd=project_root, text=True, stderr=subprocess.DEVNULL\\n        ).strip()\\n    except (OSError, subprocess.CalledProcessError):\\n        return None\\n\\n\\ndef resolve_run_directory(base_directory: str | Path, run_name: str | None) -> Path:\\n    \\"\\"\\"Return a stable output directory and reject path-like run identifiers.\\"\\"\\"\\n    base = Path(base_directory)\\n    if not run_name:\\n        return base\\n    if not _RUN_NAME_PATTERN.fullmatch(run_name):\\n        raise ValueError(\\"run_name must use only letters, numbers, dots, underscores, or hyphens\\")\\n    return base / run_name\\n\\n\\ndef write_run_manifest(\\n    output_path: str | Path,\\n    *,\\n    project_root: Path,\\n    config: dict[str, Any],\\n    input_paths: dict[str, str | Path],\\n) -> dict[str, Any]:\\n    \\"\\"\\"Write metadata without recording credentials or full dataset contents.\\"\\"\\"\\n    inputs: dict[str, dict[str, Any]] = {}\\n    for name, value in input_paths.items():\\n        path = Path(value)\\n        inputs[name] = {\\n            \\"path\\": str(path),\\n            \\"exists\\": path.exists(),\\n            \\"sha256\\": sha256_file(path) if path.is_file() else None,\\n        }\\n    manifest = {\\n        \\"created_at_utc\\": datetime.now(UTC).isoformat(),\\n        \\"git_commit\\": git_commit(project_root),\\n        \\"python_version\\": sys.version,\\n        \\"platform\\": platform.platform(),\\n        \\"config\\": config,\\n        \\"inputs\\": inputs,\\n    }\\n    target = Path(output_path)\\n    target.parent.mkdir(parents=True, exist_ok=True)\\n    write_json(manifest, target)\\n    return manifest\\n", "src/verification/__init__.py": "\\"\\"\\"Claim-verification baselines and later learned verifiers.\\"\\"\\"\\n", "src/verification/llm_zero_shot.py": "\\"\\"\\"Evidence-only zero-shot LLM baseline with strict structured outputs.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport hashlib\\nfrom typing import Any, Literal\\n\\nfrom pydantic import BaseModel, Field\\n\\nfrom src.data.averitec import CANONICAL_LABELS\\n\\nLLMVariant = Literal[\\"classify_top3\\", \\"rerank_top20_and_classify\\"]\\n\\n\\nclass LLMVerdict(BaseModel):\\n    \\"\\"\\"Required model response; confidence is self-reported, not calibrated.\\"\\"\\"\\n\\n    label: Literal[\\"Supported\\", \\"Refuted\\", \\"Not Enough Evidence\\", \\"Conflicting Evidence\\"]\\n    selected_passage_ids: list[str] = Field(min_length=1, max_length=3)\\n    reported_confidence: float = Field(ge=0.0, le=1.0)\\n    brief_rationale: str = Field(min_length=1, max_length=600)\\n\\n\\nclass ClaimOnlyLLMVerdict(BaseModel):\\n    \\"\\"\\"Required response when the model receives no evidence passages.\\"\\"\\"\\n\\n    label: Literal[\\"Supported\\", \\"Refuted\\", \\"Not Enough Evidence\\", \\"Conflicting Evidence\\"]\\n    reported_confidence: float = Field(ge=0.0, le=1.0)\\n    brief_rationale: str = Field(min_length=1, max_length=600)\\n\\n\\nSYSTEM_PROMPT = \\"\\"\\"You are an evidence-grounded claim-verification baseline.\\nUse only the supplied passages. Do not use outside knowledge, the claim\'s likely\\ntruth, or information that is absent from the passages. Return one label:\\n- Supported: evidence establishes the claim.\\n- Refuted: evidence establishes that the claim is false or materially wrong.\\n- Not Enough Evidence: supplied evidence does not establish either conclusion.\\n- Conflicting Evidence: supplied evidence contains material support and refutation.\\n\\nSelect the passage IDs that most directly support your decision. Your reported\\nconfidence is a self-assessment, not a probability. Return the required JSON\\nobject only, with exactly these keys:\\n{\\"label\\": \\"one of the four labels\\", \\"selected_passage_ids\\": [\\"passage ID\\"],\\n \\"reported_confidence\\": 0.0, \\"brief_rationale\\": \\"one brief explanation\\"}.\\"\\"\\"\\n\\n\\nCLAIM_ONLY_SYSTEM_PROMPT = \\"\\"\\"You are a claim-only claim-verification baseline.\\nYou receive a claim but no evidence passages. Do not imply that you retrieved,\\nread, or verified external evidence. Return one label:\\n- Supported\\n- Refuted\\n- Not Enough Evidence\\n- Conflicting Evidence\\n\\nReturn the required JSON object only, with exactly these keys:\\n{\\"label\\": \\"one of the four labels\\", \\"reported_confidence\\": 0.0,\\n \\"brief_rationale\\": \\"one brief explanation\\"}.\\"\\"\\"\\n\\n\\ndef build_user_prompt(\\n    claim: str,\\n    passages: list[dict[str, Any]],\\n    variant: LLMVariant,\\n    max_chars_per_passage: int = 1200,\\n    evidence_description: str = \\"retrieved passages\\",\\n) -> str:\\n    \\"\\"\\"Build the frozen evidence-only request for one claim.\\"\\"\\"\\n    expected = min(3, len(passages))\\n    if expected < 1:\\n        raise ValueError(\\"LLM baseline requires at least one candidate passage\\")\\n    instructions = (\\n        f\\"Rerank the {len(passages)} candidate passages and select exactly {expected} passage IDs before classifying.\\"\\n        if variant == \\"rerank_top20_and_classify\\"\\n        else f\\"Classify using these {len(passages)} {evidence_description} and select 1 to {expected} relevant IDs.\\"\\n    )\\n    rendered_passages = []\\n    for index, passage in enumerate(passages, start=1):\\n        passage_id = str(passage[\\"passage_id\\"])\\n        text = str(passage.get(\\"text\\") or \\"\\").strip()[:max_chars_per_passage]\\n        rendered_passages.append(f\\"[PASSAGE {index} | ID={passage_id}]\\\\n{text}\\")\\n    return f\\"{instructions}\\\\n\\\\n[CLAIM]\\\\n{claim.strip()}\\\\n\\\\n\\" + \\"\\\\n\\\\n\\".join(rendered_passages)\\n\\n\\ndef build_claim_only_prompt(claim: str) -> str:\\n    \\"\\"\\"Build the frozen no-evidence request used by the claim-only diagnostic.\\"\\"\\"\\n    return f\\"Classify this claim without evidence passages.\\\\n\\\\n[CLAIM]\\\\n{claim.strip()}\\"\\n\\n\\ndef prompt_sha256(system_prompt: str, user_prompt: str) -> str:\\n    return hashlib.sha256(f\\"{system_prompt}\\\\n\\\\n{user_prompt}\\".encode(\\"utf-8\\")).hexdigest()\\n\\n\\ndef validate_llm_verdict(result: LLMVerdict, passages: list[dict[str, Any]], variant: LLMVariant) -> None:\\n    \\"\\"\\"Verify selections against the exact candidate set after schema parsing.\\"\\"\\"\\n    candidate_ids = {str(passage[\\"passage_id\\"]) for passage in passages}\\n    selected = result.selected_passage_ids\\n    if len(selected) != len(set(selected)):\\n        raise ValueError(\\"LLM selected duplicate passage IDs\\")\\n    if set(selected) - candidate_ids:\\n        raise ValueError(\\"LLM selected a passage ID outside the supplied candidates\\")\\n    if variant == \\"rerank_top20_and_classify\\" and len(selected) != min(3, len(passages)):\\n        raise ValueError(\\"Reranking variant must select exactly three candidates when available\\")\\n\\n\\ndef call_openai_structured(\\n    *, model: str, system_prompt: str, user_prompt: str, temperature: float\\n) -> tuple[LLMVerdict, dict[str, Any]]:\\n    \\"\\"\\"Call OpenAI\'s supported Python structured-output interface.\\n\\n    The OpenAI package is imported only at call time, letting tests run without a\\n    key and preventing accidental API calls during import.\\n    \\"\\"\\"\\n    try:\\n        from openai import OpenAI\\n    except ImportError as error:\\n        raise RuntimeError(\\"Install requirements-colab.txt before using the LLM baseline.\\") from error\\n    completion = OpenAI().chat.completions.parse(\\n        model=model,\\n        messages=[{\\"role\\": \\"system\\", \\"content\\": system_prompt}, {\\"role\\": \\"user\\", \\"content\\": user_prompt}],\\n        response_format=LLMVerdict,\\n        temperature=temperature,\\n    )\\n    message = completion.choices[0].message\\n    if message.refusal:\\n        raise RuntimeError(f\\"Model refusal: {message.refusal}\\")\\n    if message.parsed is None:\\n        raise RuntimeError(\\"Model returned no parsed structured response\\")\\n    return message.parsed, completion.model_dump(mode=\\"json\\")\\n\\n\\ndef build_prediction(claim_id: str, true_label: str | None, result: LLMVerdict | ClaimOnlyLLMVerdict) -> dict[str, Any]:\\n    if result.label not in CANONICAL_LABELS:\\n        raise ValueError(f\\"Invalid LLM label: {result.label}\\")\\n    return {\\n        \\"claim_id\\": claim_id,\\n        \\"true_label\\": true_label,\\n        \\"predicted_label\\": result.label,\\n        \\"probabilities\\": None,\\n        \\"selected_passage_ids\\": getattr(result, \\"selected_passage_ids\\", []),\\n        \\"reported_confidence\\": result.reported_confidence,\\n        \\"brief_rationale\\": result.brief_rationale,\\n    }\\n", "src/verification/tfidf_logreg.py": "\\"\\"\\"Transparent TF-IDF + multinomial logistic-regression verifier baseline.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nfrom collections.abc import Iterable\\nfrom dataclasses import dataclass\\nfrom typing import Any, Literal\\n\\nfrom sklearn.feature_extraction.text import TfidfVectorizer\\nfrom sklearn.linear_model import LogisticRegression\\nfrom sklearn.pipeline import Pipeline\\n\\nfrom src.data.averitec import CANONICAL_LABELS\\n\\nEvidenceMode = Literal[\\"claim_only\\", \\"gold\\", \\"gold_all\\", \\"retrieved\\"]\\n\\n\\ndef evidence_texts(record: dict[str, Any], evidence_mode: EvidenceMode, max_evidence: int = 3) -> list[str]:\\n    \\"\\"\\"Return the exact evidence texts exposed to a verifier.\\n\\n    `gold_all` is an oracle upper bound and deliberately bypasses the user-facing\\n    three-item cap. Every deployable/end-to-end mode uses at most three items.\\n    \\"\\"\\"\\n    if max_evidence < 1:\\n        raise ValueError(\\"max_evidence must be at least one\\")\\n    if evidence_mode == \\"claim_only\\":\\n        return []\\n    if evidence_mode in {\\"gold\\", \\"gold_all\\"}:\\n        evidence = record.get(\\"evidence\\") or []\\n    elif evidence_mode == \\"retrieved\\":\\n        evidence = record.get(\\"retrieved_evidence\\") or []\\n    else:\\n        raise ValueError(f\\"Unsupported evidence mode: {evidence_mode}\\")\\n    texts = []\\n    for item in evidence:\\n        value = item.get(\\"text\\") if isinstance(item, dict) else item\\n        text = str(value or \\"\\").strip()\\n        if text:\\n            texts.append(text)\\n    return texts if evidence_mode == \\"gold_all\\" else texts[:max_evidence]\\n\\n\\ndef compose_input(record: dict[str, Any], evidence_mode: EvidenceMode, max_evidence: int = 3) -> str:\\n    \\"\\"\\"Format one canonical record for the linear verifier.\\n\\n    Gold answers are allowed only for the isolated gold-evidence experiment. The\\n    end-to-end BM25 experiment will provide retrieved passages in a later phase.\\n    \\"\\"\\"\\n    claim = str(record[\\"claim\\"]).strip()\\n    passages = evidence_texts(record, evidence_mode, max_evidence)\\n    evidence_text = \\"\\\\n\\".join(f\\"[EVIDENCE {index}] {text}\\" for index, text in enumerate(passages, start=1))\\n    return f\\"[CLAIM] {claim}\\\\n{evidence_text}\\" if evidence_text else f\\"[CLAIM] {claim}\\"\\n\\n\\n@dataclass\\nclass TfidfLogRegVerifier:\\n    \\"\\"\\"A reproducible supervised baseline with explicit label ordering.\\"\\"\\"\\n\\n    evidence_mode: EvidenceMode = \\"gold\\"\\n    max_evidence: int = 3\\n    min_df: int = 1\\n    max_features: int | None = None\\n    class_weight: str | None = \\"balanced\\"\\n    random_state: int = 718\\n    max_iter: int = 2000\\n\\n    def __post_init__(self) -> None:\\n        self.pipeline = Pipeline(\\n            [\\n                (\\n                    \\"tfidf\\",\\n                    TfidfVectorizer(\\n                        ngram_range=(1, 2),\\n                        min_df=self.min_df,\\n                        max_features=self.max_features,\\n                        sublinear_tf=True,\\n                    ),\\n                ),\\n                (\\n                    \\"logreg\\",\\n                    LogisticRegression(\\n                        max_iter=self.max_iter,\\n                        class_weight=self.class_weight,\\n                        random_state=self.random_state,\\n                    ),\\n                ),\\n            ]\\n        )\\n\\n    def fit(self, records: Iterable[dict[str, Any]]) -> \\"TfidfLogRegVerifier\\":\\n        materialized = list(records)\\n        labels = [record.get(\\"label\\") for record in materialized]\\n        unknown = set(labels) - set(CANONICAL_LABELS)\\n        if unknown:\\n            raise ValueError(f\\"Training records contain invalid labels: {sorted(unknown)}\\")\\n        self.pipeline.fit([compose_input(record, self.evidence_mode, self.max_evidence) for record in materialized], labels)\\n        return self\\n\\n    def predict_records(self, records: Iterable[dict[str, Any]]) -> list[dict[str, Any]]:\\n        materialized = list(records)\\n        inputs = [compose_input(record, self.evidence_mode, self.max_evidence) for record in materialized]\\n        predicted = self.pipeline.predict(inputs)\\n        probabilities = self.pipeline.predict_proba(inputs)\\n        class_order = list(self.pipeline.named_steps[\\"logreg\\"].classes_)\\n        output = []\\n        for record, label, row_probabilities in zip(materialized, predicted, probabilities, strict=True):\\n            prediction = {\\n                \\"claim_id\\": record[\\"claim_id\\"],\\n                \\"true_label\\": record.get(\\"label\\"),\\n                \\"predicted_label\\": str(label),\\n                \\"probabilities\\": {\\n                    class_label: float(probability)\\n                    for class_label, probability in zip(class_order, row_probabilities, strict=True)\\n                },\\n                \\"evidence_item_count\\": len(evidence_texts(record, self.evidence_mode, self.max_evidence)),\\n            }\\n            if self.evidence_mode == \\"retrieved\\":\\n                prediction[\\"evidence_passage_ids\\"] = [\\n                    str(item[\\"passage_id\\"])\\n                    for item in (record.get(\\"retrieved_evidence\\") or [])[: self.max_evidence]\\n                    if isinstance(item, dict) and item.get(\\"passage_id\\")\\n                ]\\n            output.append(prediction)\\n        return output\\n"}')
SOURCE_SHA256 = 'f1f84a9e703523ed48dc6e05e23bb3bb2ddc8745606ddf562e0e5b36d43069fd'
assert hashlib.sha256(json.dumps(STUDY_SOURCES, sort_keys=True, ensure_ascii=True).encode()).hexdigest() == SOURCE_SHA256
RUNTIME = Path('/content') / ('averitec_limitation_study_' + SOURCE_SHA256[:12])
for relative, source in STUDY_SOURCES.items():
    destination = RUNTIME / relative
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_text(source, encoding='utf-8')
os.chdir(RUNTIME)
sys.path.insert(0, str(RUNTIME))
print('Study source SHA256:', SOURCE_SHA256)
print('Runtime:', RUNTIME)


## 3. Paths and cost controls

Check the printed paths. Only the completed BM25 rankings and saved BM25-trained DistilBERT are required. Existing dev BGE/fusion/top-100 files are optional. No dense training rankings are required. Keep these settings unchanged to resume; use a new `STUDY_NAME` if you change inputs.

In [ ]:
DATA_ROOT = Path('/content/drive/MyDrive/10718_averitec/v2_source_documents')
STUDY_NAME = 'initial_ml_solution_limitation_study_v1'
OUT = DATA_ROOT / 'outputs' / STUDY_NAME
TRAIN_CLAIMS = DATA_ROOT / 'processed/averitec/train.jsonl'
DEV_CLAIMS = DATA_ROOT / 'processed/averitec/dev.jsonl'
DEV_QRELS = DATA_ROOT / 'processed/averitec/dev_qrels.jsonl'
TRAIN_BM25 = DATA_ROOT / 'outputs/bm25/train_bm25_top20_v2/train_rankings.jsonl'
DEV_BM25 = DATA_ROOT / 'outputs/bm25/dev_bm25_top20_v2/dev_rankings.jsonl'
OLD = DATA_ROOT / 'outputs/initial_ml_solution_rerun_bs16'
OLD_VERIFIER = OLD / 'transformer/bm25_top3'
MODEL_DIR = OLD_VERIFIER / 'final_model'
MODEL_CONFIG = OLD_VERIFIER / 'transformer_run_config.json'
DEV100 = DATA_ROOT / 'outputs/initial_ml_solution_answer_aligned_top100_v1/bm25/dev_bm25_top100_v2/dev_rankings.jsonl'
EXTRA_DEV = {
    'bge': OLD / 'dense/dev_bge_small_v15/dev_rankings.jsonl',
    'fusion': OLD / 'hybrid/dev_rrf60/hybrid_rankings.jsonl',
}
RUN_TRAINING = True
REFIT_ALL_VARIANTS = True   # False refits only the holdout-selected evidence model and claim-only control.
RUN_TEMPORAL_CHECK = True  # Train-only; skipped visibly if dates are incomplete.
RUN_STRONGER_ENCODER = False  # Optional later step, using the same evidence passages.
RUN_SEED_CHECK = False        # Optional extra train-holdout seed, not a dev-based search.
RUN_REFIT_DURATION_CHECK = True  # Reuse the head/balanced trial to compare original full-epoch refitting.
AUDIT_CLAIMS = 24             # About 72-96 passage judgments; use 8 for a short first audit.
BATCH_SIZE = 8
from src.utils.io import read_jsonl, write_json
from src.utils.runs import sha256_file
required = [TRAIN_CLAIMS, DEV_CLAIMS, DEV_QRELS, TRAIN_BM25, DEV_BM25,
            MODEL_CONFIG, MODEL_DIR / 'config.json']
missing = [str(p) for p in required if not p.is_file()]
if missing:
    raise FileNotFoundError('Update the paths above to your completed artifacts:\n' + '\n'.join(missing))
if not [*MODEL_DIR.glob('*.safetensors'), *MODEL_DIR.glob('pytorch_model*.bin')]:
    raise FileNotFoundError(f'Missing trained model weights: {MODEL_DIR}')
train_records, dev_records = read_jsonl(TRAIN_CLAIMS), read_jsonl(DEV_CLAIMS)
assert len(train_records) == 3067 and len(dev_records) == 500, 'Expected 3067 usable train and 500 official dev claims.'
settings = json.loads(MODEL_CONFIG.read_text(encoding='utf-8'))
expected_budgets = {'evidence_token_budget_per_passage': 110, 'claim_token_budget': 96, 'max_length': 512}
assert all(settings.get(k) == v for k, v in expected_budgets.items()), 'This study assumes the original 96/110/512 token budgets.'
OUT.mkdir(parents=True, exist_ok=True)
bundle_config = {'source_sha256': SOURCE_SHA256, 'study_name': STUDY_NAME}
from scripts.run_limitation_diagnostics import lock_config
lock_config(OUT / 'notebook_source.json', bundle_config)
for path in required:
    print('Required:', path)
print('Top-100 available:', DEV100.is_file())
print('Optional completed rankings:', {k: p.is_file() for k, p in EXTRA_DEV.items()})
print('All new results/checkpoints:', OUT)

def run_script(label, script, arguments):
    previous_argv = sys.argv[:]
    started = time.monotonic()
    print('\n===', label, '===', flush=True)
    try:
        sys.argv = [str(RUNTIME / 'scripts' / script), *map(str, arguments)]
        runpy.run_path(sys.argv[0], run_name='__main__')
    except BaseException:
        (OUT / 'last_error.txt').write_text(traceback.format_exc(), encoding='utf-8')
        raise
    finally:
        sys.argv = previous_argv
        gc.collect()
        torch.cuda.empty_cache()
        print(f'{label}: {(time.monotonic()-started)/60:.1f} minutes this invocation', flush=True)


## 4. Audit data, retrieval opportunity, redundancy and truncation

This prepares the same top-three passages in two text formats. Sentence packing uses the claim to select verbatim source sentences within 110 tokens; it does not use labels or annotated answers. Source diversity chooses three distinct URLs from the existing top 20. The oracle chooses URL-matched raw passages from the available dev list and remains a source-level diagnostic.

In [ ]:
arguments = ['prepare', '--train-claims', TRAIN_CLAIMS, '--dev-claims', DEV_CLAIMS,
             '--train-rankings', TRAIN_BM25, '--dev-rankings', DEV_BM25, '--qrels', DEV_QRELS,
             '--model-dir', MODEL_DIR, '--output', OUT, '--budget', 110, '--audit-size', AUDIT_CLAIMS]
if DEV100.is_file():
    arguments += ['--dev-top100', DEV100]
for name, path in EXTRA_DEV.items():
    if path.is_file():
        arguments += ['--extra', f'{name}={path}']
run_script('Data and candidate audit', 'run_limitation_diagnostics.py', arguments)
audit = json.loads((OUT / 'data_audit.json').read_text(encoding='utf-8'))
print('Saved-pool URL coverage:', audit['candidate_coverage'])
print('Passages truncated:', audit['passages_over_budget'], '/', audit['passages_measured'])
print('Claims with repeated source in top three:', audit['claims_with_repeated_top3_source'])
print('Lexical term retention (not decisive relevance):', audit['truncation'])
print('Gold-source coverage by label:', audit['coverage_by_gold_label'])
SILVER = DATA_ROOT / 'outputs/initial_ml_solution_answer_aligned_top100_v1/alignment/gold_answer_top100_v1/train_alignment.jsonl'
if SILVER.is_file():
    from scripts.run_limitation_diagnostics import index_rows
    aligned = index_rows(read_jsonl(SILVER), expected={r['claim_id'] for r in train_records})
    silver_report = {'source_sha256': sha256_file(SILVER), 'by_label': {}}
    for label in ['Supported', 'Refuted', 'Not Enough Evidence', 'Conflicting Evidence']:
        group = [r for r in train_records if r['label'] == label]
        silver_report['by_label'][label] = {
            'train_claims': len(group),
            'accepted_positive_claims': sum(bool(aligned[r['claim_id']].get('positive_passage_id')) for r in group),
            'matched_source_claims': sum(bool(aligned[r['claim_id']].get('matched_source_candidate_count')) for r in group),
        }
    silver_report['warning'] = 'Automatic lexical answer alignment is silver supervision; coverage does not establish label quality.'
    write_json(silver_report, OUT / 'silver_supervision_audit.json')
    print('Prior reranker supervision:', json.dumps(silver_report, indent=2))
else:
    print('Optional silver-label audit skipped: previous alignment file was not found.')


## 5. Does the existing verifier use the evidence?

The classifier stays fixed in this section. Blank, unrelated and reversed evidence test sensitivity; sentence packing tests a new input format. Differences may reflect unfamiliar inputs as well as evidence use. URL-oracle and annotated-answer rows use information unavailable for a new claim and cannot be chosen as a deployable method. Every batch is saved; rerunning resumes safely.

In [ ]:
arguments = ['frozen', '--dev-claims', DEV_CLAIMS, '--model-dir', MODEL_DIR,
             '--model-config', MODEL_CONFIG, '--output', OUT, '--batch-size', 16, '--device', 'cuda']
reference = OLD_VERIFIER / 'final_dev_predictions.jsonl'
if reference.is_file():
    arguments += ['--reference-predictions', reference]
run_script('Frozen-model evidence interventions', 'run_limitation_diagnostics.py', arguments)
if (OUT / 'frozen/baseline_reproduction.json').is_file():
    print('Original versus recomputed baseline:', (OUT / 'frozen/baseline_reproduction.json').read_text())


## 6. Read a small passage sample

Open `manual_passage_audit.csv` in Drive. Fill `relevance` with `decisive_support`, `decisive_refute`, `partial`, `irrelevant`, or `unclear`, plus optional notes. Judge whether the passage establishes or contradicts a material part of the claim; mere topic overlap is insufficient. Multi-part claims may require more than one passage. Rank, gold verdict and proxy-match status are hidden to reduce anchoring. Avoid opening the key until after judging. This balanced sample diagnoses failure types; it is not a population estimate. Rerunning preparation preserves your CSV judgments.

In [ ]:
print('Annotate this CSV:', OUT / 'manual_passage_audit.csv')
print('You can continue training before completing the audit; rerun the final summary after entering judgments.')


## 7. Train controlled alternatives on the same train holdout

The seven comparisons share the original stratified 307-claim holdout, seed, optimizer and three-passage limit. Each has one fixed learning rate (5e-5) and a three-epoch maximum, with checkpoint selection by holdout macro-F1. This tests text packing and class weighting without a large hyperparameter sweep. Claim-only is a control. No dev metrics are computed in this cell. Resume uses Trainer checkpoints; completed trials are reused.

In [ ]:
VARIANTS = {
    'head_balanced': ('head', 'balanced'),
    'head_unweighted': ('head', 'none'),
    'sentences_balanced': ('sentences', 'balanced'),
    'sentences_unweighted': ('sentences', 'none'),
    'diverse_sentences_balanced': ('diverse_sentences', 'balanced'),
    'claim_only_balanced': ('claim_only', 'balanced'),
    'claim_only_unweighted': ('claim_only', 'none'),
}
DISTIL_REVISION = '12040accade4e8a0f71eabdb258fecc2e7e948be'

def train_arguments(name, mode, weighting, *, seed=718, selection_only=False,
                    model='distilbert-base-uncased', revision=DISTIL_REVISION,
                    rates='5e-5', policy='stratified', refit_policy='best_checkpoint'):
    arguments = ['--data-root', DATA_ROOT,
                 '--train-rankings', OUT / 'prepared' / mode / 'train_rankings.jsonl',
                 '--eval-rankings', OUT / 'prepared' / mode / 'dev_rankings.jsonl',
                 '--output-dir', OUT / 'training', '--run-name', name, '--model', model,
                 '--learning-rates', rates, '--epoch-grid', '3', '--batch-size', BATCH_SIZE,
                 '--gradient-accumulation-steps', 2, '--checkpoint-steps', 100,
                 '--class-weighting', weighting, '--validation-policy', policy,
                 '--refit-epoch-policy', refit_policy, '--seed', seed, '--validation-seed', 718, '--resume']
    if revision:
        arguments += ['--revision', revision]
    if selection_only:
        arguments += ['--selection-only']
    return arguments

if RUN_TRAINING:
    for name, (mode, weighting) in VARIANTS.items():
        run_script('Train holdout / ' + name, 'train_transformer_verifier.py',
                   train_arguments(name, mode, weighting, selection_only=True))
else:
    print('Training disabled; audit and frozen-model diagnostics are still complete.')


## 8. Select using train holdout, then refit and evaluate

We record the holdout-selected evidence pipeline before computing its dev result. Refitting uses the selected checkpoint's epoch rather than automatically repeating the full three epochs. With `REFIT_ALL_VARIANTS=True`, the other trained variants also get dev rows for diagnosis. Their dev scores do not change the recorded choice. New training runs include this refit-duration change, so compare the frozen experiments to isolate text effects and the matched trials to assess the new training procedure.

In [ ]:
def read_selection(name):
    return json.loads((OUT / 'training' / name / 'model_selection.json').read_text(encoding='utf-8'))['selected_trial']

if RUN_TRAINING:
    ranked = sorted((name for name in VARIANTS if not name.startswith('claim_only')),
                    key=lambda name: (-read_selection(name)['macro_f1'],
                                      read_selection(name)['decision_error_rates']['false_reassurance_rate'],
                                      read_selection(name)['decision_error_rates']['false_alarm_rate'], name))
    CHOSEN = ranked[0]
    selection = {'name': CHOSEN, 'selection': read_selection(CHOSEN),
                 'rule': 'train-holdout macro-F1, then fewer false reassurances and false alarms',
                 'oracle_inputs_allowed': False}
    lock_config(OUT / 'chosen_pipeline_before_dev.json', selection)
    for name in VARIANTS:
        chosen = read_selection(name)
        print(name, 'holdout macro-F1=', round(chosen['macro_f1'], 4),
              'requested epochs=', chosen['epochs'], 'selected checkpoint epoch=', chosen['best_checkpoint_epoch'])
    print('Chosen evidence pipeline:', CHOSEN)
    names = list(VARIANTS) if REFIT_ALL_VARIANTS else [CHOSEN, 'claim_only_unweighted']
    for name in names:
        mode, weighting = VARIANTS[name]
        run_script('Refit and dev / ' + name, 'train_transformer_verifier.py', train_arguments(name, mode, weighting))

    if RUN_REFIT_DURATION_CHECK:
        source_run = OUT / 'training/head_balanced'
        original_selection = read_selection('head_balanced')
        epoch_gap = abs(original_selection['epochs'] - original_selection['best_checkpoint_epoch'])
        if epoch_gap < 0.05:
            print('Refit-duration check skipped: selected epoch already matches the requested epoch.')
        else:
            # Only small completed trial artifacts are copied; optimizer/model checkpoints stay in place.
            import shutil
            name = 'head_balanced_requested_epochs'
            target_run = OUT / 'training' / name
            target_run.mkdir(parents=True, exist_ok=True)
            original_config = json.loads((source_run / 'transformer_run_config.json').read_text())
            original_config['refit_epoch_policy'] = 'requested'
            lock_config(target_run / 'transformer_run_config.json', original_config)
            for summary_file in (source_run / 'trials').glob('*/trial_summary.json'):
                target_trial = target_run / 'trials' / summary_file.parent.name
                target_trial.mkdir(parents=True, exist_ok=True)
                summary = json.loads(summary_file.read_text())
                source_predictions = Path(summary['predictions_path'])
                target_predictions = target_trial / 'validation_predictions.jsonl'
                shutil.copy2(source_predictions, target_predictions)
                summary['predictions_path'] = str(target_predictions)
                write_json(summary, target_trial / 'trial_summary.json')
            run_script('Original versus selected refit duration', 'train_transformer_verifier.py',
                       train_arguments(name, 'head', 'balanced', refit_policy='requested'))


## 9. Check temporal shift within official train

This extra run holds out the latest roughly 10% of train claims, keeping every claim from the boundary date together. It may hold out more than 307 claims, so its score is not directly comparable to the stratified holdout. It never uses official dev for selection. Incomplete dates are reported rather than silently discarded.

In [ ]:
if RUN_TRAINING and RUN_TEMPORAL_CHECK:
    if audit['claim_date_coverage']['train'] != len(train_records):
        print('Temporal check skipped: only', audit['claim_date_coverage']['train'], 'train claims have parseable dates.')
    else:
        run_script('Temporal train holdout / BM25 head', 'train_transformer_verifier.py',
                   train_arguments('temporal_head_balanced', 'head', 'balanced', selection_only=True, policy='temporal'))
        report = json.loads((OUT / 'training/temporal_head_balanced/model_selection.json').read_text())
        print('Temporal holdout claims:', report['selection_validation_claim_count'])
        print('Temporal macro-F1:', report['selected_trial']['macro_f1'])


## 10. Optional: test a stronger encoder and a second seed

Enable the flags in section 3, then run this cell. [Microsoft's DeBERTa-v3-small](https://huggingface.co/microsoft/deberta-v3-small) provides a different pretrained encoder without rerunning retrieval. We test two learning rates on train holdout with the chosen passage format. The model revision is resolved once and recorded. Text excerpts still come from the same 110-token DistilBERT-based preparation; any gain is an encoder/pipeline comparison, not a pure tokenizer-controlled architecture test. A second seed checks sensitivity on the same stratified holdout; it does not establish stability from two runs.

In [ ]:
if RUN_TRAINING and RUN_STRONGER_ENCODER:
    from huggingface_hub import model_info
    revision_path = OUT / 'deberta_revision.json'
    if revision_path.is_file():
        revision = json.loads(revision_path.read_text())['revision']
    else:
        revision = model_info('microsoft/deberta-v3-small').sha
        write_json({'model': 'microsoft/deberta-v3-small', 'revision': revision}, revision_path)
    mode, weighting = VARIANTS[CHOSEN]
    arguments = train_arguments('deberta_selected_format', mode, weighting, model='microsoft/deberta-v3-small',
                                revision=revision, rates='2e-5,5e-5')
    run_script('DeBERTa train-only selection', 'train_transformer_verifier.py', arguments + ['--selection-only'])
    print('DeBERTa holdout:', read_selection('deberta_selected_format'))
    print('DistilBERT holdout:', read_selection(CHOSEN))
    run_script('DeBERTa refit and exploratory dev', 'train_transformer_verifier.py', arguments)
if RUN_TRAINING and RUN_SEED_CHECK:
    # Same validation seed/partition: vary model training randomness only.
    mode, weighting = VARIANTS[CHOSEN]
    run_script('Extra training seed, same holdout', 'train_transformer_verifier.py',
               train_arguments('sensitivity_seed_719', mode, weighting, seed=719, selection_only=True))


## 11. Export the results to share

This exports full metrics, per-class precision/recall/F1, confusion matrices, calibration, safety counts with both denominators, descriptive subgroups, and paired bootstrap intervals. The intervals condition on the current label counts and omit training randomness, repeated dev inspection and claim dependence. They cannot turn exploratory dev findings into a final test result.

In [ ]:
run_script('Summarize limitation study', 'run_limitation_diagnostics.py',
           ['summarize', '--output', OUT, '--dev-claims', DEV_CLAIMS, '--bootstrap-trials', 1000])
from IPython.display import Markdown, display
display(Markdown((OUT / 'limitation_results.md').read_text(encoding='utf-8')))
print('Share these artifacts:')
for name in ['limitation_results.json', 'limitation_results.md', 'data_audit.json',
             'chosen_pipeline_before_dev.json', 'manual_passage_audit.csv']:
    print(OUT / name, 'exists=', (OUT / name).is_file())


## What to send back and how we will use it

Send `limitation_results.json`, `data_audit.json`, and any completed manual audit. The small Markdown summary is useful too. If a run interrupts, reconnect Drive and rerun from setup with the same study name; completed inference batches, trials and training checkpoints are reused. A changed input or source bundle requires a new study name. Checkpoints can consume several GB; this notebook never deletes existing runs.

We will use the results to choose the next main method:

- A large gap between saved-pool coverage and top-three coverage, plus useful lower-ranked raw passages in the audit, supports training a passage reranker with decisive-passage supervision and hard negatives.
- If the model benefits from oracle answers but cannot use relevant raw passages, prioritize passage-level support/refutation reasoning and an evidence aggregator rather than another expensive embedding index.
- If short query-selected excerpts help in matched training, carry that format into the next reader. Keep exact-source provenance.
- If weighting or refit duration explains the gap, fix training before changing retrieval. Inspect each verdict class and both safety errors.
- If the stronger encoder helps on train holdout and later claims, test it with a frozen pipeline and untouched evaluation. If gains vary across seeds or temporal windows, collect that evidence before claiming a large improvement.

We still cannot measure actual trust/share decisions, global open-web search quality, exact publication-time availability, or foundation-model training contamination with these artifacts. Those require additional data or a user study. No notebook can guarantee a large metric gain; this iteration is designed to identify the most defensible next experiment.

Training API reference: [Transformers 4.57.1 Trainer](https://huggingface.co/docs/transformers/v4.57.1/en/main_classes/trainer).